# Note:
* Importing complete source files containing the Lung and Heart transplant datasets and the feature descriptions, including a data dictionary with categorical mapping files in CSV format.
* Separating Adult heart transplant data into CSV format, including a categorical mapping dataset. 

In [1]:
# path to user functions
import sys  
sys.path.append('../Src/')

# import libraries
from platform import python_version
import numpy as np
import pandas as pd

# import user functions
import UserUtilityFunctions as uf
import utilities as u

# print versions
print("Numpy Version: " + np.__version__)
print("Pandas Version: " + pd.__version__)
print("Python Version: " + python_version())

# initializing variables
UNKNOWN = '** UNKNOWN **'
DROP = '** DROP **'
LABEL = '** LABEL **'

# # adjust pandas display options to max
# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)
# # adjust pandas display options to ensure full display of content
# pd.set_option('display.max_colwidth', None)

Numpy Version: 2.3.2
Pandas Version: 3.0.2
Python Version: 3.11.13


## Import Dataset

In [2]:
# export column names
columns_name = pd.read_html('../Data/THORACIC_DATA.htm',)
columns_name_df = columns_name[0]
column_name = list(columns_name_df['LABEL'])

# import into DataFrame
df_main = pd.read_csv('../Data/THORACIC_DATA.DAT', sep='\t', header=None, names = column_name, encoding='latin', low_memory=False)

In [3]:
columns_name = pd.read_html('../Data/THORACIC_FORMATS_FLATFILE.htm',)
columns_name_df = columns_name[0]
column_name = list(columns_name_df['LABEL'])
# import into DataFrame
df_flat = pd.read_csv('../Data/../Data/THORACIC_FORMATS_FLATFILE.DAT', sep='\t', header=None, names = column_name,\
                        encoding='latin')

In [4]:
# open the Excel file
xls_star = pd.ExcelFile('../Docs/optn-star-files-data-dictionary.xlsx')
# get the list of sheet names
print(xls_star.sheet_names)

['Document map', 'DECEASED_DONOR_DATA', 'DECEASED_DONOR_DCD_MEASURES', 'DECEASED_DONOR_INOTROPIC_MEDS', 'INTESTINE_ADDTL_HLA', 'INTESTINE_DATA', 'INTESTINE_FOLLOWUP_DATA', 'INTESTINE_IMMUNO_DISCHARGE_DATA', 'INTESTINE_IMMUNO_FOLLOWUP_DATA', 'INTESTINE_MALIG_FOLLOWUP_DATA', 'INTESTINE_PRA_CROSSMATCH_DATA', 'INTESTINE_WLHISTORY_DATA', 'KIDNEY_FOLLOWUP_DATA', 'KIDNEY_MALIG_FOLLOWUP_DATA', 'KIDPAN_ADDTL_HLA', 'KIDPAN_DATA', 'KIDPAN_FOLLOWUP_DATA', 'KIDPAN_IMMUNO_DISCHARGE_DATA', 'KIDPAN_IMMUNO_FOLLOWUP_DATA', 'KIDPAN_MALIG_FOLLOWUP_DATA', 'KIDPAN_PRA_CROSSMATCH_DATA', 'KIDPAN_WLHISTORY_DATA', 'LIVER_ADDTL_HLA', 'LIVER_DATA', 'LIVER_EXCEPTION_DATA', 'LIVER_EXPLANT_DATA', 'LIVER_FOLLOWUP_DATA', 'LIVER_IMMUNO_DISCHARGE_DATA', 'LIVER_IMMUNO_FOLLOWUP_DATA', 'LIVER_MALIG_FOLLOWUP_DATA', 'LIVER_PRA_CROSSMATCH_DATA', 'LIVER_WLHISTORY_DATA', 'LIVING_DONOR_DATA', 'LIVING_DONOR_FOLLOWUP_DATA', 'PANCREAS_FOLLOWUP_DATA', 'PANCREAS_MALIG_FOLLOWUP_DATA', 'THORACIC_ADDTL_HLA', 'THORACIC_DATA (2)', 'THORAC

In [5]:
# Open XLS workbook - data dictionary
df_dict = pd.read_excel('../Docs/optn-star-files-data-dictionary.xlsx',  sheet_name='THORACIC_DATA', header=1)
# display
df_dict.head()

,VARIABLE NAME,DESCRIPTION,FORM,VAR START DATE,VAR END DATE,FORM SECTION,DATA TYPE,SAS ANALYSIS FORMAT,COMMENT
0,ABN_CONGEN_DON,DDR:Structural Abnormalities //Congenital:,DDR,2004-06-30,NaT,ORGAN RECOVERY,CHAR(1),NaN,NaN
1,ABN_LVH_DON,DDR:Structural Abnormalities //LVH:,DDR,2004-06-30,NaT,ORGAN RECOVERY,CHAR(1),NaN,NaN
2,ABN_VALVES_DON,DDR:Structural Abnormalities //Valves:,DDR,2004-06-30,NaT,ORGAN RECOVERY,CHAR(1),NaN,NaN
3,ABO,RECIPIENT BLOOD GROUP @ REGISTRATION,TCR,1987-10-01,NaT,CLINICAL INFORMATION,CHAR(3),ABO,NaN
4,ABO_DON,DONOR BLOOD TYPE,DDR/LDR,1987-10-01,NaT,DONOR INFORMATION,CHAR(3),ABO,NaN


In [6]:
# rename column name dictionary
renameCols = {"VARIABLE NAME": "Feature", "DESCRIPTION": "Description", "FORM": "Form", \
                  "VAR START DATE": "FeatureStartDate" ,"VAR END DATE": "FeatureEndDate", "FORM SECTION": "FormSection", \
                  "DATA TYPE": "DataType", "SAS ANALYSIS FORMAT": "SASAnalysisFormat", "COMMENT": "Comment"}

# rename columns
df_dict = df_dict.rename(columns=renameCols)

# create additional feature information
df_dict['OrginalFeature'] = df_dict['Feature']
df_dict['FeatureType'] = 'Unknown'
df_dict['Information'] = 'Unknown'
df_dict[['FormSection', 'SASAnalysisFormat', 'Comment']] = df_dict[['FormSection', 'SASAnalysisFormat', 'Comment']].fillna('')

# display
df_dict.head()

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
0,ABN_CONGEN_DON,DDR:Structural Abnormalities //Congenital:,DDR,2004-06-30,NaT,ORGAN RECOVERY,CHAR(1),,,ABN_CONGEN_DON,Unknown,Unknown
1,ABN_LVH_DON,DDR:Structural Abnormalities //LVH:,DDR,2004-06-30,NaT,ORGAN RECOVERY,CHAR(1),,,ABN_LVH_DON,Unknown,Unknown
2,ABN_VALVES_DON,DDR:Structural Abnormalities //Valves:,DDR,2004-06-30,NaT,ORGAN RECOVERY,CHAR(1),,,ABN_VALVES_DON,Unknown,Unknown
3,ABO,RECIPIENT BLOOD GROUP @ REGISTRATION,TCR,1987-10-01,NaT,CLINICAL INFORMATION,CHAR(3),ABO,,ABO,Unknown,Unknown
4,ABO_DON,DONOR BLOOD TYPE,DDR/LDR,1987-10-01,NaT,DONOR INFORMATION,CHAR(3),ABO,,ABO_DON,Unknown,Unknown


In [7]:
# open the flat file from the workbook
df_flat_excel = pd.read_excel('../Docs/optn-star-files-data-dictionary.xlsx',  sheet_name='Flatfile Formats', header=1)

# display
df_flat_excel.head()

,SASAnalysis Format,Data Field Value,Data Field Formatted Value,Data Type
0,ABNBRONC,Null or Missing,Not Reported ...,N
1,ABNBRONC,1,No Bronchoscopy ...,N
2,ABNBRONC,2,Bronchoscopy Results normal ...,N
3,ABNBRONC,3,"Bronchoscopy Results, Abnormal-purulent secret...",N
4,ABNBRONC,4,"Bronchoscopy Results, Abnormal-aspiration of f...",N


In [8]:
# rename column name dictionary
renameCols = {'SASAnalysis Format': 'SASAnalysisFormat', 'Data Field Value':'DataFieldValue', \
               'Data Field Formatted Value':'DataFieldFormattedValue','Data Type':'DataType'}

# rename columns
df_flat_excel.rename(columns=renameCols, inplace=True)

# display
df_flat_excel.head()

,SASAnalysisFormat,DataFieldValue,DataFieldFormattedValue,DataType
0,ABNBRONC,Null or Missing,Not Reported ...,N
1,ABNBRONC,1,No Bronchoscopy ...,N
2,ABNBRONC,2,Bronchoscopy Results normal ...,N
3,ABNBRONC,3,"Bronchoscopy Results, Abnormal-purulent secret...",N
4,ABNBRONC,4,"Bronchoscopy Results, Abnormal-aspiration of f...",N


In [9]:
# display
df_flat.head()

,LABEL,FMTNAME,TYPE,CODE
0,Not Reported,ABNBRONC,N,Null or Missing
1,No Bronchoscopy,ABNBRONC,N,1
2,Bronchoscopy Results normal,ABNBRONC,N,2
3,"Bronchoscopy Results, Abnormal-purulent secret...",ABNBRONC,N,3
4,"Bronchoscopy Results, Abnormal-aspiration of f...",ABNBRONC,N,4


In [10]:
# display the length of each DataFrame
df_main.shape, df_flat.shape, df_dict.shape, df_flat_excel.shape

((200217, 546), (36478, 4), (530, 12), (39057, 4))

In [11]:
# display first 5 rows
df_main.head()

,WL_ORG,COD_WL,COD_OSTXT_WL,TRANSPLANT_COUNTRY,NUM_PREV_TX,THORACIC_DGN,GROUPING,TAH,VAS,ONVENT,...,ABN_CONGEN_DON,WALL_ABN_SEG_DON,WALL_ABN_GLOB_DON,DATA_TRANSPLANT,DATA_WAITLIST,CTR_CODE,OPO_CTR_CODE,INIT_OPO_CTR_CODE,END_OPO_CTR_CODE,LISTING_CTR_CODE
0,LU,.,NaN,.,0,1604,D,NaN,NaN,NaN,...,NaN,NaN,NaN,N,Y,Unknown,Unknown,01054,01054,11191
1,HR,.,NaN,.,0,1201,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,N,Y,Unknown,Unknown,03720,03720,13919
2,HR,.,NaN,.,0,1003,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,N,Y,Unknown,Unknown,01519,01519,19530
3,HR,.,NaN,.,1,1101,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,N,Y,Unknown,Unknown,20243,20243,15283
4,HR,.,NaN,.,0,1007,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,N,Y,Unknown,Unknown,20429,20429,11253


In [12]:
# main count columns
print(f"Number of columns: {df_main.shape[1]:,} & Number of Records: {df_main.shape[0]:,}")

Number of columns: 546 & Number of Records: 200,217


# Data Separate

In [13]:
# extract adults & Heart only from main DataFrame
df_heart = df_main[(df_main.AGE_GROUP == 'A') & (df_main.WL_ORG == "HR")].copy()
# shape
df_heart.shape

(72411, 546)

In [14]:
# display 
df_heart.head()

,WL_ORG,COD_WL,COD_OSTXT_WL,TRANSPLANT_COUNTRY,NUM_PREV_TX,THORACIC_DGN,GROUPING,TAH,VAS,ONVENT,...,ABN_CONGEN_DON,WALL_ABN_SEG_DON,WALL_ABN_GLOB_DON,DATA_TRANSPLANT,DATA_WAITLIST,CTR_CODE,OPO_CTR_CODE,INIT_OPO_CTR_CODE,END_OPO_CTR_CODE,LISTING_CTR_CODE
68957,HR,.,NaN,.,0,1999,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,04464,14911,14911,14911,04464
68958,HR,.,NaN,.,0,1999,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,19034,11346,24149,24149,19034
68959,HR,.,NaN,.,0,1999,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,16616,08866,19809,19809,Unknown
68961,HR,.,NaN,.,0,1999,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,07905,12772,25172,25172,07905
68964,HR,.,NaN,.,0,1999,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,03627,11346,04743,04743,03627


In [15]:
# replace '.' with np.nan using map (map operation across the entire DataFrame)
df_heart = df_heart.map(lambda x: np.nan if x == '.' else x)

# display random 10 rows
df_heart.sample(10)

,WL_ORG,COD_WL,COD_OSTXT_WL,TRANSPLANT_COUNTRY,NUM_PREV_TX,THORACIC_DGN,GROUPING,TAH,VAS,ONVENT,...,ABN_CONGEN_DON,WALL_ABN_SEG_DON,WALL_ABN_GLOB_DON,DATA_TRANSPLANT,DATA_WAITLIST,CTR_CODE,OPO_CTR_CODE,INIT_OPO_CTR_CODE,END_OPO_CTR_CODE,LISTING_CTR_CODE
86659,HR,NaN,NaN,NaN,0,1000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,23312,12865,12865,12865,23312
128541,HR,NaN,NaN,NaN,0,1999,NaN,N,N,N,...,NaN,NaN,NaN,Y,Y,08401,01054,01054,01054,08401
120505,HR,NaN,NaN,NaN,0,1999,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,00124,17856,14012,14012,00124
169062,HR,NaN,NaN,NaN,0,1007,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,06510,03348,03348,03348,06510
87596,HR,NaN,NaN,NaN,0,1000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,12245,22847,22847,22847,12245
91329,HR,NaN,NaN,NaN,0,1000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,19530,01519,01519,01519,19530
137271,HR,NaN,NaN,NaN,0,1200,NaN,N,N,N,...,NaN,NaN,NaN,Y,Y,00248,07006,05952,05952,00248
104782,HR,NaN,NaN,NaN,0,1000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,13919,03720,03720,03720,13919
111819,HR,NaN,NaN,NaN,0,1049,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,11749,24614,04433,04433,11749
164029,HR,NaN,NaN,NaN,0,1007,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,01953,22847,22847,22847,01953


In [16]:
# sanity check
df_heart.WL_ORG.unique(), df_heart.AGE_GROUP.unique()

(<ArrowStringArray>
 ['HR']
 Length: 1, dtype: str,
 <ArrowStringArray>
 ['A']
 Length: 1, dtype: str)

## Eleven Years

In [17]:
# initialize variable
year = 2010
# convert
df_heart['LISTYR'] = pd.to_numeric(df_heart['LISTYR'], errors='coerce')
df_heart['TX_YEAR'] = pd.to_numeric(df_heart['TX_YEAR'], errors='coerce')

# last eleven years' worth of data & max year is 2021
print(f"The Max year for LISTYR is {np.max(df_heart.LISTYR)} & record count >= {year} is:  {df_heart.LISTYR[df_heart.LISTYR >= year].count():,} rows.")

# last eleven years' worth of data & max year is 2021
print(f"The Max year for TX_YEAR is {np.max(df_heart.TX_YEAR)} & record count >= {year} is:  {df_heart.TX_YEAR[df_heart.TX_YEAR >= year].count():,} rows.")

The Max year for LISTYR is 2021 & record count >= 2010 is:  29,541 rows.
The Max year for TX_YEAR is 2021 & record count >= 2010 is:  30,725 rows.


In [18]:
# display data dictionary
df_dict[df_dict.Feature.isin(['LISTYR','TX_YEAR'])]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
317,LISTYR,ACTUAL YEAR REGISTRANT LISTED (WITHOUT DATE OF...,CALCULATED,1987-10-01,NaT,,NUM,,,LISTYR,Unknown,Unknown
478,TX_YEAR,TRANSPLANT YEAR,CALCULATED,NaT,NaT,,NUM,,,TX_YEAR,Unknown,Unknown


### Select Records Greater than equal to 2010 (TX_YEAR)

In [19]:
# copy LISTYR greater than 2010 & reindex
df = df_heart[df_heart.TX_YEAR >= year].reset_index(drop=True).copy()

# display
df.head()

,WL_ORG,COD_WL,COD_OSTXT_WL,TRANSPLANT_COUNTRY,NUM_PREV_TX,THORACIC_DGN,GROUPING,TAH,VAS,ONVENT,...,ABN_CONGEN_DON,WALL_ABN_SEG_DON,WALL_ABN_GLOB_DON,DATA_TRANSPLANT,DATA_WAITLIST,CTR_CODE,OPO_CTR_CODE,INIT_OPO_CTR_CODE,END_OPO_CTR_CODE,LISTING_CTR_CODE
0,HR,NaN,NaN,NaN,0,1000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,23901,07657,07657,07657,23901
1,HR,NaN,NaN,NaN,0,1049,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,05487,11377,11377,11377,05487
2,HR,NaN,NaN,NaN,0,1007,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,12834,14012,14012,14012,12834
3,HR,NaN,NaN,NaN,0,1007,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,00124,17639,14012,14012,00124
4,HR,NaN,NaN,NaN,0,1000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Y,Y,19499,14911,14911,14911,19499


## Data Wrangling

#### User Function(s)

In [20]:
def findMappingDfFlat(dataSeries, dfFlat, formatStr, NaN):
    # fill NaNs
    dataSeries = dataSeries.fillna(NaN)
    # check the datatype of NaN
    if not isinstance(NaN, str):
        # convert to integer
        dataSeries = dataSeries.astype(int)
    # initialize variables
    codeList = list(dataSeries.unique().astype(str))
    # code from flatfile
    flatList = dfFlat['CODE'][dfFlat.FMTNAME == formatStr].to_list()
    # intersection
    intersectionList = list(set(codeList).intersection(set(flatList)))
    # compare length
    print(f"Compare Length: {len(codeList)} & {len(intersectionList)}\n")
    # print
    print(dfFlat[['CODE','LABEL']][(dfFlat.FMTNAME == formatStr) & dfFlat.CODE.isin(intersectionList)].to_string(index=False, index_names=False))

In [21]:
# get columns
dictList = df_dict.Feature.to_list()
dfList = df.columns.to_list()

# remove from dictionary
remove_cols = list((set(dictList)) - (set(dfList)))

# remove not existing features in DataFrame
df_dict = uf.remove_row_using_mask(df_dict, remove_cols, 'Feature').copy()

Remove 26 row(s) from Feature column in a DataFrame.


In [22]:
# remove columns where NaNs >= 80%
dataNaN = uf.percentage_null(df)
# get features
remove_cols = dataNaN.index[dataNaN.percentage >= 80]

# remove features & rows
df = uf.remove_column(df, remove_cols).copy()
df_dict = uf.remove_row_using_mask(df_dict, remove_cols, 'Feature').copy()


Removed Features: ['ABN_CONGEN_DON', 'ABN_LVH_DON', 'ABN_VALVES_DON', 'ACADEMIC_LEVEL_TCR', 'ACADEMIC_LEVEL_TRR', 'ACADEMIC_PRG_TCR', 'ACADEMIC_PRG_TRR', 'BLOOD_INF_CONF_DON', 'CALC_LAS_LISTDATE', 'CANCER_FREE_INT_DON', 'CANCER_OTH_OSTXT_DON', 'CARDIAC_OUTPUT_CATH_INIT_DON', 'CARDIAC_OUTPUT_CATH_OLD_DON', 'CARDIAC_OUTPUT_CATH_POST_DON', 'CARD_IDX_INIT_DON', 'CARD_IDX_POST_DON', 'CIG_GRT_10_OLD', 'CITIZEN_COUNTRY', 'CMV_IGG_DON', 'CMV_IGM_DON', 'CMV_NUCLEIC_DON', 'CMV_OLD_LIV_DON', 'CMV_TEST_DON', 'COD2', 'COD2_OSTXT', 'COD3', 'COD3_OSTXT', 'COD_LIV_DON', 'COD_OSTXT', 'COD_OSTXT_DON', 'COD_OSTXT_WL', 'COD_WL', 'COGNITIVE_DEV_TRR', 'CONTIN_ALCOHOL_OLD_DON', 'CONTIN_CIG_DON', 'CONTIN_CIG_OLD', 'CONTIN_IV_DRUG_OLD_DON', 'CONTROLLED', 'CREAT2_OLD', 'CVP_CATH_INIT_DON', 'CVP_CATH_OLD_DON', 'CVP_CATH_POST_DON', 'DANTIARR_OLD', 'DEATH_DATE', 'DIABDUR_DON', 'DIAG_OSTXT', 'DIASTOLIC_PA_CATH_OLD_DON', 'DIAST_PA_CATH_INIT_DON', 'DIAST_PA_CATH_POST_DON', 'DIET_DON', 'DIURETICS_DON', 'DOBUT_DON_OLD

##### REMOVE ENCRYPTED Features
- **WL_ORG**: ORGAN LISTED FOR
- **WL_ID_CODE**: ENCRYPTED REGISTRATION IDENTIFIER
- **PT_CODE**: ENCRYPTED RECIPIENT IDENTIFIER
- **DONOR_ID**: ENCRYPTED DONOR IDENTIFIER
- **CTR_CODE**: ENCRYPTED REMOVAL/CURRENT OPO MAPPED FROM LISTING CENTER AND ENDING DATE
- **TRR_ID_CODE**: ENCRYPTED TRANSPLANT IDENTIFIER
- **OPO_CTR_CODE**: ENCRYPTED INITIAL OPO MAPPED FROM LISTING CENTER AND BEGINNING DATE
- **INIT_OPO_CTR_CODE**: ENCRYPTED INITIAL OPO MAPPED FROM LISTING CENTER AND BEGINNING DATE
- **LISTING_CTR_CODE**: ENCRYPTED WL LISTING CENTER
- **END_OPO_CTR_CODE**: ENCRYPTED REMOVAL/CURRENT OPO MAPPED FROM LISTING CENTER AND ENDING DATE
- **AGE_GROUP**: RECIPIENT AGE GROUP A=ADULT P=PEDS

In [23]:
# remove fetures
remove_cols = ['WL_ORG', 'WL_ID_CODE', 'PT_CODE', 'DONOR_ID', 'CTR_CODE', 'OPO_CTR_CODE', 'INIT_OPO_CTR_CODE', 
               'LISTING_CTR_CODE','END_OPO_CTR_CODE', 'AGE_GROUP', 'TRR_ID_CODE']

# display data dictionary
uf.dictionary_search(df_dict, remove_cols)

               Feature                                                                   Description FormSection  DataType SASAnalysisFormat Comment Information
9            AGE_GROUP                                            RECIPIENT AGE GROUP A=ADULT P=PEDS               CHAR(1)                               Unknown
51            CTR_CODE                                              ENCRYPTED TRANSPLANT CENTER CODE               CHAR(7)                               Unknown
84            DONOR_ID                                                    ENCRYPTED DONOR IDENTIFIER                   NUM                               Unknown
106   END_OPO_CTR_CODE      ENCRYPTED REMOVAL/CURRENT OPO MAPPED FROM LISTING CENTER AND ENDING DATE               CHAR(7)                               Unknown
174  INIT_OPO_CTR_CODE           ENCRYPTED INITIAL OPO MAPPED FROM LISTING CENTER AND BEGINNING DATE               CHAR(7)                               Unknown
200   LISTING_CTR_CODE            

In [24]:
# remove features & rows
df = uf.remove_column(df, remove_cols).copy()
df_dict = uf.remove_row_using_mask(df_dict, remove_cols, 'Feature').copy()


Removed Features: ['AGE_GROUP', 'CTR_CODE', 'DONOR_ID', 'END_OPO_CTR_CODE', 'INIT_OPO_CTR_CODE', 'LISTING_CTR_CODE', 'OPO_CTR_CODE', 'PT_CODE', 'TRR_ID_CODE', 'WL_ID_CODE', 'WL_ORG']

Total Row(s) & Column(s) Before Removing Column(s): 30,725 & columns: 321
Total Row(s) & Column(s) After Removing Column(s): 30,725 & columns: 310
Remove 11 row(s) from Feature column in a DataFrame.


### LABELS

#### ACUTE_REJ_EPI
* AcuteRejectionEpisode

In [25]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'ACUTE', True)

               count unique top   freq
ACUTE_REJ_EPI  30253      3   3  24559

NaNs:
ACUTE_REJ_EPI    472
dtype: int64

Datatypes:
ACUTE_REJ_EPI    str
dtype: object


         Feature                                                     Description                           FormSection DataType SASAnalysisFormat Comment Information
4  ACUTE_REJ_EPI  DID RECIPIENT HAVE ANY ACUTE REJECTION EPISODES PRE DISCHARGE?  POST TRANSPLANT CLINICAL INFORMATION      NUM          REJEPIKI             Unknown


ACUTE_REJ_EPI: <ArrowStringArray>
['3', '1', '2', nan]
Length: 4, dtype: str


In [26]:
# fill NaN with 999: Missing
df[features] = df[features].fillna('Unknown')

# df_flat FMTNAME: REJEPIKI
mapping = {
    1: "Yes, at least one episode treated with an anti-rejection agent",
    2: "Yes, none treated with additional anti-rejection agent",
    3: "No"
}

# map
df = uf.mapping_columns(df, 'ACUTE_REJ_EPI', mapping, False)

# mapping
colMap = {'ACUTE_REJ_EPI': 'AcuteRejectionEpisode'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"{LABEL} FMTNAME: REJEPIKI")

# convert to category
df = uf.convert_to_category(df, list(colMap.values()))

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
4,AcuteRejectionEpisode,DID RECIPIENT HAVE ANY ACUTE REJECTION EPISODE...,TRR,2004-06-30,NaT,POST TRANSPLANT CLINICAL INFORMATION,NUM,REJEPIKI,,ACUTE_REJ_EPI,Category,** LABEL ** FMTNAME: REJEPIKI


#### PST_
* PST_AIRWAY & PST_STROKE & PST_DIAL & PST_PACEMAKER
    * Airway Dehiscence Post-Transplant
    * Stroke Post Transplant
    * Dialysis Post Discharge
    * Pacemaker Post Transplant

In [27]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'PST_', True)

               count unique top   freq
PST_AIRWAY     30261      3   N  30029
PST_STROKE     30262      3   N  29125
PST_DIAL       30259      3   N  26040
PST_PACEMAKER  30257      3   N  29362

NaNs:
PST_AIRWAY       464
PST_STROKE       463
PST_DIAL         466
PST_PACEMAKER    468
dtype: int64

Datatypes:
PST_AIRWAY       str
PST_STROKE       str
PST_DIAL         str
PST_PACEMAKER    str
dtype: object


           Feature                                     Description                           FormSection DataType SASAnalysisFormat Comment Information
237     PST_AIRWAY    EVENTS PRIOR TO DISCHARGE: AIRWAY DEHISCENCE  POST TRANSPLANT CLINICAL INFORMATION  CHAR(1)                               Unknown
238       PST_DIAL             EVENTS PRIOR TO DISCHARGE: DIALYSIS  POST TRANSPLANT CLINICAL INFORMATION  CHAR(1)                               Unknown
239  PST_PACEMAKER  EVENTS PRIOR TO DISCHARGE: PERMANENT PACEMAKER  POST TRANSPLANT CLINICAL INFORMATION  CHAR(1)                    

In [28]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'PST_AIRWAY': 'AirwayDehiscencePostTransplant', 'PST_STROKE': 'StrokePostTransplant', 
          'PST_PACEMAKER': 'PacemakerPostTransplant', 'PST_DIAL':'DialysisPostDischarge'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"{LABEL} N/Y/U/X to No/Yes/Unknow")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column PST_AIRWAY Unique Vaue(s) ['No', 'Unknown', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column PST_STROKE Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column PST_DIAL Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column PST_PACEMAKER Unique Vaue(s) ['No', 'Unknown', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
237,AirwayDehiscencePostTransplant,EVENTS PRIOR TO DISCHARGE: AIRWAY DEHISCENCE,TRR,1994-04-01,NaT,POST TRANSPLANT CLINICAL INFORMATION,CHAR(1),,,PST_AIRWAY,Category,** LABEL ** N/Y/U/X to No/Yes/Unknow
238,DialysisPostDischarge,EVENTS PRIOR TO DISCHARGE: DIALYSIS,TRR,1994-04-01,NaT,POST TRANSPLANT CLINICAL INFORMATION,CHAR(1),,,PST_DIAL,Category,** LABEL ** N/Y/U/X to No/Yes/Unknow
239,PacemakerPostTransplant,EVENTS PRIOR TO DISCHARGE: PERMANENT PACEMAKER,TRR,1994-04-01,NaT,POST TRANSPLANT CLINICAL INFORMATION,CHAR(1),,,PST_PACEMAKER,Category,** LABEL ** N/Y/U/X to No/Yes/Unknow
240,StrokePostTransplant,EVENTS PRIOR TO DISCHARGE: STROKE,TRR,1994-04-01,NaT,POST TRANSPLANT CLINICAL INFORMATION,CHAR(1),,,PST_STROKE,Category,** LABEL ** N/Y/U/X to No/Yes/Unknow


#### GSTATUS
* GraftFailStatus

In [29]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'GSTATUS', True)

         count unique top   freq
GSTATUS  30299      2   0  23737

NaNs:
GSTATUS    426
dtype: int64

Datatypes:
GSTATUS    str
dtype: object


     Feature           Description FormSection DataType SASAnalysisFormat Comment Information
116  GSTATUS  GRAFT FAILED (1=YES)                  NUM                               Unknown


GSTATUS: <ArrowStringArray>
['1', '0', nan]
Length: 3, dtype: str


In [30]:
# fill NaN with 9: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'0': 'Success', '1': 'Failure'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'GSTATUS': 'GraftFailStatus'}


# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"{LABEL}")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column GSTATUS Unique Vaue(s) ['Failure', 'Success', 'Unknown']
Categories (3, str): ['Failure', 'Success', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
116,GraftFailStatus,GRAFT FAILED (1=YES),CALCULATED,NaT,NaT,,NUM,,,GSTATUS,Category,** LABEL **


#### GTIME
* GraftLifeSpanDay

In [31]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'GTIME', True)

       count unique top freq
GTIME  30299   3731   0  225

NaNs:
GTIME    426
dtype: int64

Datatypes:
GTIME    str
dtype: object


    Feature                                                          Description FormSection DataType SASAnalysisFormat Comment Information
117   GTIME  GRAFT LIFESPAN-Days From Transplant to Failure/Death/Last Follow-Up                  NUM                               Unknown


GTIME: <ArrowStringArray>
[ '329', '2592', '2913', '1225', '2785', '2842', '2853', '2921',  '406',
 '2905',
 ...
  '836', '3210', '2600', '2643',  '261', '1578', '3079', '3096', '2626',
 '2012']
Length: 3732, dtype: str


In [32]:
# change datatype
df[features] = df[features].astype(float)

# mapping
colMap = {'GTIME': 'GraftLifeSpanDay'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"{LABEL}")

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
117,GraftLifeSpanDay,GRAFT LIFESPAN-Days From Transplant to Failure...,CALCULATED,NaT,NaT,,NUM,,,GTIME,Numeric,** LABEL **


#### LASTFUNO
* LastFollowupNumber

In [33]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'LASTFUNO', True)

          count unique  top  freq
LASTFUNO  30725     16  999  4837

NaNs:
LASTFUNO    0
dtype: int64

Datatypes:
LASTFUNO    str
dtype: object


      Feature           Description FormSection DataType SASAnalysisFormat Comment Information
192  LASTFUNO  LAST FOLLOWUP NUMBER                  NUM                               Unknown


LASTFUNO: <ArrowStringArray>
['999',  '80',  '20', '800',   '1',  '30',  '70', '998',  '60',  '50',  '10',
  '40',   '6', '110',  '90', '100']
Length: 16, dtype: str


In [34]:
# mapping
colMap = {'LASTFUNO': 'LastFollowupNumber'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"{LABEL}")

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
192,LastFollowupNumber,LAST FOLLOWUP NUMBER,CALCULATED,NaT,NaT,,NUM,,,LASTFUNO,Numeric,** LABEL **


#### GRF_STAT
* GraftStatus

In [35]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'GRF_STAT', True)

          count unique top   freq
GRF_STAT  30005      2   Y  28269

NaNs:
GRF_STAT    720
dtype: int64

Datatypes:
GRF_STAT    str
dtype: object


      Feature   Description           FormSection DataType SASAnalysisFormat Comment Information
115  GRF_STAT  GRAFT STATUS  CLINICAL INFORMATION  CHAR(1)           GRFSTAT             Unknown


GRF_STAT: <ArrowStringArray>
['N', 'Y', nan]
Length: 3, dtype: str


In [36]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'GRF_STAT':'GraftStatus'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"{LABEL} N/Y/U/X to No/Yes/Unknown/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column GRF_STAT Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
115,GraftStatus,GRAFT STATUS,TRR/TRF,2004-06-30,NaT,CLINICAL INFORMATION,CHAR(1),GRFSTAT,,GRF_STAT,Category,** LABEL ** N/Y/U/X to No/Yes/Unknown/Missing


#### PSTATUS
* TransplantStatus

In [37]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'PSTATUS', True)

         count unique top   freq
PSTATUS  30300      2   0  23942

NaNs:
PSTATUS    425
dtype: int64

Datatypes:
PSTATUS    str
dtype: object


     Feature                                                                           Description     FormSection DataType SASAnalysisFormat Comment Information
241  PSTATUS  Boolean Most Recent Patient Status (based on composite death date) (1=Dead, 0=Alive)  PATIENT STATUS      NUM                               Unknown


PSTATUS: <ArrowStringArray>
['1', '0', nan]
Length: 3, dtype: str


In [38]:
# fill NaN with 9: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'0': 'Alive', '1': 'Dead'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'PSTATUS': 'TransplantStatus'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"{LABEL}")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column PSTATUS Unique Vaue(s) ['Dead', 'Alive', 'Unknown']
Categories (3, str): ['Alive', 'Dead', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
241,TransplantStatus,Boolean Most Recent Patient Status (based on c...,TRR/TRF-CALCULATED,1987-10-01,NaT,PATIENT STATUS,NUM,,,PSTATUS,Category,** LABEL **


#### PTIME
* TransplantSurvivalDay

In [39]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'PTIME', False)

       count unique  top freq
PTIME  30300   3733  365  179

NaNs:
PTIME    425
dtype: int64

Datatypes:
PTIME    str
dtype: object


    Feature                                                    Description FormSection DataType SASAnalysisFormat Comment Information
246   PTIME  Patient Survival Time in days (based on composite death date)                  NUM                               Unknown




In [40]:
# convert
df[features] = df[features].astype(float)
# mapping
colMap = {'PTIME': 'TransplantSurvivalDay'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"{LABEL}")

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
246,TransplantSurvivalDay,Patient Survival Time in days (based on compos...,CALCULATED,NaT,NaT,,NUM,,,PTIME,Numeric,** LABEL **


#### PX_STAT
* RecipientStatus

In [41]:
# display feature info
features, idx = uf.feature_information(df, df_dict, r'^PX_STAT$', True)

         count unique top   freq
PX_STAT  30335      4   A  23503

NaNs:
PX_STAT    390
dtype: int64

Datatypes:
PX_STAT    str
dtype: object


     Feature                                Description     FormSection DataType SASAnalysisFormat Comment Information
250  PX_STAT  RECIPIENT STATUS(Died, ReTX, Lost, Alive)  PATIENT STATUS  CHAR(1)            PXSTAT             Unknown


PX_STAT: <ArrowStringArray>
['D', 'A', 'R', 'L', nan]
Length: 5, dtype: str


In [42]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# df_flat FMTNAME: PXSTAT
mapping = { 
    "A": "Living",
    "D": "Dead",
    "L": "Lost to Follow Up",
    "N": "Not Seen",
    "R": "Retransplanted"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'PX_STAT': 'RecipientStatus'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"{LABEL} FMTNAME: PXSTAT")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column PX_STAT Unique Vaue(s) ['Dead', 'Living', 'Retransplanted', 'Lost to Follow Up', 'Unknown']
Categories (5, str): ['Dead', 'Living', 'Lost to Follow Up', 'Retransplanted', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
250,RecipientStatus,"RECIPIENT STATUS(Died, ReTX, Lost, Alive)",TRR/TRF-CALCULATED,1987-10-01,NaT,PATIENT STATUS,CHAR(1),PXSTAT,,PX_STAT,Category,** LABEL ** FMTNAME: PXSTAT


#### TRTREJ1Y

In [43]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'TRTREJ1Y', True)

          count unique top   freq
TRTREJ1Y  23818      2   N  19458

NaNs:
TRTREJ1Y    6907
dtype: int64

Datatypes:
TRTREJ1Y    str
dtype: object


      Feature                          Description FormSection DataType SASAnalysisFormat Comment Information
280  TRTREJ1Y  TREATED FOR REJECTION WITHIN 1 YEAR              CHAR(1)                               Unknown


TRTREJ1Y: <ArrowStringArray>
['N', nan, 'Y']
Length: 3, dtype: str


In [44]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'TRTREJ1Y': 'RejectionTreatmentWithinOneYear'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"{LABEL} N/Y/X to No/Yes/Unknown")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column TRTREJ1Y Unique Vaue(s) ['No', 'Unknown', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
280,RejectionTreatmentWithinOneYear,TREATED FOR REJECTION WITHIN 1 YEAR,CALCULATED,NaT,NaT,,CHAR(1),,,TRTREJ1Y,Category,** LABEL ** N/Y/X to No/Yes/Unknown


#### DATE FEATURES

In [45]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DATE|VAL_DT|LISTYR', False)

                        count unique                     top freq        mean       std     min     25%     50%     75%     max
ACTIVATE_DATE           30725   4194              12/21/2012   32         NaN       NaN     NaN     NaN     NaN     NaN     NaN
END_DATE                30725   4372              02/12/2021   23         NaN       NaN     NaN     NaN     NaN     NaN     NaN
INIT_DATE               30725   4118              12/21/2012   32         NaN       NaN     NaN     NaN     NaN     NaN     NaN
COMPOSITE_DEATH_DATE     6358   3068              01/29/2021    8         NaN       NaN     NaN     NaN     NaN     NaN     NaN
VAL_DT_TCR              22964   5992  28FEB1995:00:00:00.000  943         NaN       NaN     NaN     NaN     NaN     NaN     NaN
ADMISSION_DATE          30344   4417              06/28/2018   26         NaN       NaN     NaN     NaN     NaN     NaN     NaN
RECOVERY_DATE_DON       30712   4375              02/12/2021   21         NaN       NaN     NaN     NaN 

In [46]:
# deep copy
convertDate = features.copy()

# remove YYYY
convertDate.remove('LISTYR')
convertDate.remove('VAL_DT_TCR')
convertDate.remove('VAL_DT_TRR')
convertDate.remove('VAL_DT_DDR')

# converting date columns in the 'convertDate' list
for col in convertDate:
    df[col] = pd.to_datetime(df[col], format='%m/%d/%Y', errors='coerce')


# new list
convertDate = ['VAL_DT_TCR','VAL_DT_TRR','VAL_DT_DDR']
for col in convertDate:
    df[col] = pd.to_datetime(df[col], format='%d%b%Y:%H:%M:%S.%f', errors='coerce')

In [47]:
# mapping
colMap = {'ACTIVATE_DATE': 'AllocationBeginDate_CAN', 'ADMISSION_DATE':'AdmissionDate_CAN', 'ADMIT_DATE_DON':'AdmissionDate_DON',
          'VAL_DT_TCR':'ValidationDateTCR_CAN','VAL_DT_TRR':'ValidationDateTRR_CAN', 'VAL_DT_DDR':'ValidationDateTCR_DDR','LISTYR':'ListingYear_CAN',
          'DISCHARGE_DATE':'CenterDischargeDate_CAN', 'END_DATE':'RemovalWaitListDate_CAN', 'INIT_DATE':'InitialWaitListDate_CAN',
          'PX_STAT_DATE':'StatusDate_CAN', 'RECOVERY_DATE_DON':'OrganRecoveryDate_DON', 'REFERRAL_DATE':'ReferralDate_DON', 'TX_DATE':'TransplantDate_CAN',
          'COMPOSITE_DEATH_DATE': 'PatientDeathDate'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='DateTime', txt='mm/dd/yyyy')
df_dict = uf.update_dictionary_information(df_dict, [192], txt=f"YYYY")
df_dict = uf.update_dictionary_information(df_dict, [288,289,290], txt=f"ddmonthyyyy:time").copy()

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
3,AllocationBeginDate_CAN,ALLOCATION TIME BEGINNING DATE,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,,,ACTIVATE_DATE,DateTime,mm/dd/yyyy
5,AdmissionDate_CAN,RECIPIENT DATE OF ADMISSION TO TX CENTER,TRR,1999-10-25,NaT,PATIENT STATUS,NUM,,,ADMISSION_DATE,DateTime,mm/dd/yyyy
6,AdmissionDate_DON,DONOR ADMIT DATE,DDR,2006-04-26,NaT,DONOR INFORMATION,NUM,,,ADMIT_DATE_DON,DateTime,mm/dd/yyyy
41,PatientDeathDate,Composite Patient Death Date from OPTN or Veri...,TRR/TRF-CALCULATED,NaT,NaT,,NUM,,,COMPOSITE_DEATH_DATE,DateTime,mm/dd/yyyy
78,CenterDischargeDate_CAN,RECIPIENT DISCHARGE DATE FROM TX CENTER,TRR,1994-04-01,NaT,PATIENT STATUS,NUM,,,DISCHARGE_DATE,DateTime,mm/dd/yyyy
100,RemovalWaitListDate_CAN,EARLIEST OF DATES OF REMOVAL FROM WAITING LIST...,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,,"IF PATIENT TRANSPLANTED OR DIED, BUT WAS REMOV...",END_DATE,DateTime,mm/dd/yyyy
167,InitialWaitListDate_CAN,BEGINNING DATE FOR REGISTRATION,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,,,INIT_DATE,DateTime,mm/dd/yyyy
195,ListingYear_CAN,ACTUAL YEAR REGISTRANT LISTED (WITHOUT DATE OF...,CALCULATED,1987-10-01,NaT,,NUM,,,LISTYR,DateTime,mm/dd/yyyy
251,StatusDate_CAN,RECIPIENT STATUS DATE,TRR/TRF-CALCULATED,1987-10-02,NaT,PATIENT STATUS,NUM,,,PX_STAT_DATE,DateTime,mm/dd/yyyy
259,OrganRecoveryDate_DON,ORGAN RECOVERY DATE,DDR / LDR,1987-10-01,NaT,ORGAN RECOVERY,NUM,,,RECOVERY_DATE_DON,DateTime,mm/dd/yyyy


### ABO

In [48]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'ABO', True)

           count unique  top   freq      mean       std  min  25%  50%  75%  max
ABO        30725      8    A  12145       NaN       NaN  NaN  NaN  NaN  NaN  NaN
ABO_DON    30725      8    O  15653       NaN       NaN  NaN  NaN  NaN  NaN  NaN
ABO_MAT  30725.0    NaN  NaN    NaN  1.144117  0.351307  1.0  1.0  1.0  1.0  3.0

NaNs:
ABO        0
ABO_DON    0
ABO_MAT    0
dtype: int64

Datatypes:
ABO            str
ABO_DON        str
ABO_MAT    float64
dtype: object


   Feature                           Description           FormSection DataType SASAnalysisFormat Comment Information
0      ABO  RECIPIENT BLOOD GROUP @ REGISTRATION  CLINICAL INFORMATION  CHAR(3)               ABO             Unknown
1  ABO_DON                      DONOR BLOOD TYPE     DONOR INFORMATION  CHAR(3)               ABO             Unknown
2  ABO_MAT       DONOR-RECIPIENT ABO MATCH LEVEL                        CHAR(1)            ABOMAT             Unknown


ABO: <ArrowStringArray>
['A', 'O', 'AB', 'B', 'A1', 'A1B',

In [49]:
# update to integer
df.ABO_MAT = df.ABO_MAT.astype(int) 

# df_flat FMTNAME: ABOMAT
mapping = {1: 'Identical', 2: 'Compatible', 3: 'Incompatible'}

# mapping feature
df = uf.mapping_columns(df, 'ABO_MAT', mapping, display=True)

# mapping
colMap = {'ABO': 'BloodGroup_CAN', 'ABO_DON':'BloodGroup_DON', 'ABO_MAT':'BloodGroupMatchLevel'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='')
df_dict = uf.update_dictionary_information(df_dict, [2], txt='FMTNAME: ABOMAT').copy()

# convert to category
df = uf.convert_to_category(df, list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column ABO_MAT Unique Vaue(s) ['Identical', 'Compatible', 'Incompatible']
Categories (3, str): ['Compatible', 'Identical', 'Incompatible']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
0,BloodGroup_CAN,RECIPIENT BLOOD GROUP @ REGISTRATION,TCR,1987-10-01,NaT,CLINICAL INFORMATION,CHAR(3),ABO,,ABO,Category,
1,BloodGroup_DON,DONOR BLOOD TYPE,DDR/LDR,1987-10-01,NaT,DONOR INFORMATION,CHAR(3),ABO,,ABO_DON,Category,
2,BloodGroupMatchLevel,DONOR-RECIPIENT ABO MATCH LEVEL,CALCULATED,NaT,NaT,,CHAR(1),ABOMAT,,ABO_MAT,Category,FMTNAME: ABOMAT


### AGE

In [50]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'AGE', True)

                     count unique top  freq
INIT_AGE             30725     68  61  1210
AGE_DON              30725     62  21  1146
AGE                  30725     62  64  1203
INO_PROCURE_AGENT_1  12185      6   5  4944

NaNs:
INIT_AGE                   0
AGE_DON                    0
AGE                        0
INO_PROCURE_AGENT_1    18540
dtype: int64

Datatypes:
INIT_AGE               str
AGE_DON                str
AGE                    str
INO_PROCURE_AGENT_1    str
dtype: object


                 Feature                                  Description            FormSection DataType SASAnalysisFormat Comment Information
7                    AGE                          RECIPIENT AGE (YRS)  RECIPIENT INFORMATION      NUM                               Unknown
8                AGE_DON                              DONOR AGE (YRS)      DONOR INFORMATION      NUM                               Unknown
164             INIT_AGE              AGE IN YEARS AT TIME OF LISTING                   

In [51]:
# fill NaN with 998: Unknown & convert to integer
df[features] = df[features].fillna(998).astype(int)

# SASAnalysisFormat: INOMED                                                                                       
mapping = {
    1: "Dopamine",                                                                                                
    2: "Dobutamine",                                                                                              
    3: "Epinephrine",                                                                                             
    4: "Levophed",                                                                                                
    5: "Neosynephrine",                                                                                           
    6: "Isoproterenol (Isuprel)",
    998: "Unknown",
    999: "Other, specify"
}

# mapping feature
df = uf.mapping_columns(df, 'INO_PROCURE_AGENT_1', mapping, display=True)

# mapping
colMap = {'AGE_DON':'Age_DON', 'AGE': 'Age_CAN', 'INIT_AGE':'Age_Listing_CAN', 'INO_PROCURE_AGENT_1':'InotropicAgent_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt='')
df_dict = uf.update_dictionary_information(df_dict, [171], txt=f"SASAnalysisFormat: INOMED" , feature_type='Category').copy()

# display
df_dict.iloc[idx]

Converted Column INO_PROCURE_AGENT_1 Unique Vaue(s) ['Unknown', 'Dopamine', 'Other, specify', 'Levophed', 'Neosynephrine', 'Dobutamine', 'Epinephrine']
Categories (7, str): ['Dobutamine', 'Dopamine', 'Epinephrine', 'Levophed', 'Neosynephrine', 'Other, specify', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
7,Age_CAN,RECIPIENT AGE (YRS),TRR-CALCULATED,1987-10-01,NaT,RECIPIENT INFORMATION,NUM,,,AGE,Numeric,
8,Age_DON,DONOR AGE (YRS),DDR/LDR-CALCULATED,1987-10-01,NaT,DONOR INFORMATION,NUM,,,AGE_DON,Numeric,
164,Age_Listing_CAN,AGE IN YEARS AT TIME OF LISTING,CALCULATED,1987-10-01,NaT,,NUM,,,INIT_AGE,Numeric,
173,InotropicAgent_DON,DECEASED DONOR-INOTROPIC MEDICATION AGENT 1,DDR,2003-01-27,NaT,CLINICAL INFORMATION,NUM,INOMED,,INO_PROCURE_AGENT_1,Numeric,


### ALCOHOL

In [52]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'ALCOHOL', True)

                   count unique top   freq
ALCOHOL_HEAVY_DON  30712      3   N  24789

NaNs:
ALCOHOL_HEAVY_DON    13
dtype: int64

Datatypes:
ALCOHOL_HEAVY_DON    str
dtype: object


             Feature                              Description        FormSection DataType SASAnalysisFormat Comment Information
9  ALCOHOL_HEAVY_DON  Heavy Alcohol Use (heavy=2+ drinks/day)  LIFESTYLE FACTORS  CHAR(1)                               Unknown


ALCOHOL_HEAVY_DON: <ArrowStringArray>
['N', 'Y', 'U', nan]
Length: 4, dtype: str


In [53]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'ALCOHOL_HEAVY_DON': 'HeavyAlcoholUse_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknown")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column ALCOHOL_HEAVY_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
9,HeavyAlcoholUse_DON,Heavy Alcohol Use (heavy=2+ drinks/day),DDR,2004-06-30,NaT,LIFESTYLE FACTORS,CHAR(1),,,ALCOHOL_HEAVY_DON,Category,N/Y/U/X to No/Yes/Unknown


### [LOCUS MISMATCH LEVEL](https://www.sciencedirect.com/science/article/pii/S1071916423002324?casa_token=Lq4LHsmUCLQAAAAA:TvHFPaDMW0rIPYF7E4SDPC20Cg1jEpCws6ztyM-jKLMrkxbuK0oDq4q5huBI0OmiSi3JSsTfd9Yu)
* AMIS & BMIS & DRMIS & HLAMIS

In [54]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'MIS', True)

        count unique top   freq
AMIS    28087      3   2  14469
BMIS    28086      3   2  20235
DRMIS   28079      3   2  15395
HLAMIS  28076      7   5  10469

NaNs:
AMIS      2638
BMIS      2639
DRMIS     2646
HLAMIS    2649
dtype: int64

Datatypes:
AMIS      str
BMIS      str
DRMIS     str
HLAMIS    str
dtype: object


    Feature              Description FormSection DataType SASAnalysisFormat Comment Information
10     AMIS   A LOCUS MISMATCH LEVEL                  NUM                               Unknown
18     BMIS   B LOCUS MISMATCH LEVEL                  NUM                               Unknown
90    DRMIS  DR Locus MISMATCH LEVEL                  NUM                               Unknown
155  HLAMIS       HLA MISMATCH LEVEL                  NUM                               Unknown


AMIS: <ArrowStringArray>
['2', '1', '0', nan]
Length: 4, dtype: str
BMIS: <ArrowStringArray>
['2', '1', nan, '0']
Length: 4, dtype: str
DRMIS: <ArrowStringArray>
['1', '2', nan, '0']
Length: 4, 

In [55]:
# fill NaN with 999: Missing
df[features] = df[features].fillna('Unknown')

# mapping
colMap = {'AMIS': 'MismatchLevel_AMIS', 'HLAMIS':'MismatchLevel_HLAMIS', 'BMIS': 'MismatchLevel_BMIS', 'DRMIS': 'MismatchLevel_DRMIS'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
10,MismatchLevel_AMIS,A LOCUS MISMATCH LEVEL,CALCULATED,NaT,NaT,,NUM,,,AMIS,Category,
18,MismatchLevel_BMIS,B LOCUS MISMATCH LEVEL,CALCULATED,NaT,NaT,,NUM,,,BMIS,Category,
90,MismatchLevel_DRMIS,DR Locus MISMATCH LEVEL,CALCULATED,NaT,NaT,,NUM,,,DRMIS,Category,
155,MismatchLevel_HLAMIS,HLA MISMATCH LEVEL,CALCULATED,NaT,NaT,,NUM,,,HLAMIS,Category,


### ANTIHYPERTENSIVE

In [56]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'ANTIHYPE', True)

              count unique top   freq
ANTIHYPE_DON  30518      3   N  20571

NaNs:
ANTIHYPE_DON    207
dtype: int64

Datatypes:
ANTIHYPE_DON    str
dtype: object


         Feature                                                   Description           FormSection DataType SASAnalysisFormat Comment Information
11  ANTIHYPE_DON  DECEASED DONOR-ANTIHYPERTENSIVES W/IN 24 HRS PRE-CROSS CLAMP  CLINICAL INFORMATION  CHAR(1)                               Unknown


ANTIHYPE_DON: <ArrowStringArray>
['Y', 'N', nan, 'U']
Length: 4, dtype: str


In [57]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'ANTIHYPE_DON': 'AntiHypertensive_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknown")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column ANTIHYPE_DON Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
11,AntiHypertensive_DON,DECEASED DONOR-ANTIHYPERTENSIVES W/IN 24 HRS P...,DDR,1994-04-01,NaT,CLINICAL INFORMATION,CHAR(1),,,ANTIHYPE_DON,Category,N/Y/U/X to No/Yes/Unknown


### ARGININE

In [58]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'ARGININE', True)

              count unique top   freq
ARGININE_DON  30518      3   Y  21012

NaNs:
ARGININE_DON    207
dtype: int64

Datatypes:
ARGININE_DON    str
dtype: object


         Feature                                                                         Description           FormSection DataType SASAnalysisFormat Comment Information
12  ARGININE_DON  DECEASED DONOR-WAS DONOR GIVEN ARGININE VASOPRESSIN WITHIN 24 HRS PRE CROSS CLAMP?  CLINICAL INFORMATION  CHAR(1)                               Unknown


ARGININE_DON: <ArrowStringArray>
['N', 'Y', nan, 'U']
Length: 4, dtype: str


In [59]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'ARGININE_DON':'ArginnieManagement_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknown")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column ARGININE_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
12,ArginnieManagement_DON,DECEASED DONOR-WAS DONOR GIVEN ARGININE VASOPR...,DDR,2004-06-30,NaT,CLINICAL INFORMATION,CHAR(1),,,ARGININE_DON,Category,N/Y/U/X to No/Yes/Unknown


### DIAGNOSIS

In [60]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DGN|DIAG', True)

              count unique   top   freq
THORACIC_DGN  30725     37  1000  10608
TCR_DGN       30652     37  1000  10608
DIAG          30654     36  1000  10411
BIOPSY_DGN    30518      4     1  30485

NaNs:
THORACIC_DGN      0
TCR_DGN          73
DIAG             71
BIOPSY_DGN      207
dtype: int64

Datatypes:
THORACIC_DGN    str
TCR_DGN         str
DIAG            str
BIOPSY_DGN      str
dtype: object


          Feature                                                                              Description                          FormSection DataType SASAnalysisFormat                                                                 Comment Information
13     BIOPSY_DGN  BIOPSY PERFORMED: NO, YES MYOCARDITIS, YES NEG. BIOPSY RESULT, YES OTHER DIAG. SPECIFY.       HEART DONOR'S CARDIAC FUNCTION      NUM                                                                                               Unknown
74           DIAG                                                              REC

In [61]:
# check for differences between two sets
uf.symmetric_difference(set(df.TCR_DGN.dropna().unique().astype(int)), set(df.THORACIC_DGN.dropna().unique().astype(int)))

Symmetric difference: []


In [62]:
# check for differences between two sets
uf.symmetric_difference(set(df.DIAG.dropna().unique().astype(int)), set(df.THORACIC_DGN.dropna().unique().astype(int)))

Symmetric difference: [np.int64(1205)]


In [63]:
# check for differences between two sets
uf.symmetric_difference(set(df.DIAG.dropna().unique().astype(int)), set(df.TCR_DGN.dropna().unique().astype(int)))

Symmetric difference: [np.int64(1205)]


In [64]:
findMappingDfFlat(df.BIOPSY_DGN, df_flat, 'BIOPCONF', NaN=998)

Compare Length: 5 & 3

CODE                        LABEL
   1              Biopsy not done
   2     Yes, rejection confirmed
   3 Yes, rejection not confirmed


In [65]:
# df_flat FMTNAME: TH_DGN
mapping = {
    998: 'Unknown',
    999: 'OTHER - SPECIFY',
    1000: 'DILATED MYOPATHY: IDIOPATHIC',
    1001: 'DILATED MYOPATHY: ADRIAMYCIN',
    1002: 'DILATED MYOPATHY: POST PARTUM',
    1003: 'DILATED MYOPATHY: FAMILIAL',
    1004: 'DILATED MYOPATHY: MYOCARDITIS',
    1005: 'DILATED MYOPATHY: ALCOHOLIC',
    1006: 'DILATED MYOPATHY: VIRAL',
    1007: 'DILATED MYOPATHY: ISCHEMIC',
    1008: 'DILATED MYOPATHY: VIRAL (NOT COVID-19)',
    1009: 'COVID-19: DILATED MYOPATHY: ACTIVE MYOCARDITIS',
    1010: 'COVID-19: DILATED MYOPATHY: HISTORY OF MYOCARDITIS',
    1049: 'DILATED MYOPATHY: OTHER SPECIFY',
    1050: 'RESTRICTIVE MYOPATHY: IDIOPATHIC',
    1051: 'RESTRICTIVE MYOPATHY: AMYLOIDOSIS',
    1052: 'RESTRICTIVE MYOPATHY: ENDOCARDIAL FIBROS',
    1053: 'RESTRICTIVE MYOPATHY: SARCOIDOSIS',
    1054: 'RESTRICTIVE MYOPATHY: SEC TO RADIAT/CHEM',
    1099: 'RESTRICTIVE MYOPATHY: OTHER SPECIFY',
    1100: 'HEART RE-TX/GF: HYPERACUTE REJECTION',
    1101: 'HEART RE-TX/GF: ACUTE REJECTION',
    1102: 'HEART RE-TX/GF: CORONARY ARTERY DISEASE',
    1103: 'HEART RE-TX/GF: NON-SPECIFIC',
    1104: 'HEART RE-TX/GF: RESTRICTIVE/CONSTRICTIVE',
    1105: 'HEART RE-TX/GF: CHRONIC REJECTION',
    1106: 'HEART RE-TX/GF: PRIMARY FAILURE',
    1199: 'HEART RE-TX/GF: OTHER SPECIFY',
    1200: 'CORONARY ARTERY DISEASE',
    1201: 'HYPERTROPHIC CARDIOMYOPATHY',
    1202: 'VALVULAR HEART DISEASE',
    1203: 'CONGENITAL HEART DEFECT - PRIOR SURGERY UNKNOWN',
    1204: 'CANCER',
    1205: 'CONGENITAL HEART DEFECT - HYPOPLASTIC LEFT HEART SYNDROME - UNOPERATED',
    1206: 'CONGENITAL HEART DEFECT - WITHOUT SURGERY',
    1207: 'CONGENITAL HEART DEFECT - WITH SURGERY',
    1208: 'ARRHYTHMOGENIC RIGHT VENTRICULAR DYSPLASIA/CARDIOMYOPATHY',
    1209: 'MUSCULAR DYSTROPHY: OTHER SPECIFY'
}

# fill NaN with 998: Missing
df[features] = df[features].fillna(998)
# convert to integer
df[features] = df[features].astype(int)

# mapping
df = uf.mapping_columns(df, 'THORACIC_DGN', mapping)
df = uf.mapping_columns(df, 'TCR_DGN', mapping)
df = uf.mapping_columns(df, 'DIAG', mapping)

# df_flat FMTNAME: BIOPCONF
mapping = {
    1: 'Biopsy not done',
    2: 'Yes, rejection confirmed',
    3: 'Yes, rejection not confirmed',
    4: 'Unknown',
    998: 'Unknown'
}

# mapping
df = uf.mapping_columns(df, 'BIOPSY_DGN', mapping)

# mapping
colMap = {'BIOPSY_DGN':'Biopsy_DON', 'THORACIC_DGN': 'WaitListDiagnosisCode_CAN', 'TCR_DGN': 'DiagnosisAtListing_CAN', 'DIAG':'PrimaryDiagnosisType_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='FMTNAME: TH_DGN')
df_dict = uf.update_dictionary_information(df_dict, [13], txt='FMTNAME: BIOPCONF').copy()

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column THORACIC_DGN Unique Vaue(s) ['DILATED MYOPATHY: IDIOPATHIC', 'DILATED MYOPATHY: OTHER SPECIFY', 'DILATED MYOPATHY: ISCHEMIC', 'HEART RE-TX/GF: CORONARY ARTERY DISEASE', 'HYPERTROPHIC CARDIOMYOPATHY', ..., 'CANCER', 'DILATED MYOPATHY: VIRAL (NOT COVID-19)', 'COVID-19: DILATED MYOPATHY: ACTIVE MYOCARDITIS', 'COVID-19: DILATED MYOPATHY: HISTORY OF MYOCAR..., 'CONGENITAL HEART DEFECT - HYPOPLASTIC LEFT HE...]
Length: 37
Categories (37, str): ['ARRHYTHMOGENIC RIGHT VENTRICULAR DYSPLASIA/CA..., 'CANCER', 'CONGENITAL HEART DEFECT - HYPOPLASTIC LEFT HE..., 'CONGENITAL HEART DEFECT - PRIOR SURGERY UNKNOWN', ..., 'RESTRICTIVE MYOPATHY: OTHER SPECIFY', 'RESTRICTIVE MYOPATHY: SARCOIDOSIS', 'RESTRICTIVE MYOPATHY: SEC TO RADIAT/CHEM', 'VALVULAR HEART DISEASE']
Converted Column TCR_DGN Unique Vaue(s) ['DILATED MYOPATHY: IDIOPATHIC', 'DILATED MYOPATHY: VIRAL', 'DILATED MYOPATHY: ISCHEMIC', 'VALVULAR HEART DISEASE', 'HEART RE-TX/GF: CORONARY ARTERY DISEASE', ..., 'DILATED MYOPATHY: VIR

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
13,Biopsy_DON,"BIOPSY PERFORMED: NO, YES MYOCARDITIS, YES NEG...",DDR,1999-10-25,NaT,HEART DONOR'S CARDIAC FUNCTION,NUM,,,BIOPSY_DGN,Category,FMTNAME: BIOPCONF
74,PrimaryDiagnosisType_CAN,RECIPIENT PRIMARY DIAGNOSIS,TRR>TCR,1987-10-01,NaT,PATIENT STATUS/CLINICAL INFORMATION,NUM,ALL_DGN,"THIS FIELD DRAWS FROM ""AT TRANSPLANT"" AND IF ...",DIAG,Category,FMTNAME: TH_DGN
274,DiagnosisAtListing_CAN,CANDIDATE DIAGNOSIS AT LISTING,TCR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,ALL_DGN,,TCR_DGN,Category,FMTNAME: TH_DGN
276,WaitListDiagnosisCode_CAN,Waitlist CANDIDATE DIAGNOSIS,WL DATA,NaT,NaT,,NUM,ALL_DGN,,THORACIC_DGN,Category,FMTNAME: TH_DGN


### DEATH

In [66]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'COD|DEATH', True)

                  count unique  top   freq
COD                6238     72  998    734
COD_CAD_DON       30721      5    3  14283
DEATH_CIRCUM_DON  30710      7    6   7180
DEATH_MECH_DON    30717     12    9   8673

NaNs:
COD                 24487
COD_CAD_DON             4
DEATH_CIRCUM_DON       15
DEATH_MECH_DON          8
dtype: int64

Datatypes:
COD                 str
COD_CAD_DON         str
DEATH_CIRCUM_DON    str
DEATH_MECH_DON      str
dtype: object


             Feature                           Description        FormSection DataType SASAnalysisFormat Comment Information
39               COD      RECIPIENT PRIMARY CAUSE OF DEATH     PATIENT STATUS      NUM           ALL_COD             Unknown
40       COD_CAD_DON         DECEASED DONOR-CAUSE OF DEATH  DONOR INFORMATION      NUM           DON_COD             Unknown
70  DEATH_CIRCUM_DON  DECEASED DONOR-CIRCUMSTANCE OF DEATH  DONOR INFORMATION      NUM           DTHCIRC             Unknown
71    DEATH_MECH_DON     DECEASED DON

In [67]:
# fill NaN with 1000: Unknown
df[features] = df[features].fillna('998')

# df_flat FMTNAME: DON_COD
mapping = {
    '1': 'ANOXIA',
    '2': 'CEREBROVASCULAR/STROKE',
    '3': 'HEAD TRAUMA',
    '4': 'CNS TUMOR',
    '999': 'OTHER SPECIFY',
    '998': 'Unknown'
}

# mapping feature
df = uf.mapping_columns(df, 'COD_CAD_DON', mapping, display=True)

# df_flat FMTNAME: DTHCIRC
mapping = {
    '1': "MVA",
    '2': "SUICIDE",
    '3': "HOMICIDE",
    '4': "CHILD-ABUSE",
    '5': "Accident, Non-MVA",
    '6': "DEATH FROM NATURAL CAUSES",
    '997': "NONE OF THE ABOVE",
    '998': "Unknown"
}

# mapping feature
df = uf.mapping_columns(df, 'DEATH_CIRCUM_DON', mapping, display=True)

# df_flat FMTNAME: DEATH_MECH_DON
mapping = {
    '1': "DROWNING",
    '2': "SEIZURE",
    '3': "DRUG INTOXICATION",
    '4': "ASPHYXIATION",
    '5': "CARDIOVASCULAR",
    '6': "ELECTRICAL",
    '7': "GUNSHOT WOUND",
    '8': "STAB",
    '9': "BLUNT INJURY",
    '10': "SIDS",
    '11': "INTRACRANIAL HEMORRHAGE/STROKE",
    '12': "DEATH FROM NATURAL CAUSES",
    '995': "995-Gunshot/stab wound (Pre-OTIS)",
    '997': "NONE OF THE ABOVE",
    '998': "Unknwn"
}
# mapping feature
df = uf.mapping_columns(df, 'DEATH_MECH_DON', mapping, display=True)

# mapping
colMap = {'COD_CAD_DON': 'CauseOfDeath_DON','DEATH_CIRCUM_DON':'DeathCircumstance_DON', 'DEATH_MECH_DON':'DeathMechanism_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"")
df_dict = uf.update_dictionary_information(df_dict, [39], txt='FMTNAME: DON_COD')
df_dict = uf.update_dictionary_information(df_dict, [68], txt='FMTNAME: DTHCIRC')
df_dict = uf.update_dictionary_information(df_dict, [69], txt='FMTNAME: DEATH_MECH_DON')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column COD_CAD_DON Unique Vaue(s) ['HEAD TRAUMA', 'ANOXIA', 'CEREBROVASCULAR/STROKE', 'OTHER SPECIFY', 'CNS TUMOR', 'Unknown']
Categories (6, str): ['ANOXIA', 'CEREBROVASCULAR/STROKE', 'CNS TUMOR', 'HEAD TRAUMA', 'OTHER SPECIFY', 'Unknown']
Converted Column DEATH_CIRCUM_DON Unique Vaue(s) ['HOMICIDE', 'Accident, Non-MVA', 'NONE OF THE ABOVE', 'DEATH FROM NATURAL CAUSES', 'SUICIDE', 'MVA', 'Unknown', 'CHILD-ABUSE']
Categories (8, str): ['Accident, Non-MVA', 'CHILD-ABUSE', 'DEATH FROM NATURAL CAUSES', 'HOMICIDE', 'MVA', 'NONE OF THE ABOVE', 'SUICIDE', 'Unknown']
Converted Column DEATH_MECH_DON Unique Vaue(s) ['GUNSHOT WOUND', 'BLUNT INJURY', 'DRUG INTOXICATION', 'INTRACRANIAL HEMORRHAGE/STROKE', 'CARDIOVASCULAR', ..., 'SEIZURE', 'STAB', 'Unknwn', 'DROWNING', 'ELECTRICAL']
Length: 13
Categories (13, str): ['ASPHYXIATION', 'BLUNT INJURY', 'CARDIOVASCULAR', 'DEATH FROM NATURAL CAUSES', ..., 'NONE OF THE ABOVE', 'SEIZURE', 'STAB', 'Unknwn']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
39,COD,RECIPIENT PRIMARY CAUSE OF DEATH,TRF/TRR,1987-10-01,NaT,PATIENT STATUS,NUM,ALL_COD,,COD,Category,FMTNAME: DON_COD
40,CauseOfDeath_DON,DECEASED DONOR-CAUSE OF DEATH,DDR,1987-10-01,NaT,DONOR INFORMATION,NUM,DON_COD,,COD_CAD_DON,Category,
70,DeathCircumstance_DON,DECEASED DONOR-CIRCUMSTANCE OF DEATH,DDR,1994-04-01,NaT,DONOR INFORMATION,NUM,DTHCIRC,,DEATH_CIRCUM_DON,Category,
71,DeathMechanism_DON,DECEASED DONOR-MECHANISM OF DEATH,DDR,1994-04-01,NaT,DONOR INFORMATION,NUM,DTHMECH,,DEATH_MECH_DON,Category,


### INFECTION

#### BLOOD_INF_DON & OTHER_INF_DON & PULM_INF_DON & URINE_INF_DON

###### BLOOD_INF_DON & OTHER_INF_DON & PULM_INF_DON & URINE_INF_DON

In [68]:
# display feature info
features, idx = uf.feature_information(df, df_dict, '_INF_', True)

                   count unique top   freq
BLOOD_INF_DON      30722      2   0  27744
OTHER_INF_DON      23059      2   0  20847
PULM_INF_DON       30722      2   1  21025
PULM_INF_CONF_DON   6660      2   Y   6582
URINE_INF_DON      30722      2   0  27177

NaNs:
BLOOD_INF_DON            3
OTHER_INF_DON         7666
PULM_INF_DON             3
PULM_INF_CONF_DON    24065
URINE_INF_DON            3
dtype: int64

Datatypes:
BLOOD_INF_DON        str
OTHER_INF_DON        str
PULM_INF_DON         str
PULM_INF_CONF_DON    str
URINE_INF_DON        str
dtype: object


               Feature                                          Description           FormSection DataType SASAnalysisFormat Comment Information
14       BLOOD_INF_DON             DECEASED DONOR-BLOOD AS INFECTION SOURCE  CLINICAL INFORMATION      NUM                               Unknown
210      OTHER_INF_DON                DECEASED DONOR INFECTION OTHER SOURCE  CLINICAL INFORMATION      NUM                               Unknown

In [69]:
# fill NaN with 999: Missing
df[features] = df[features].fillna('Unknown')

# df_flat FMTNAME: DTHCIRC
mapping = {
    '0': "No",
    '1': "Yes",
    'Y': "Yes",
    'N': "No"
}

# mapping feature
for feature in features:
    df = uf.mapping_columns(df, feature, mapping, display=True)

# convert to category
df = uf.convert_to_category(df, features)

# mapping
colMap = {'BLOOD_INF_DON': 'BloodInfectionSource_DON', 'OTHER_INF_DON': 'OtherInfectionSource_DON',
         'PULM_INF_DON': 'PulmonaryInfection_DON','URINE_INF_DON': 'UrineInfection_DON'}


# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column BLOOD_INF_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column OTHER_INF_DON Unique Vaue(s) ['No', 'Unknown', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column PULM_INF_DON Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column PULM_INF_CONF_DON Unique Vaue(s) ['Yes', 'Unknown', 'No']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column URINE_INF_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
14,BloodInfectionSource_DON,DECEASED DONOR-BLOOD AS INFECTION SOURCE,DDR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,,BLOOD_INF_DON,Category,
210,OtherInfectionSource_DON,DECEASED DONOR INFECTION OTHER SOURCE,DDR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,,OTHER_INF_DON,Category,
248,PULM_INF_CONF_DON,DECEASED DONOR-INFECTION PULMONARY SOURCE-CONF...,DDR,1994-04-01,2015-03-31,CLINICAL INFORMATION,CHAR(1),,,PULM_INF_CONF_DON,Category,
249,PulmonaryInfection_DON,DECEASED DONOR-INFECTION PULMONARY SOURCE,DDR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,,PULM_INF_DON,Category,
287,UrineInfection_DON,DECEASED DONOR-INFECTION URINE SOURCE,DDR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,,URINE_INF_DON,Category,


### INFECT_IV_DRUG_TRR & CLIN_INFECT_DON

In [70]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'INFECT', True)

                    count unique top   freq
INFECT_IV_DRUG_TRR  30297      3   N  26800
CLIN_INFECT_DON     30519      3   Y  23059

NaNs:
INFECT_IV_DRUG_TRR    428
CLIN_INFECT_DON       206
dtype: int64

Datatypes:
INFECT_IV_DRUG_TRR    str
CLIN_INFECT_DON       str
dtype: object


                Feature                                                               Description                         FormSection DataType SASAnalysisFormat Comment Information
34      CLIN_INFECT_DON                                   DECEASED DONOR-CLINICAL INFECTION (Y,N)                CLINICAL INFORMATION  CHAR(1)                               Unknown
160  INFECT_IV_DRUG_TRR  INFECTION REQUIRING IV DRUG THERAPY (WITHIN 2 WEEKS PRIOR TO TRANSPLANT)  PRETRANSPLANT CLINICAL INFORMATION  CHAR(1)                               Unknown


INFECT_IV_DRUG_TRR: <ArrowStringArray>
['N', 'U', 'Y', nan]
Length: 4, dtype: str
CLIN_INFECT_DON: <ArrowStringArray>
['Y', 'N', nan, 'U']
Length: 4, dtype: str


###### INFECT_IV_DRUG_TRR & CLIN_INFECT_DON

In [71]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'INFECT_IV_DRUG_TRR': 'InfectionTherapyIV_CAN', 'CLIN_INFECT_DON': 'InfectionClinical_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknow/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column INFECT_IV_DRUG_TRR Unique Vaue(s) ['No', 'Unknown', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column CLIN_INFECT_DON Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
34,InfectionClinical_DON,"DECEASED DONOR-CLINICAL INFECTION (Y,N)",DDR,1994-04-01,NaT,CLINICAL INFORMATION,CHAR(1),,,CLIN_INFECT_DON,Category,N/Y/U/X to No/Yes/Unknow/Missing
160,InfectionTherapyIV_CAN,INFECTION REQUIRING IV DRUG THERAPY (WITHIN 2 ...,TRR,1994-04-01,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(1),,,INFECT_IV_DRUG_TRR,Category,N/Y/U/X to No/Yes/Unknow/Missing


### BMI
BMI = $\frac{weight(kg)}{height(m)^2}$

In [72]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'BMI_', False)

               count unique         top freq
BMI_TCR        30606  13659     28.1228   46
INIT_BMI_CALC  30682    310        25.8  312
END_BMI_CALC   30712    295        25.8  323
BMI_DON_CALC   30715  14973  22.8571429   39
BMI_CALC       30712    299        25.1  304

NaNs:
BMI_TCR          119
INIT_BMI_CALC     43
END_BMI_CALC      13
BMI_DON_CALC      10
BMI_CALC          13
dtype: int64

Datatypes:
BMI_TCR          str
INIT_BMI_CALC    str
END_BMI_CALC     str
BMI_DON_CALC     str
BMI_CALC         str
dtype: object


           Feature                                       Description                      FormSection DataType SASAnalysisFormat Comment Information
15        BMI_CALC                          Calculated Recipient BMI                                       NUM                               Unknown
16    BMI_DON_CALC            Donor BMI - Pre/At Donation Calculated                                       NUM                               Unknown
17         BMI_TCR       

In [73]:
# change datatype
df[features] = df[features].astype(float)

# mapping
colMap = {'BMI_CALC': 'BMI_Calc_CAN',
          'BMI_DON_CALC':'BMI_Donation_DON',
          'BMI_TCR':'BMI_Listing_CAN',
          'END_BMI_CALC': 'BMI_Removal_CAN',
          'INIT_BMI_CALC': 'BMI_Listing_CALC_CAN'
         }

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt='')

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
15,BMI_Calc_CAN,Calculated Recipient BMI,CALCULATED,NaT,NaT,,NUM,,,BMI_CALC,Numeric,
16,BMI_Donation_DON,Donor BMI - Pre/At Donation Calculated,CALCULATED,NaT,NaT,,NUM,,,BMI_DON_CALC,Numeric,
17,BMI_Listing_CAN,BMI AT LISTING,TCR,2004-06-30,NaT,CLINICAL INFORMATION AT LISTING,NUM,,,BMI_TCR,Numeric,
99,BMI_Removal_CAN,Calculated Candidate BMI at Removal/Current Time,CALCULATED,NaT,NaT,,NUM,,,END_BMI_CALC,Numeric,
166,BMI_Listing_CALC_CAN,Calculated Candidate BMI at Listing,CALCULATED,NaT,NaT,,NUM,,,INIT_BMI_CALC,Numeric,


### BRONCHOSCOPY

In [74]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'BRONCHO', True)

                count unique top   freq
BRONCHO_LT_DON  15384      8   2  10980
BRONCHO_RT_DON  15139      8   2  10269

NaNs:
BRONCHO_LT_DON    15341
BRONCHO_RT_DON    15586
dtype: int64

Datatypes:
BRONCHO_LT_DON    str
BRONCHO_RT_DON    str
dtype: object


           Feature                  Description     FormSection DataType SASAnalysisFormat Comment Information
19  BRONCHO_LT_DON   DDR LEFT LUNG BRONCHOSCOPY  ORGAN RECOVERY      NUM          ABNBRONC             Unknown
20  BRONCHO_RT_DON  DDR RIGHT LUNG BRONCHOSCOPY  ORGAN RECOVERY      NUM          ABNBRONC             Unknown


BRONCHO_LT_DON: <ArrowStringArray>
['2', nan, '5', '1', '3', '6', '7', '4', '998']
Length: 9, dtype: str
BRONCHO_RT_DON: <ArrowStringArray>
['2', nan, '1', '3', '5', '6', '7', '4', '998']
Length: 9, dtype: str


In [75]:
# fill NaN with 999: Unknown
df[features] = df[features].fillna(999).astype(int)

# df_flat FMTNAME: ABNBRONC
mapping = {
    1: "No Bronchoscopy",
    2: "Normal",
    3: "Abnormal-purulent secretions",
    4: "Abnormal-aspiration of foreign body",
    5: "Abnormal-blood",
    6: "Abnormal-anatomy/other lesion",
    7: "Unknown",
    998: "Unknown if bronchoscopy performed",
    999: "Unknwn"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'BRONCHO_LT_DON':'BronchoscopyLeft_DON', 'BRONCHO_RT_DON':'BronchoscopyRight_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: ABNBRONC")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column BRONCHO_LT_DON Unique Vaue(s) ['Normal', 'Unknwn', 'Abnormal-blood', 'No Bronchoscopy', 'Abnormal-purulent secretions', 'Abnormal-anatomy/other lesion', 'Unknown', 'Abnormal-aspiration of foreign body', 'Unknown if bronchoscopy performed']
Categories (9, str): ['Abnormal-anatomy/other lesion', 'Abnormal-aspiration of foreign body', 'Abnormal-blood', 'Abnormal-purulent secretions', ..., 'Normal', 'Unknown', 'Unknown if bronchoscopy performed', 'Unknwn']
Converted Column BRONCHO_RT_DON Unique Vaue(s) ['Normal', 'Unknwn', 'No Bronchoscopy', 'Abnormal-purulent secretions', 'Abnormal-blood', 'Abnormal-anatomy/other lesion', 'Unknown', 'Abnormal-aspiration of foreign body', 'Unknown if bronchoscopy performed']
Categories (9, str): ['Abnormal-anatomy/other lesion', 'Abnormal-aspiration of foreign body', 'Abnormal-blood', 'Abnormal-purulent secretions', ..., 'Normal', 'Unknown', 'Unknown if bronchoscopy performed', 'Unknwn']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
19,BronchoscopyLeft_DON,DDR LEFT LUNG BRONCHOSCOPY,DDR,1999-10-25,NaT,ORGAN RECOVERY,NUM,ABNBRONC,,BRONCHO_LT_DON,Category,FMTNAME: ABNBRONC
20,BronchoscopyRight_DON,DDR RIGHT LUNG BRONCHOSCOPY,DDR,1999-10-25,NaT,ORGAN RECOVERY,NUM,ABNBRONC,,BRONCHO_RT_DON,Category,FMTNAME: ABNBRONC


### BUN_DON
* Blood Urea Nitrogen Level

In [76]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'BUN', False)

         count unique    top  freq
BUN_DON  30518    231  11.00  1435

NaNs:
BUN_DON    207
dtype: int64

Datatypes:
BUN_DON    str
dtype: object


    Feature                                  Description           FormSection DataType SASAnalysisFormat Comment Information
21  BUN_DON  DECEASED DONOR-TERMINAL BLOOD UREA NITROGEN  CLINICAL INFORMATION      NUM                               Unknown




In [77]:
# change datatype
df[features] = df[features].astype(float)
# mapping
colMap = {'BUN_DON': 'BloodUreaNitrogenLevel_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"")

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
21,BloodUreaNitrogenLevel_DON,DECEASED DONOR-TERMINAL BLOOD UREA NITROGEN,DDR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,,BUN_DON,Numeric,


### BW4 & BW6
* BW4 and BW6 are mutually exclusive epitopes associated with all HLA-B antigens.
* BW4: Candidate Most Recent/at Removal BW4 Antigen From Waiting List
* BW6: Candidate Most Recent/at Removal BW6 Antigen From Waiting List

**Note:**
* [HLA Bw4 and Bw6 Epitopes Recognized by Antibodies and Natural Killer Cells](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC5742561/)
* The removal of the BW4 antigen from a waiting list could signify the clearance of individuals or groups that have been screened for this antigen and deemed safe from its threat.
* The `unit of measurement` for Bw4 and Bw6 epitopes would be more appropriately described in terms of their presence or absence than any quantitative measurement.  

In [78]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'BW', True)

     count unique top   freq
BW4  30725      4   0  21182
BW6  30725      4   0  20825

NaNs:
BW4    0
BW6    0
dtype: int64

Datatypes:
BW4    str
BW6    str
dtype: object


   Feature                                                     Description        FormSection DataType SASAnalysisFormat Comment Information
22     BW4  Candidate Most Recent/at Removal BW4 Antigen From Waiting List  WAITING LIST DATA      NUM          WKGRPHLA             Unknown
23     BW6  Candidate Most Recent/at Removal BW6 Antigen From Waiting List  WAITING LIST DATA      NUM          WKGRPHLA             Unknown


BW4: <ArrowStringArray>
['0', '96', '95', '99']
Length: 4, dtype: str
BW6: <ArrowStringArray>
['0', '95', '96', '99']
Length: 4, dtype: str


In [79]:
# df_flat FMTNAME: WKGRPHLA
# A negative result for AntigenBW4 & AntigenBW6 (HLA-Bw4 & HLA-Bw6 antigen) means that this specific antigen is not present in the sample tested. 
# This is consistent with a value of 0, indicating the absence of the HLA-Bw4 & HLA-Bw6 antigen.
mapping = {
    '0': "No Antigen",
    '95': "Positive",
    '96': "Negative",
    '98': "Confirmed Blank",
    '99': "Not Done"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'BW4': 'AntigenBW4_CAN', 'BW6':'AntigenBW6_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: WKGRPHLA")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column BW4 Unique Vaue(s) ['No Antigen', 'Negative', 'Positive', 'Not Done']
Categories (4, str): ['Negative', 'No Antigen', 'Not Done', 'Positive']
Converted Column BW6 Unique Vaue(s) ['No Antigen', 'Positive', 'Negative', 'Not Done']
Categories (4, str): ['Negative', 'No Antigen', 'Not Done', 'Positive']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
22,AntigenBW4_CAN,Candidate Most Recent/at Removal BW4 Antigen F...,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,WKGRPHLA,,BW4,Category,FMTNAME: WKGRPHLA
23,AntigenBW6_CAN,Candidate Most Recent/at Removal BW6 Antigen F...,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,WKGRPHLA,,BW6,Category,FMTNAME: WKGRPHLA


### [C1 & C2](https://pubmed.ncbi.nlm.nih.gov/30946220/)

In [80]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'C1|C2', True)

    count unique top   freq
C1  30725     37   0  21137
C2  30725     45   0  22036

NaNs:
C1    0
C2    0
dtype: int64

Datatypes:
C1    str
C2    str
dtype: object


   Feature                                                    Description        FormSection DataType SASAnalysisFormat Comment Information
24      C1  Candidate Most Recent/at Removal C1 Antigen From Waiting List  WAITING LIST DATA      NUM             CWHLA             Unknown
25      C2  Candidate Most Recent/at Removal C2 Antigen From Waiting List  WAITING LIST DATA      NUM             CWHLA             Unknown


C1: <ArrowStringArray>
[   '0',    '7',    '5',    '4',   '10',    '1',    '6',    '2',   '15',
    '8',   '16',    '9',   '14',   '12',    '3',   '17',   '18',  '701',
  '702',  '401',  '802',  '303',  '202',  '804', '1601',  '210',  '304',
  '102',  '704', '1203',  '306',  '302', '1505', '1701',  '602',  '501',
  '718']
Length: 37, dtype: str
C2: <ArrowStringArray>
[   '0',   '16',   '18',   '10',    '7',

In [81]:
# convert datatype
df[features] = df[features].astype(int)

# df_flat FMTNAME: CWHLA
mapping = {
    0: "No Antigen",
    1: "01",
    2: "02",
    3: "03",
    4: "04",
    5: "05",
    6: "06",
    7: "07",
    8: "08",
    9: "09",
    10: "10",
    11: "11",
    12: "12",
    13: "13",
    14: "14",
    15: "15",
    16: "16",
    17: "17",
    18: "18",
    97: "Unknown",
    98: "No second antigen detected",
    99: "Not Done",
    100: "No antigen detected",
    102: "01:02",
    103: "01:03",
    202: "02:02",
    210: "02:10",
    302: "03:02",
    303: "03:03",
    304: "03:04",
    305: "03:05",
    306: "03:06",
    401: "04:01",
    403: "04:03",
    404: "04:04",
    407: "04:07",
    501: "05:01",
    602: "06:02",
    701: "07:01",
    702: "07:02",
    704: "07:04",
    706: "07:06",
    718: "07:18",
    801: "08:01",
    802: "08:02",
    803: "08:03",
    804: "08:04",
    1202: "12:02",
    1203: "12:03",
    1204: "12:04",
    1402: "14:02",
    1403: "14:03",
    1502: "15:02",
    1504: "15:04",
    1505: "15:05",
    1506: "15:06",
    1509: "15:09",
    1601: "16:01",
    1602: "16:02",
    1604: "16:04",
    1701: "17:01",
    1703: "17:03",
    1801: "18:01",
    1802: "18:02"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'C1': 'AntigenC1_CAN', 'C2':'AntigenC2_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: CWHLA")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column C1 Unique Vaue(s) ['No Antigen', '07', '05', '04', '10', ..., '15:05', '17:01', '06:02', '05:01', '07:18']
Length: 37
Categories (37, str): ['01', '01:02', '02', '02:02', ..., '17', '17:01', '18', 'No Antigen']
Converted Column C2 Unique Vaue(s) ['No Antigen', '16', '18', '10', '07', ..., '07:18', '06:02', '08:03', '02:10', '16:04']
Length: 45
Categories (45, str): ['01', '01:02', '02', '02:02', ..., '18', '18:01', '18:02', 'No Antigen']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
24,AntigenC1_CAN,Candidate Most Recent/at Removal C1 Antigen Fr...,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,CWHLA,,C1,Category,FMTNAME: CWHLA
25,AntigenC2_CAN,Candidate Most Recent/at Removal C2 Antigen Fr...,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,CWHLA,,C2,Category,FMTNAME: CWHLA


### [DDR](https://pmc.ncbi.nlm.nih.gov/articles/PMC5141243/)

In [82]:
# display feature info
features, idx = uf.feature_information(df, df_dict, '^DDR', False)

      count unique top  freq
DDR1  30720     43   4  7634
DDR2  30708     47  15  6356

NaNs:
DDR1     5
DDR2    17
dtype: int64

Datatypes:
DDR1    str
DDR2    str
dtype: object


   Feature        Description                             FormSection DataType SASAnalysisFormat Comment              Information
68    DDR1  DONOR DR1 ANTIGEN  DONOR CENTER HISTOCOMPATIBILITY TYPING      NUM           DRLOCUS                 FMTNAME: DTHCIRC
69    DDR2  DONOR DR2 ANTIGEN  DONOR CENTER HISTOCOMPATIBILITY TYPING      NUM           DRLOCUS          FMTNAME: DEATH_MECH_DON




In [83]:
# fill NaN with 999: Missing
df[features] = df[features].fillna(999).astype(int)

# df_flat FMTNAME: DRLOCUS

DRLOCUS = {
    999: "Unknown",
    0: "No Antigen",
    1: "1",
    2: "2",
    3: "3",
    4: "4",
    5: "5",
    6: "6",
    7: "7",
    8: "8",
    9: "9",
    10: "10",
    11: "11",
    12: "12",
    13: "13",
    14: "14",
    15: "15",
    16: "16",
    17: "17",
    18: "18",
    97: "Unknown",
    98: "No second antigen detected",
    99: "Not Done",
    101: "01:01",
    102: "01:02",
    103: "01:03",
    301: "03:01",
    302: "03:02",
    303: "03:03",
    401: "04:01",
    402: "04:02",
    403: "04:03",
    404: "04:04",
    405: "04:05",
    406: "04:06",
    407: "04:07",
    410: "04:10",
    411: "04:11",
    801: "08:01",
    802: "08:02",
    803: "08:03",
    807: "08:07",
    901: "09:01",
    902: "09:02",
    1101: "11:01",
    1103: "11:03",
    1104: "11:04",
    1201: "12:01",
    1202: "12:02",
    1301: "13:01",
    1302: "13:02",
    1303: "13:03",
    1305: "13:05",
    1401: "14:01",
    1402: "14:02",
    1403: "14:03",
    1404: "14:04",
    1405: "14:05",
    1406: "14:06",
    1454: "14:54",
    1501: "15:01",
    1502: "15:02",
    1503: "15:03",
    1601: "16:01",
    1602: "16:02",
    10300: "103"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, DRLOCUS, display=True)

# mapping
colMap = {'DDR1': 'AntigenDDR1_DON', 'DDR2':'AntigenDDR2_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: DRLOCUS")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column DDR1 Unique Vaue(s) ['4', '10', '11', '1', '7', ..., '08:02', '14:02', '04:11', '11:04', '103']
Length: 44
Categories (44, str): ['01:01', '01:02', '01:03', '03:01', ..., '7', '8', '9', 'Unknown']
Converted Column DDR2 Unique Vaue(s) ['15', '16', '8', '10', '11', ..., '04:07', '14:01', '103', '12:02', '5']
Length: 48
Categories (48, str): ['01:01', '01:03', '03:01', '04:01', ..., '8', '9', 'No second antigen detected', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
68,AntigenDDR1_DON,DONOR DR1 ANTIGEN,CALCULATED,1987-10-01,NaT,DONOR CENTER HISTOCOMPATIBILITY TYPING,NUM,DRLOCUS,,DDR1,Category,FMTNAME: DRLOCUS
69,AntigenDDR2_DON,DONOR DR2 ANTIGEN,CALCULATED,1987-10-01,NaT,DONOR CENTER HISTOCOMPATIBILITY TYPING,NUM,DRLOCUS,,DDR2,Category,FMTNAME: DRLOCUS


### DR5
- Antigen

In [84]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DR5', True)

        count unique top   freq
DR51    30725      9   0  22427
DR51_2  30725      8   0  29154
DR52    30725     10   0  21805
DR52_2  30725     10   0  29059
DR53    30725      7   0  22155
DR53_2  30725      6   0  29147

NaNs:
DR51      0
DR51_2    0
DR52      0
DR52_2    0
DR53      0
DR53_2    0
dtype: int64

Datatypes:
DR51      str
DR51_2    str
DR52      str
DR52_2    str
DR53      str
DR53_2    str
dtype: object


   Feature                                                      Description        FormSection DataType SASAnalysisFormat Comment Information
84    DR51  Candidate Most Recent/at Removal DR51 Antigen From Waiting List  WAITING LIST DATA      NUM          WKGRPHLA             Unknown
85  DR51_2  Candidate Most Recent/at Removal DR51 Antigen From Waiting List  WAITING LIST DATA      NUM          WKGRPHLA             Unknown
86    DR52  Candidate Most Recent/at Removal DR52 Antigen From Waiting List  WAITING LIST DATA      NUM          WKGRPHLA             Unknown
87  

In [85]:
# change datatype
df[features] = df[features].astype(int)

# df_flat FMTNAME: WKGRPHLA
mapping = {
    0: "No Antigen",
    1: "1",
    2: "2",
    3: "3",
    4: "4",
    5: "5",
    7: "7",
    95: "Positive",
    96: "Negative",
    98: "Confirmed Blank",
    99: "Not Done",
    998: "Unknown"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'DR51': 'AntigenDR51_CAN', 'DR51_2':'AntigenDR51_2_CAN', 'DR52':'AntigenDR52_CAN', 'DR52_2':'AntigenDR52_2_CAN', 
          'DR53':'AntigenDR53_CAN', 'DR53_2':'AntigenDR53_2_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: WKGRPHLA")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column DR51 Unique Vaue(s) ['No Antigen', 'Positive', 'Negative', 'Not Done', '1', '3', '4', '5', '2']
Categories (9, str): ['1', '2', '3', '4', ..., 'Negative', 'No Antigen', 'Not Done', 'Positive']
Converted Column DR51_2 Unique Vaue(s) ['No Antigen', 'Negative', '3', 'Positive', '5', 'Not Done', '1', '2']
Categories (8, str): ['1', '2', '3', '5', 'Negative', 'No Antigen', 'Not Done', 'Positive']
Converted Column DR52 Unique Vaue(s) ['No Antigen', 'Positive', 'Negative', 'Not Done', '1', '4', '2', '5', '7', '3']
Categories (10, str): ['1', '2', '3', '4', ..., 'Negative', 'No Antigen', 'Not Done', 'Positive']
Converted Column DR52_2 Unique Vaue(s) ['No Antigen', 'Positive', '1', 'Negative', '5', '2', '3', 'Not Done', '4', '7']
Categories (10, str): ['1', '2', '3', '4', ..., 'Negative', 'No Antigen', 'Not Done', 'Positive']
Converted Column DR53 Unique Vaue(s) ['No Antigen', 'Negative', 'Not Done', 'Positive', '3', '2', '1']
Categories (7, str): ['1', '2', '3', 'Negative', 'N

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
84,AntigenDR51_CAN,Candidate Most Recent/at Removal DR51 Antigen ...,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,WKGRPHLA,,DR51,Category,FMTNAME: WKGRPHLA
85,AntigenDR51_2_CAN,Candidate Most Recent/at Removal DR51 Antigen ...,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,WKGRPHLA,,DR51_2,Category,FMTNAME: WKGRPHLA
86,AntigenDR52_CAN,Candidate Most Recent/at Removal DR52 Antigen ...,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,WKGRPHLA,,DR52,Category,FMTNAME: WKGRPHLA
87,AntigenDR52_2_CAN,Candidate Most Recent/at Removal DR52 Antigen ...,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,WKGRPHLA,,DR52_2,Category,FMTNAME: WKGRPHLA
88,AntigenDR53_CAN,Candidate Most Recent/at Removal DR53 Antigen ...,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,WKGRPHLA,,DR53,Category,FMTNAME: WKGRPHLA
89,AntigenDR53_2_CAN,Candidate Most Recent/at Removal DR53 Antigen ...,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,WKGRPHLA,,DR53_2,Category,FMTNAME: WKGRPHLA


### PREV_TX
- Previous Transplant Information

In [86]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'PREV_TX', True)

             count unique top   freq
NUM_PREV_TX  30725      4   0  29714
PREV_TX      30725      2   N  29763
PREV_TX_ANY  30725      2   N  29687

NaNs:
NUM_PREV_TX    0
PREV_TX        0
PREV_TX_ANY    0
dtype: int64

Datatypes:
NUM_PREV_TX    str
PREV_TX        str
PREV_TX_ANY    str
dtype: object


         Feature                                                                Description        FormSection DataType SASAnalysisFormat Comment Information
206  NUM_PREV_TX                                         THE NUMBER OF PREVIOUS TRANSPLANTS  WAITING LIST DATA      NUM                               Unknown
222      PREV_TX  HISTORY of a PREVIOUS TRANSPLANT INVOLVING EXACT SAME ORGAN AS CURRENT TX                     CHAR(1)                               Unknown
223  PREV_TX_ANY                           CALCULATED Previous Transplant of Any Organ Type                     CHAR(1)                               Unknown


NUM_PREV_TX: <ArrowStringArray>
['0', '1', '2', '3']
Length: 4

In [87]:
# change datatype
df.NUM_PREV_TX = df.NUM_PREV_TX.astype(str)
# value mapping
mapping = {'N': 'No', 'Y': 'Yes'}

# mapping feature
df = uf.mapping_columns(df, 'PREV_TX', mapping, display=True)
df = uf.mapping_columns(df, 'PREV_TX_ANY', mapping, display=True)

# mapping feature 
colMap = {'NUM_PREV_TX': 'PreviousTransplantNumber_CAN', 'PREV_TX': 'PreviousTransplantSameOrgan_CAN', 'PREV_TX_ANY':'PreviousTransplantAnyOrgan_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Number', txt='')
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, [244,245], feature_info='Category', txt='Y/N to Yes/No')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column PREV_TX Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']
Converted Column PREV_TX_ANY Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
206,PreviousTransplantNumber_CAN,THE NUMBER OF PREVIOUS TRANSPLANTS,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,,,NUM_PREV_TX,Number,
222,PreviousTransplantSameOrgan_CAN,HISTORY of a PREVIOUS TRANSPLANT INVOLVING EXA...,CALCULATED,NaT,NaT,,CHAR(1),,,PREV_TX,Number,
223,PreviousTransplantAnyOrgan_CAN,CALCULATED Previous Transplant of Any Organ Type,CALCULATED,NaT,NaT,,CHAR(1),,,PREV_TX_ANY,Number,


### GENDER

In [88]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'GENDER', True)

            count unique top   freq
GENDER      30725      2   M  22554
GENDER_DON  30725      2   M  21704

NaNs:
GENDER        0
GENDER_DON    0
dtype: int64

Datatypes:
GENDER        str
GENDER_DON    str
dtype: object


        Feature       Description            FormSection DataType SASAnalysisFormat Comment Information
113      GENDER  RECIPIENT GENDER  CANDIDATE INFORMATION  CHAR(1)               SEX             Unknown
114  GENDER_DON      DONOR GENDER      DONOR INFORMATION  CHAR(1)               SEX             Unknown


GENDER: <ArrowStringArray>
['M', 'F']
Length: 2, dtype: str
GENDER_DON: <ArrowStringArray>
['M', 'F']
Length: 2, dtype: str


In [89]:
# mapping
colMap = {'GENDER': 'Gender_CAN', 'GENDER_DON':'Gender_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
113,Gender_CAN,RECIPIENT GENDER,TCR,1987-10-01,NaT,CANDIDATE INFORMATION,CHAR(1),SEX,,GENDER,Category,
114,Gender_DON,DONOR GENDER,DDR/LDR,1987-10-01,NaT,DONOR INFORMATION,CHAR(1),SEX,,GENDER_DON,Category,


### WGT_KG

In [90]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'WGT_KG', False)

                  count unique      top freq
WGT_KG_TCR        30673   3402  81.6470  276
INIT_WGT_KG_CALC  30703    977     81.6  346
END_WGT_KG_CALC   30718    959     81.6  337
WGT_KG_DON_CALC   30721   1150     70.0  517
WGT_KG_CALC       30718   1020     77.1  261

NaNs:
WGT_KG_TCR          52
INIT_WGT_KG_CALC    22
END_WGT_KG_CALC      7
WGT_KG_DON_CALC      4
WGT_KG_CALC          7
dtype: int64

Datatypes:
WGT_KG_TCR          str
INIT_WGT_KG_CALC    str
END_WGT_KG_CALC     str
WGT_KG_DON_CALC     str
WGT_KG_CALC         str
dtype: object


              Feature                                                Description           FormSection DataType SASAnalysisFormat Comment Information
105   END_WGT_KG_CALC  Calculated Candidate Weight in KG at Removal/Current Time                            NUM                               Unknown
172  INIT_WGT_KG_CALC               Calculated Candidate Weight in KG at Listing                            NUM                               Unkno

In [91]:
# change datatype
df[features] = df[features].astype(float)

# mapping
colMap = {'END_WGT_KG_CALC': 'Weight_kg_Removal_CAN',
          'INIT_WGT_KG_CALC': 'Weight_kg_Listing_CAN',
          'WGT_KG_CALC': 'Weight_kg_CAN',
          'WGT_KG_DON_CALC': 'Weight_kg_DON',
          'WGT_KG_TCR': 'Weight_kg_Registrsation_CAN'
         }

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt='')

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
105,Weight_kg_Removal_CAN,Calculated Candidate Weight in KG at Removal/C...,CALCULATED,NaT,NaT,,NUM,,,END_WGT_KG_CALC,Numeric,
172,Weight_kg_Listing_CAN,Calculated Candidate Weight in KG at Listing,CALCULATED,NaT,NaT,,NUM,,,INIT_WGT_KG_CALC,Numeric,
301,Weight_kg_CAN,CALCULATED RECIPIENT WEIGHT (kg),CALCULATED,NaT,NaT,,NUM,,,WGT_KG_CALC,Numeric,
302,Weight_kg_DON,CALCULATED DONOR WEIGHT (KG),CALCULATED,NaT,NaT,,NUM,,,WGT_KG_DON_CALC,Numeric,
303,Weight_kg_Registrsation_CAN,RECIPIENT WEIGHT (kg) @ REGISTRATION,TCR,1987-10-01,NaT,CLINICAL INFORMATION,NUM,,,WGT_KG_TCR,Numeric,


### HGT_CM

In [92]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'HGT_CM', False)

                  count unique    top  freq
HGT_CM_TCR        30608     82    178  2937
INIT_HGT_CM_CALC  30724    384  177.8  2740
END_HGT_CM_CALC   30725    388  177.8  2747
HGT_CM_DON_CALC   30725    185  178.0  1769
HGT_CM_CALC       30725    363  177.8  2612

NaNs:
HGT_CM_TCR          117
INIT_HGT_CM_CALC      1
END_HGT_CM_CALC       0
HGT_CM_DON_CALC       0
HGT_CM_CALC           0
dtype: int64

Datatypes:
HGT_CM_TCR          str
INIT_HGT_CM_CALC    str
END_HGT_CM_CALC     str
HGT_CM_DON_CALC     str
HGT_CM_CALC         str
dtype: object


              Feature                                                Description           FormSection DataType SASAnalysisFormat Comment Information
101   END_HGT_CM_CALC  Calculated Candidate Height in CM at Removal/Current Time                            NUM                               Unknown
142       HGT_CM_CALC                            CALCULATED RECIPIENT HEIGHT(cm)                            NUM                               Unknow

In [93]:
# change datatype
df[features] = df[features].astype(float)

# mapping
colMap = {'END_HGT_CM_CALC': 'Height_cm_Removal_CAN',
          'HGT_CM_CALC': 'Height_cm_CAN',
          'HGT_CM_DON_CALC': 'Height_cm_DON',
          'HGT_CM_TCR': 'Height_cm_Registration_CAN',
          'INIT_HGT_CM_CALC': 'Height_cm_Listing_CAN'
         }

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt='')

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
101,Height_cm_Removal_CAN,Calculated Candidate Height in CM at Removal/C...,CALCULATED,NaT,NaT,,NUM,,,END_HGT_CM_CALC,Numeric,
142,Height_cm_CAN,CALCULATED RECIPIENT HEIGHT(cm),CALCULATED,NaT,NaT,,NUM,,,HGT_CM_CALC,Numeric,
143,Height_cm_DON,CALCULATED DONOR HEIGHT (CM),CALCULATED,NaT,NaT,,NUM,,,HGT_CM_DON_CALC,Numeric,
144,Height_cm_Registration_CAN,RECIPIENT HEIGHT @ REGISTRATION,TCR,1987-10-01,NaT,CLINICAL INFORMATION,NUM,,,HGT_CM_TCR,Numeric,
168,Height_cm_Listing_CAN,Calculated Candidate Height in CM at Listing,CALCULATED,NaT,NaT,,NUM,,,INIT_HGT_CM_CALC,Numeric,


### CITIZENSHIP

In [94]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'CITIZENSHIP', True)

                 count unique top   freq
CITIZENSHIP      30653      6   1  29604
CITIZENSHIP_DON  30522      6   1  27831

NaNs:
CITIZENSHIP         72
CITIZENSHIP_DON    203
dtype: int64

Datatypes:
CITIZENSHIP        str
CITIZENSHIP_DON    str
dtype: object


            Feature                           Description            FormSection DataType SASAnalysisFormat                                                                                                                                                   Comment Information
32      CITIZENSHIP  RECIPIENT CITIZENSHIP @ REGISTRATION  CANDIDATE INFORMATION      NUM           CITIZEN                                                                                                                                                               Unknown
33  CITIZENSHIP_DON                     DONOR CITIZENSHIP      DONOR INFORMATION      NUM                    This field uses separate SAS Analysis Format types for donor type. For deceased 

In [95]:
findMappingDfFlat(df.CITIZENSHIP_DON, df_flat, 'CITIZEN', 999)

Compare Length: 7 & 5

CODE                                                                           LABEL
   1                                                                      US Citizen
   2                                                                  RESIDENT ALIEN
   3                                                              NON-RESIDENT ALIEN
   4                                                      Non-US Citizen/US Resident
   5 Non-US Citizen/Non-US Resident, Traveled to US for Reason Other Than Transplant


In [96]:
df_flat[df_flat.FMTNAME == 'CITIZEN']

,LABEL,FMTNAME,TYPE,CODE
16750,Not Reported,CITIZEN,N,Null or Missing
16751,US Citizen,CITIZEN,N,1
16752,RESIDENT ALIEN,CITIZEN,N,2
16753,NON-RESIDENT ALIEN,CITIZEN,N,3
16754,Non-US Citizen/US Resident,CITIZEN,N,4
16755,"Non-US Citizen/Non-US Resident, Traveled to US...",CITIZEN,N,5
16756,"Non-US Citizen/Non-US Resident, Traveled to US...",CITIZEN,N,6
16757,Unknown,CITIZEN,N,**OTHER**


In [97]:
# df_flat FMTNAME: CITIZEN
mapping = {
    1: 'US Citizen',
    2: 'RESIDENT ALIEN',
    3: 'NON-RESIDENT ALIEN',
    4: 'Non-US Citizen/US Resident',
    5: 'Non-US Citizen/Non-US Resident, Traveled to US for Reason Other Than Transplant',
    6: 'Non-US Citizen/Non-US Resident, Traveled to US for Transplant',
    998: 'Unknown',
    999: 'Unknown'
}

# fill NaN with 998: Missing & convert to integer
df[features] = df[features].fillna(999).astype(int)

# mapping feature
df = uf.mapping_columns(df, 'CITIZENSHIP', mapping, display=True)
df = uf.mapping_columns(df, 'CITIZENSHIP_DON', mapping, display=True)

# mapping
colMap = {'CITIZENSHIP': 'Citizenship_CAN', 'CITIZENSHIP_DON':'Citizenship_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='Flat file - FMTNAME: CITIZEN')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column CITIZENSHIP Unique Vaue(s) ['US Citizen', 'Non-US Citizen/US Resident', 'RESIDENT ALIEN', 'Non-US Citizen/Non-US Resident, Traveled to U..., 'Non-US Citizen/Non-US Resident, Traveled to U..., 'NON-RESIDENT ALIEN', 'Unknown']
Categories (7, str): ['NON-RESIDENT ALIEN', 'Non-US Citizen/Non-US Resident, Traveled to U..., 'Non-US Citizen/Non-US Resident, Traveled to U..., 'Non-US Citizen/US Resident', 'RESIDENT ALIEN', 'US Citizen', 'Unknown']
Converted Column CITIZENSHIP_DON Unique Vaue(s) ['US Citizen', 'Non-US Citizen/US Resident', 'Non-US Citizen/Non-US Resident, Traveled to U..., 'Unknown', 'RESIDENT ALIEN', 'NON-RESIDENT ALIEN']
Categories (6, str): ['NON-RESIDENT ALIEN', 'Non-US Citizen/Non-US Resident, Traveled to U..., 'Non-US Citizen/US Resident', 'RESIDENT ALIEN', 'US Citizen', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
32,Citizenship_CAN,RECIPIENT CITIZENSHIP @ REGISTRATION,TCR,1987-10-01,NaT,CANDIDATE INFORMATION,NUM,CITIZEN,,CITIZENSHIP,Category,Flat file - FMTNAME: CITIZEN
33,Citizenship_DON,DONOR CITIZENSHIP,DDR/LDR,1987-10-01,NaT,DONOR INFORMATION,NUM,,This field uses separate SAS Analysis Format t...,CITIZENSHIP_DON,Category,Flat file - FMTNAME: CITIZEN


### STATE

In [98]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'STATE', True)

                count unique top  freq
PERM_STATE      30625     55  CA  3590
PERM_STATE_TRR  30244     54  CA  3565
HOME_STATE_DON  30392     54  CA  3163

NaNs:
PERM_STATE        100
PERM_STATE_TRR    481
HOME_STATE_DON    333
dtype: int64

Datatypes:
PERM_STATE        str
PERM_STATE_TRR    str
HOME_STATE_DON    str
dtype: object


            Feature                                   Description            FormSection DataType SASAnalysisFormat Comment Information
156  HOME_STATE_DON                                 DR HOME STATE      DONOR INFORMATION  CHAR(2)             STATE             Unknown
212      PERM_STATE  RECIPIENT STATE OF RESIDENCY  @ REGISTRATION  CANDIDATE INFORMATION  CHAR(2)             STATE             Unknown
213  PERM_STATE_TRR    RECIPIENT STATE OF RESIDENCY  @ TRANSPLANT  CANDIDATE INFORMATION  CHAR(2)             STATE             Unknown


PERM_STATE: <ArrowStringArray>
['CA', 'TX', 'NY', 'NC', 'KY', 'MA', 'MI', 'IL', 'MN', 'FL', 'MD', 'CT', 'GA',
 'PA', '

In [99]:
findMappingDfFlat(df.PERM_STATE, df_flat, 'STATE', 'XX')

Compare Length: 56 & 55

CODE             LABEL
  AK            ALASKA
  AL           ALABAMA
  AR          ARKANSAS
  AZ           ARIZONA
  CA        CALIFORNIA
  CO          COLORADO
  CT       CONNECTICUT
  DC DIST. OF COLUMBIA
  DE          DELAWARE
  FL           FLORIDA
  GA           GEORGIA
  GU              GUAM
  HI            HAWAII
  IA              IOWA
  ID             IDAHO
  IL          ILLINOIS
  IN           INDIANA
  KS            KANSAS
  KY          KENTUCKY
  LA         LOUISIANA
  MA     MASSACHUSETTS
  MD          MARYLAND
  ME             MAINE
  MI          MICHIGAN
  MN         MINNESOTA
  MO          MISSOURI
  MS       MISSISSIPPI
  MT           MONTANA
  NC    NORTH CAROLINA
  ND      NORTH DAKOTA
  NE          NEBRASKA
  NH     NEW HAMPSHIRE
  NJ        NEW JERSEY
  NM        NEW MEXICO
  NV            NEVADA
  NY          NEW YORK
  OH              OHIO
  OK          OKLAHOMA
  OR            OREGON
  PA      PENNSYLVANIA
  PR       PUERTO RICO
  RI     

In [100]:
# fill NaN with XX: Missing & convert to category data type
df[features] = df[features].fillna('Unknown')

# mapping
colMap = {'HOME_STATE_DON':'ResidencyState_DON', 'PERM_STATE': 'ResidencyStateRegistration_CAN', 'PERM_STATE_TRR':'ResidencyStateTransplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
156,ResidencyState_DON,DR HOME STATE,DDR/LDR,1987-10-01,NaT,DONOR INFORMATION,CHAR(2),STATE,,HOME_STATE_DON,Category,
212,ResidencyStateRegistration_CAN,RECIPIENT STATE OF RESIDENCY @ REGISTRATION,TCR,1997-10-01,NaT,CANDIDATE INFORMATION,CHAR(2),STATE,,PERM_STATE,Category,
213,ResidencyStateTransplant_CAN,RECIPIENT STATE OF RESIDENCY @ TRANSPLANT,TRR,2004-06-30,NaT,CANDIDATE INFORMATION,CHAR(2),STATE,,PERM_STATE_TRR,Category,


### EDUCATION

In [101]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'EDUCATION', True)

           count unique top   freq
EDUCATION  30651      7   3  11292

NaNs:
EDUCATION    74
dtype: int64

Datatypes:
EDUCATION    str
dtype: object


      Feature                                         Description            FormSection DataType SASAnalysisFormat Comment Information
97  EDUCATION  RECIPIENT HIGHEST EDUCATIONAL LEVEL @ REGISTRATION  CANDIDATE INFORMATION      NUM           EDLEVEL             Unknown


EDUCATION: <ArrowStringArray>
['4', '3', '5', '6', '2', '998', '1', nan]
Length: 8, dtype: str


In [102]:
findMappingDfFlat(df.EDUCATION, df_flat, 'EDLEVEL', 999)

Compare Length: 8 & 7

CODE                             LABEL
   1                              NONE
   2                GRADE SCHOOL (0-8)
   3         HIGH SCHOOL (9-12) or GED
   4 ATTENDED COLLEGE/TECHNICAL SCHOOL
   5         ASSOCIATE/BACHELOR DEGREE
   6      POST-COLLEGE GRADUATE DEGREE
 998                           UNKNOWN


In [103]:
# df_flat FMTNAME: EDLEVEL
mapping = {
    1: 'NONE',
    2: 'GRADE SCHOOL (0-8)',
    3: 'HIGH SCHOOL (9-12) or GED',
    4: 'ATTENDED COLLEGE/TECHNICAL SCHOOL',
    5: 'ASSOCIATE/BACHELOR DEGREE',
    6: 'POST-COLLEGE GRADUATE DEGREE',
    998: 'Unknown',
    999: 'Unknown'
}

# fill NaN with 998: Missing & covert to integer
df[features] = df[features].fillna(999).astype(int)

# mapping feature
df = uf.mapping_columns(df, 'EDUCATION', mapping, display=True)

# mapping
colMap = {'EDUCATION': 'EducationLevel_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='FMTNAME: EDLEVEL')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values())).copy()

# display
df_dict.loc[idx]

Converted Column EDUCATION Unique Vaue(s) ['ATTENDED COLLEGE/TECHNICAL SCHOOL', 'HIGH SCHOOL (9-12) or GED', 'ASSOCIATE/BACHELOR DEGREE', 'POST-COLLEGE GRADUATE DEGREE', 'GRADE SCHOOL (0-8)', 'Unknown', 'NONE']
Categories (7, str): ['ASSOCIATE/BACHELOR DEGREE', 'ATTENDED COLLEGE/TECHNICAL SCHOOL', 'GRADE SCHOOL (0-8)', 'HIGH SCHOOL (9-12) or GED', 'NONE', 'POST-COLLEGE GRADUATE DEGREE', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
97,EducationLevel_CAN,RECIPIENT HIGHEST EDUCATIONAL LEVEL @ REGISTRA...,TCR,1994-04-01,NaT,CANDIDATE INFORMATION,NUM,EDLEVEL,,EDUCATION,Category,FMTNAME: EDLEVEL


### Life Support

#### ECMO & IABP & INHALED & OTH_LIFE & PGE

In [104]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'ECMO|IABP|INHALED|OTH_LIFE|PGE', True)

                    count unique  top   freq      mean       std  min  25%  50%  75%  max
ECMO_TCR          30725.0    NaN  NaN    NaN  0.017705  0.131881  0.0  0.0  0.0  0.0  1.0
IABP_TCR          30725.0    NaN  NaN    NaN  0.081562  0.273701  0.0  0.0  0.0  0.0  1.0
INHALED_NO          30725      2    0  30662       NaN       NaN  NaN  NaN  NaN  NaN  NaN
PGE_TCR           30725.0    NaN  NaN    NaN  0.000651  0.025506  0.0  0.0  0.0  0.0  1.0
OTH_LIFE_SUP_TCR  30725.0    NaN  NaN    NaN  0.047323  0.212332  0.0  0.0  0.0  0.0  1.0
ECMO_TRR            30725      2    0  30011       NaN       NaN  NaN  NaN  NaN  NaN  NaN
PGE_TRR             30725      2    0  30638       NaN       NaN  NaN  NaN  NaN  NaN  NaN
IABP_TRR            30725      2    0  26645       NaN       NaN  NaN  NaN  NaN  NaN  NaN
OTH_LIFE_SUP_TRR    30725      2    0  28561       NaN       NaN  NaN  NaN  NaN  NaN  NaN
INHALED_NO_TRR      30725      2    0  30663       NaN       NaN  NaN  NaN  NaN  NaN  NaN
INHALED_NO

In [105]:
# fill NaN with 999: Missing & covert to integer
df[features] = df[features].fillna(999).astype(int)

# mapping
mapping = {
    0: 'No',
    1: 'Yes',
    999: 'Unknown'
}

for feature in features:
    # mapping feature
    df = uf.mapping_columns(df, feature, mapping, display=True)


# mapping
colMap = {'ECMO_TCR': 'LifeSupportRegistration_ECMO_CAN', 'ECMO_TRR':'LifeSupportTransplant_ECMO_CAN',
          'IABP_TCR': 'LifeSupportRegistration_IABP_CAN', 'IABP_TRR':'LifeSupportTransplant_IABP_CAN',
          'INHALED_NO_TCR': 'LifeSupportInhaledRegistration_CAN','INHALED_NO_TRR':'LifeSupportInhaledTransplant_CAN', 'INHALED_NO': 'LifeSupportInhaled_CAN',
          'OTH_LIFE_SUP_TCR': 'LifeSupportMechanismRegistration_OTHER_CAN', 'OTH_LIFE_SUP_TRR':'LifeSupportMechanismTransplant_OTHER_CAN',
          'PGE_TCR': 'LifeSupportRegistration_PGE_CAN', 'PGE_TRR':'LifeSupportTransplant_PGE_CAN'
         }

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column ECMO_TCR Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']
Converted Column IABP_TCR Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']
Converted Column INHALED_NO Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']
Converted Column PGE_TCR Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']
Converted Column OTH_LIFE_SUP_TCR Unique Vaue(s) ['Yes', 'No']
Categories (2, str): ['No', 'Yes']
Converted Column ECMO_TRR Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']
Converted Column PGE_TRR Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']
Converted Column IABP_TRR Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']
Converted Column OTH_LIFE_SUP_TRR Unique Vaue(s) ['Yes', 'No']
Categories (2, str): ['No', 'Yes']
Converted Column INHALED_NO_TRR Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']
Converted Column INHALED_NO_TCR Unique Vaue(s) ['No', 'Yes']
Categories (2, st

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
95,LifeSupportRegistration_ECMO_CAN,PATIENT ON LIFE SUPPORT - ECMO @ REGISTRATION,TCR,1995-04-01,NaT,CANDIDATE INFORMATION,NUM,,,ECMO_TCR,Category,
96,LifeSupportTransplant_ECMO_CAN,PATIENT ON LIFE SUPPORT - ECMO @ TRANSPLANT,TRR,1995-04-01,NaT,PATIENT STATUS,NUM,,,ECMO_TRR,Category,
157,LifeSupportRegistration_IABP_CAN,PATIENT ON LIFE SUPPORT - IABP @ REGISTRATION,TCR,1994-04-01,NaT,CANDIDATE INFORMATION,NUM,,,IABP_TCR,Category,
158,LifeSupportTransplant_IABP_CAN,PATIENT ON LIFE SUPPORT - IABP @ TRANSPLANT,TRR,1987-10-01,NaT,PATIENT STATUS,NUM,,,IABP_TRR,Category,
161,LifeSupportInhaled_CAN,CANDIDATE INHALED NO,WL,NaT,NaT,WAITING LIST DATA,NUM,,,INHALED_NO,Category,
162,LifeSupportInhaledRegistration_CAN,TCR Patient on Life Support://Inhaled NO,TCR,2004-06-30,NaT,CANDIDATE INFORMATION,NUM,,,INHALED_NO_TCR,Category,
163,LifeSupportInhaledTransplant_CAN,TRR Patient on Life Support://Inhaled NO,TRR,NaT,NaT,PATIENT STATUS,NUM,,,INHALED_NO_TRR,Category,
208,LifeSupportMechanismRegistration_OTHER_CAN,"OTHER MECHANISM OF LIFE Y/N, 1=Y @ REGISTRATION",TCR,1994-04-01,NaT,CANDIDATE INFORMATION,NUM,,,OTH_LIFE_SUP_TCR,Category,
209,LifeSupportMechanismTransplant_OTHER_CAN,"OTHER MECHANISM OF LIFE Y/N, 1=Y @ TRANSPLANT",TRR,1990-10-01,NaT,PATIENT STATUS,NUM,,,OTH_LIFE_SUP_TRR,Category,
214,LifeSupportRegistration_PGE_CAN,PATIENT ON LIFE SUPPORT: PGE @ REGISTRATION,TCR,1995-04-01,2004-06-30,CANDIDATE INFORMATION,NUM,,,PGE_TCR,Category,


### DROP FEATURES

#### PROS_INFUS_TCR & PROSTACYCLIN_TCR & PROS_INFUS_TRR & PROSTACYCLIN_TRR & LT_ONE_WEEK_DON & ORGAN & RECOV_OUT_US & DATA_TRANSPLANT & DATA_WAITLIST

In [106]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'PROS|PROSTACYCLIN|LT_ONE_WEEK_DON|ORGAN|RECOV_OUT_US|DATA', True)

                    count unique  top   freq mean  std  min  25%  50%  75%  max
PROS_INFUS_TCR    30725.0    NaN  NaN    NaN  0.0  0.0  0.0  0.0  0.0  0.0  0.0
PROSTACYCLIN_TCR  30725.0    NaN  NaN    NaN  0.0  0.0  0.0  0.0  0.0  0.0  0.0
PROS_INFUS_TRR      30725      1    0  30725  NaN  NaN  NaN  NaN  NaN  NaN  NaN
PROSTACYCLIN_TRR    30725      1    0  30725  NaN  NaN  NaN  NaN  NaN  NaN  NaN
ORGAN               30725      1   HR  30725  NaN  NaN  NaN  NaN  NaN  NaN  NaN
RECOV_OUT_US        30722      2    N  30679  NaN  NaN  NaN  NaN  NaN  NaN  NaN
LT_ONE_WEEK_DON     30725      1    N  30725  NaN  NaN  NaN  NaN  NaN  NaN  NaN
DATA_TRANSPLANT     30725      1    Y  30725  NaN  NaN  NaN  NaN  NaN  NaN  NaN
DATA_WAITLIST       30725      1    Y  30725  NaN  NaN  NaN  NaN  NaN  NaN  NaN

NaNs:
PROS_INFUS_TCR      0
PROSTACYCLIN_TCR    0
PROS_INFUS_TRR      0
PROSTACYCLIN_TRR    0
ORGAN               0
RECOV_OUT_US        3
LT_ONE_WEEK_DON     0
DATA_TRANSPLANT     0
DATA_WAITLIST    

In [107]:
# mapping
colMap = {}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Unknown', txt=f"{DROP} No Value ADDED.")

# drop columns
df = df.drop(columns=features).copy()

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
52,DATA_TRANSPLANT,Recipient TRR Data Reported,CALCULATED,NaT,NaT,,CHAR(1),,,DATA_TRANSPLANT,Unknown,** DROP ** No Value ADDED.
53,DATA_WAITLIST,Candidate WL Data Reported,CALCULATED,NaT,NaT,,CHAR(1),,,DATA_WAITLIST,Unknown,** DROP ** No Value ADDED.
197,LT_ONE_WEEK_DON,Donor Less Than 7 Days Old at Time of Donation,CALCULATED,2016-09-13,NaT,,CHAR(1),,,LT_ONE_WEEK_DON,Unknown,** DROP ** No Value ADDED.
207,ORGAN,ORGAN TYPE TRANSPLANTED,CALCULATED,NaT,NaT,,CHAR(2),,,ORGAN,Unknown,** DROP ** No Value ADDED.
232,PROS_INFUS_TCR,TCR CANDIDATE PROSTACYCLIN INFUSION,TCR,NaT,NaT,,NUM,,,PROS_INFUS_TCR,Unknown,** DROP ** No Value ADDED.
233,PROS_INFUS_TRR,TRR RECIPIENT PROSTACYCLIN INFUSION,TRR,NaT,NaT,PATIENT STATUS,NUM,,,PROS_INFUS_TRR,Unknown,** DROP ** No Value ADDED.
234,PROSTACYCLIN_TCR,TCR CANDIDATE PROSTACYCLIN INHALATION,TCR,NaT,NaT,,NUM,,,PROSTACYCLIN_TCR,Unknown,** DROP ** No Value ADDED.
235,PROSTACYCLIN_TRR,TRR RECIPIENT PROSTACYCLIN INHALATION,TRR,2004-06-30,NaT,PATIENT STATUS,NUM,,,PROSTACYCLIN_TRR,Unknown,** DROP ** No Value ADDED.
258,RECOV_OUT_US,ORGAN RECOVERED OUTSIDE U.S.,LDR,1999-10-25,NaT,ORGAN RECOVERY,CHAR(1),,,RECOV_OUT_US,Unknown,** DROP ** No Value ADDED.


### _FLG
- INIT_LLU_FLG & INIT_RLU_FLG & INIT_BLU_FLG & END_LLU_FLG & END_RLU_FLG & END_BLU_FLG

In [108]:
# display feature info
features, idx = uf.feature_information(df, df_dict, '_FLG', True)

              count unique top   freq
INIT_LLU_FLG  30725      1   0  30725
INIT_RLU_FLG  30725      1   0  30725
INIT_BLU_FLG  30725      1   0  30725
END_LLU_FLG   30725      1   0  30725
END_RLU_FLG   30725      1   0  30725
END_BLU_FLG   30725      1   0  30725

NaNs:
INIT_LLU_FLG    0
INIT_RLU_FLG    0
INIT_BLU_FLG    0
END_LLU_FLG     0
END_RLU_FLG     0
END_BLU_FLG     0
dtype: int64

Datatypes:
INIT_LLU_FLG    str
INIT_RLU_FLG    str
INIT_BLU_FLG    str
END_LLU_FLG     str
END_RLU_FLG     str
END_BLU_FLG     str
dtype: object


          Feature                                                 Description        FormSection DataType SASAnalysisFormat Comment Information
98    END_BLU_FLG   LUNG PREFERENCE AT REMOVAL/CURRENT TIME/ TCR - BOTH (1=Y)  WAITING LIST DATA      NUM                               Unknown
102   END_LLU_FLG   LUNG PREFERENCE AT REMOVAL/CURRENT TIME/ TCR - LEFT (1=Y)  WAITING LIST DATA      NUM                               Unknown
103   END_RLU_FLG  LUNG PR

In [109]:
# mapping
colMap = {}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Unknown', txt=f"{DROP} No Value ADDED.")

# drop columns
df = df.drop(columns=features).copy()

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
98,END_BLU_FLG,LUNG PREFERENCE AT REMOVAL/CURRENT TIME/ TCR -...,WAITING LIST DATA,1995-03-01,NaT,WAITING LIST DATA,NUM,,,END_BLU_FLG,Unknown,** DROP ** No Value ADDED.
102,END_LLU_FLG,LUNG PREFERENCE AT REMOVAL/CURRENT TIME/ TCR -...,WAITING LIST DATA,1995-03-01,NaT,WAITING LIST DATA,NUM,,,END_LLU_FLG,Unknown,** DROP ** No Value ADDED.
103,END_RLU_FLG,LUNG PREFERENCE AT REMOVAL/CURRENT TIME/ TCR -...,WAITING LIST DATA,1995-03-01,NaT,WAITING LIST DATA,NUM,,,END_RLU_FLG,Unknown,** DROP ** No Value ADDED.
165,INIT_BLU_FLG,LUNG PREFERENCE AT LISTING - BOTH (1=Y),WAITING LIST DATA,1995-03-01,NaT,WAITING LIST DATA,NUM,,,INIT_BLU_FLG,Unknown,** DROP ** No Value ADDED.
169,INIT_LLU_FLG,LUNG PREFERENCE AT LISTING - LEFT (1=Y),WAITING LIST DATA,1995-03-01,NaT,WAITING LIST DATA,NUM,,,INIT_LLU_FLG,Unknown,** DROP ** No Value ADDED.
170,INIT_RLU_FLG,LUNG PREFERENCE AT LISTING - RIGHT (1=Y),WAITING LIST DATA,1995-03-01,NaT,WAITING LIST DATA,NUM,,,INIT_RLU_FLG,Unknown,** DROP ** No Value ADDED.


### INOTROPES

In [110]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'INOTROPES', True)

                 count unique  top   freq      mean       std  min  25%  50%  75%  max
INOTROPES_TCR  30725.0    NaN  NaN    NaN  0.331326  0.470698  0.0  0.0  0.0  1.0  1.0
INOTROPES_TRR    30725      2    0  19327       NaN       NaN  NaN  NaN  NaN  NaN  NaN

NaNs:
INOTROPES_TCR    0
INOTROPES_TRR    0
dtype: int64

Datatypes:
INOTROPES_TCR    int64
INOTROPES_TRR      str
dtype: object


           Feature                  Description            FormSection DataType SASAnalysisFormat Comment Information
185  INOTROPES_TCR  IV INOTROPES @ REGISTRATION  CANDIDATE INFORMATION      NUM                               Unknown
186  INOTROPES_TRR    IV INOTROPES @ TRANSPLANT         PATIENT STATUS      NUM                               Unknown


INOTROPES_TCR: [0 1]
INOTROPES_TRR: <ArrowStringArray>
['0', '1']
Length: 2, dtype: str


In [111]:
# fill NaN with 999: Missing & covert to integer
df[features] = df[features].fillna(999).astype(int)

# mapping
mapping = {
    0: 'No',
    1: 'Yes',
    999: 'Unknown'
}

for feature in features:
    # mapping feature
    df = uf.mapping_columns(df, feature, mapping, display=True)

    
# mapping
colMap = {'INOTROPES_TCR': 'InotropesIVRegistration_CAN', 'INOTROPES_TRR':'InotropesIVTransplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column INOTROPES_TCR Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']
Converted Column INOTROPES_TRR Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
185,InotropesIVRegistration_CAN,IV INOTROPES @ REGISTRATION,TCR,1994-04-01,NaT,CANDIDATE INFORMATION,NUM,,,INOTROPES_TCR,Category,
186,InotropesIVTransplant_CAN,IV INOTROPES @ TRANSPLANT,TRR,1994-04-01,NaT,PATIENT STATUS,NUM,,,INOTROPES_TRR,Category,


### VAD_DEVICE

In [112]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'VAD_DEVICE', True)

                   count unique top   freq
VAD_DEVICE_TY_TCR  30652      5   1  22087
VAD_DEVICE_TY_TRR  30325      5   1  17492

NaNs:
VAD_DEVICE_TY_TCR     73
VAD_DEVICE_TY_TRR    400
dtype: int64

Datatypes:
VAD_DEVICE_TY_TCR    str
VAD_DEVICE_TY_TRR    str
dtype: object


               Feature                              Description            FormSection DataType SASAnalysisFormat Comment       Information
290  VAD_DEVICE_TY_TCR  CANDIDATE TYPE OF VAD DEVICE AT LISTING  CANDIDATE INFORMATION      NUM          VADDEVTY          ddmonthyyyy:time
291  VAD_DEVICE_TY_TRR                      TRR VAD DEVICE TYPE         PATIENT STATUS      NUM          VADDEVTY                   Unknown


VAD_DEVICE_TY_TCR: <ArrowStringArray>
['5', '2', '1', '4', '3', nan]
Length: 6, dtype: str
VAD_DEVICE_TY_TRR: <ArrowStringArray>
['5', '2', '1', '4', '3', nan]
Length: 6, dtype: str


In [113]:
findMappingDfFlat(df.VAD_DEVICE_TY_TCR, df_flat, 'VADDEVTY', NaN=998)

Compare Length: 6 & 5

CODE     LABEL
   1      NONE
   2      LVAD
   3      RVAD
   4       TAH
   5 LVAD+RVAD


In [114]:
# fill NaN with 998: Missing & convert to integer
df[features] = df[features].fillna(998).astype(int)

# df_flat FMTNAME:  VADDEVTY
mapping = {
    1: 'None',
    2: 'Lvad',
    3: 'Rvad',
    4: 'Tah',
    5: 'Lvad+Rvad',
    6: 'Lvad/Rvad/Tah unspecified',
    998: 'Unknown'
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'VAD_DEVICE_TY_TCR': 'VentricularDeviceTypeRegistration_CAN', 'VAD_DEVICE_TY_TRR':'VentricularDeviceTypeTransplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Object', txt='FMTNAME: VADDEVTY - Type of Ventricular Device used.')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column VAD_DEVICE_TY_TCR Unique Vaue(s) ['Lvad+Rvad', 'Lvad', 'None', 'Tah', 'Rvad', 'Unknown']
Categories (6, str): ['Lvad', 'Lvad+Rvad', 'None', 'Rvad', 'Tah', 'Unknown']
Converted Column VAD_DEVICE_TY_TRR Unique Vaue(s) ['Lvad+Rvad', 'Lvad', 'None', 'Tah', 'Rvad', 'Unknown']
Categories (6, str): ['Lvad', 'Lvad+Rvad', 'None', 'Rvad', 'Tah', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
290,VentricularDeviceTypeRegistration_CAN,CANDIDATE TYPE OF VAD DEVICE AT LISTING,TCR,2004-06-30,NaT,CANDIDATE INFORMATION,NUM,VADDEVTY,,VAD_DEVICE_TY_TCR,Object,FMTNAME: VADDEVTY - Type of Ventricular Device...
291,VentricularDeviceTypeTransplant_CAN,TRR VAD DEVICE TYPE,TRR,2004-06-30,NaT,PATIENT STATUS,NUM,VADDEVTY,,VAD_DEVICE_TY_TRR,Object,FMTNAME: VADDEVTY - Type of Ventricular Device...


### VAD_BRAND

In [115]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'VAD_BRAND', True)

                count unique  top  freq
VAD_BRAND1_TCR   8565     39  205  4433
VAD_BRAND1_TRR  12825     42  205  6058

NaNs:
VAD_BRAND1_TCR    22160
VAD_BRAND1_TRR    17900
dtype: int64

Datatypes:
VAD_BRAND1_TCR    str
VAD_BRAND1_TRR    str
dtype: object


            Feature                       Description            FormSection DataType SASAnalysisFormat Comment       Information
288  VAD_BRAND1_TCR  CANDIDATE VAD BRAND 1 AT LISTING  CANDIDATE INFORMATION      NUM                            ddmonthyyyy:time
289  VAD_BRAND1_TRR      TRR LIFE SUPPORT VAD BRAND 1         PATIENT STATUS      NUM          VADBRAND          ddmonthyyyy:time


VAD_BRAND1_TCR: <ArrowStringArray>
['227', '205',   nan, '402', '221', '231', '224', '215', '999', '325', '210',
 '226', '208', '320', '228', '225', '209', '217', '313', '230', '316', '236',
 '318', '223', '237', '311', '235', '321', '238', '332', '329', '330', '216',
 '201', '232', '218', '319', '309', '204', '222']
Length: 40, dtype: str
VAD_BR

In [116]:
findMappingDfFlat(df.VAD_BRAND1_TCR, df_flat, 'VADBRAND', NaN=998)

Compare Length: 40 & 39

CODE                           LABEL
 201                Abiomed BVS 5000
 204                      Biomedicus
 205                    Heartmate II
 208                   Heartmate XVE
 209                  Heartsaver VAD
 210                     Jarvik 2000
 215     Cardiac Assist Tandem Heart
 216                        Thoratec
 217                   Thoratec IVAD
 218                          Toyobo
 221                  Abiomed AB5000
 222              Berlin Heart EXCOR
 223                        Evaheart
 224                  Heartware HVAD
 225             Impella Recover 2.5
 226             Impella Recover 5.0
 227 CentriMag (Thoratec/Levitronix)
 228          Maquet Jostra Rotaflow
 230                Terumo DuraHeart
 231                   Thoratec PVAD
 232           Ventracor VentrAssist
 235       Cardiac Assist Protek Duo
 236                   HeartMate III
 237                      Impella CP
 238                      Impella RP
 309         

In [117]:
findMappingDfFlat(df.VAD_BRAND1_TRR, df_flat, 'VADBRAND', NaN=998)

Compare Length: 43 & 42

CODE                           LABEL
 201                Abiomed BVS 5000
 204                      Biomedicus
 205                    Heartmate II
 208                   Heartmate XVE
 209                  Heartsaver VAD
 210                     Jarvik 2000
 215     Cardiac Assist Tandem Heart
 216                        Thoratec
 217                   Thoratec IVAD
 218                          Toyobo
 221                  Abiomed AB5000
 222              Berlin Heart EXCOR
 223                        Evaheart
 224                  Heartware HVAD
 225             Impella Recover 2.5
 226             Impella Recover 5.0
 227 CentriMag (Thoratec/Levitronix)
 228          Maquet Jostra Rotaflow
 230                Terumo DuraHeart
 231                   Thoratec PVAD
 232           Ventracor VentrAssist
 233              Worldheart Levacor
 235       Cardiac Assist Protek Duo
 236                   HeartMate III
 237                      Impella CP
 305         

In [118]:
# check for differences between two sets
uf.symmetric_difference(set(df.VAD_BRAND1_TCR.dropna().unique().astype(int)), set(df.VAD_BRAND1_TRR.dropna().unique().astype(int)))

Symmetric difference: [np.int64(233), np.int64(238), np.int64(305), np.int64(317), np.int64(330), np.int64(331), np.int64(401)]


In [119]:
# fill NaN with 998: Missing & convert to integer
df[features] = df[features].fillna(998).astype(int)

# df_flat FMTNAME: VADBRAND
mapping = {
    201: "Abiomed BVS 5000",
    204: "Biomedicus",
    205: "Heartmate II",
    208: "Heartmate XVE",
    209: "Heartsaver VAD",
    210: "Jarvik 2000",
    215: "Cardiac Assist Tandem Heart",
    216: "Thoratec",
    217: "Thoratec IVAD",
    218: "Toyobo",
    221: "Abiomed AB5000",
    222: "Berlin Heart EXCOR",
    223: "Evaheart",
    224: "Heartware HVAD",
    225: "Impella Recover 2.5",
    226: "Impella Recover 5.0",
    227: "CentriMag (Thoratec/Levitronix)",
    228: "Maquet Jostra Rotaflow",
    230: "Terumo DuraHeart",
    231: "Thoratec PVAD",
    232: "Ventracor VentrAssist",
    233: "Worldheart Levacor",
    235: "Cardiac Assist Protek Duo",
    236: "HeartMate III",
    237: "Impella CP",
    238: "Impella RP",
    305: "Thoratec",
    309: "Abiomed AB5000",
    311: "Cardiac Assist Tandem Heart",
    313: "Heartmate II",
    316: "Heartware HVAD",
    317: "Impella Recover 2.5",
    318: "Impella Recover 5.0",
    319: "Jarvik 2000",
    320: "CentriMag (Thoratec/Levitronix)",
    321: "Maquet Jostra Rotaflow",
    325: "Thoratec PVAD",
    329: "Cardiac Assist Protek Duo",
    330: "HeartMate III",
    331: "Impella CP",
    332: "Impella RP",
    401: "AbioCor",
    402: "SynCardia CardioWest",
    999: "Other, Specify",
    998: "Unknown"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=False)


# mapping
colMap = {'VAD_BRAND1_TCR': 'VentricularDeviceBrandRegistration_CAN', 'VAD_BRAND1_TRR':'VentricularDeviceBrandTransplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Object', txt='FMTNAME: VADBRAND - Type of Ventricular Device Brand used.')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
288,VentricularDeviceBrandRegistration_CAN,CANDIDATE VAD BRAND 1 AT LISTING,TCR,2004-06-30,NaT,CANDIDATE INFORMATION,NUM,,,VAD_BRAND1_TCR,Object,FMTNAME: VADBRAND - Type of Ventricular Device...
289,VentricularDeviceBrandTransplant_CAN,TRR LIFE SUPPORT VAD BRAND 1,TRR,2004-06-30,NaT,PATIENT STATUS,NUM,VADBRAND,,VAD_BRAND1_TRR,Object,FMTNAME: VADBRAND - Type of Ventricular Device...


### FUNC_STAT

In [120]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'FUNC', True)

               count unique   top  freq
FUNC_STAT_TCR  30650     24  2020  6905
FUNC_STAT_TRR  30315     11  2020  8635
FUNC_STAT_TRF  18707     11  2100  4330

NaNs:
FUNC_STAT_TCR       75
FUNC_STAT_TRR      410
FUNC_STAT_TRF    12018
dtype: int64

Datatypes:
FUNC_STAT_TCR    str
FUNC_STAT_TRR    str
FUNC_STAT_TRF    str
dtype: object


           Feature                                 Description                          FormSection DataType SASAnalysisFormat Comment Information
110  FUNC_STAT_TCR  RECIPIENT FUNCTIONAL STATUS @ REGISTRATION                CANDIDATE INFORMATION      NUM          FUNCSTAT             Unknown
111  FUNC_STAT_TRF             TRF FUNCTIONAL STATUS @ TRR/FOL  PATIENT STATUS AT TIME OF FOLLOW-UP      NUM          FUNCSTAT             Unknown
112  FUNC_STAT_TRR     RECIPIENT FUNCTIONAL STATUS @TRANSPLANT                       PATIENT STATUS      NUM          FUNCSTAT             Unknown


FUNC_STAT_TCR: <ArrowStringArray>
['2010', '2020', '2090', '2050', '20

In [121]:
findMappingDfFlat(df.FUNC_STAT_TCR, df_flat, 'FUNCSTAT', NaN=999)

Compare Length: 25 & 24

CODE                                                                                               LABEL
   1                                             Performs activities of daily living with NO assistance.
   2                                           Performs activities of daily living with SOME assistance.
 996                                                               Not Applicable (patient < 1 year old)
 998                                                                                             Unknown
2010                                                 10% - Moribund, fatal processes progressing rapidly
2020                              20% - Very sick, hospitalization necessary: active treatment necessary
2030                           30% - Severely disabled: hospitalization is indicated, death not imminent
2040                                                40% - Disabled: requires special care and assistance
2050                          

In [122]:
findMappingDfFlat(df.FUNC_STAT_TRR, df_flat, 'FUNCSTAT', NaN=999)

Compare Length: 12 & 11

CODE                                                                     LABEL
 998                                                                   Unknown
2010                       10% - Moribund, fatal processes progressing rapidly
2020    20% - Very sick, hospitalization necessary: active treatment necessary
2030 30% - Severely disabled: hospitalization is indicated, death not imminent
2040                      40% - Disabled: requires special care and assistance
2050          50% - Requires considerable assistance and frequent medical care
2060        60% - Requires occasional assistance but is able to care for needs
2070   70% - Cares for self: unable to carry on normal activity or active work
2080               80% - Normal activity with effort: some symptoms of disease
2090         90% - Able to carry on normal activity: minor symptoms of disease
2100                      100% - Normal, no complaints, no evidence of disease


In [123]:
findMappingDfFlat(df.FUNC_STAT_TRF, df_flat, 'FUNCSTAT', NaN=999)

Compare Length: 12 & 11

CODE                                                                     LABEL
 998                                                                   Unknown
2010                       10% - Moribund, fatal processes progressing rapidly
2020    20% - Very sick, hospitalization necessary: active treatment necessary
2030 30% - Severely disabled: hospitalization is indicated, death not imminent
2040                      40% - Disabled: requires special care and assistance
2050          50% - Requires considerable assistance and frequent medical care
2060        60% - Requires occasional assistance but is able to care for needs
2070   70% - Cares for self: unable to carry on normal activity or active work
2080               80% - Normal activity with effort: some symptoms of disease
2090         90% - Able to carry on normal activity: minor symptoms of disease
2100                      100% - Normal, no complaints, no evidence of disease


In [124]:
# fill NaN with 998: Missing & convert to integer
df[features] = df[features].fillna(998).astype(int)

# df_flat FMTNAME: FUNCSTAT
mapping = {
    1: "Performs activities of daily living with NO assistance",
    2: "Performs activities of daily living with SOME assistance",
    996: "Not Applicable (patient < 1 year old)",
    999: "Unknown",
    998: "Unknown",
    2010: "10% - Moribund, fatal processes progressing rapidly",
    2020: "20% - Very sick, hospitalization necessary: active treatment necessary",
    2030: "30% - Severely disabled: hospitalization is indicated, death not imminent",
    2040: "40% - Disabled: requires special care and assistance",
    2050: "50% - Requires considerable assistance and frequent medical care",
    2060: "60% - Requires occasional assistance but is able to care for needs",
    2070: "70% - Cares for self: unable to carry on normal activity or active work",
    2080: "80% - Normal activity with effort: some symptoms of disease",
    2090: "90% - Able to carry on normal activity: minor symptoms of disease",
    2100: "100% - Normal, no complaints, no evidence of disease",
    4010: "10% - No play; does not get out of bed",
    4020: "20% - Often sleeping; play entirely limited to very passive activities",
    4030: "30% - In bed; needs assistance even for quiet play",
    4040: "40% - Mostly in bed; participates in quiet activities",
    4050: "50% - Can dress but lies around much of day; no active play; can take part in quiet play/activities",
    4060: "60% - Up and around, but minimal active play; keeps busy with quieter activities",
    4070: "70% - Both greater restriction of and less time spent in play activity",
    4080: "80% - Active, but tires more quickly",
    4090: "90% - Minor restrictions in physically strenuous activity",
    4100: "100% - Fully active, normal"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=False)
    

# mapping
colMap = {'FUNC_STAT_TCR': 'FunctionalStatusRegistration_CAN', 'FUNC_STAT_TRF':'FunctionalStatusFollowUp', 'FUNC_STAT_TRR':'FunctionalStatusTransplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='FMTNAME: FUNCSTAT')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
110,FunctionalStatusRegistration_CAN,RECIPIENT FUNCTIONAL STATUS @ REGISTRATION,TCR,1994-04-01,NaT,CANDIDATE INFORMATION,NUM,FUNCSTAT,,FUNC_STAT_TCR,Category,FMTNAME: FUNCSTAT
111,FunctionalStatusFollowUp,TRF FUNCTIONAL STATUS @ TRR/FOL,TRR/TRF,1997-04-01,NaT,PATIENT STATUS AT TIME OF FOLLOW-UP,NUM,FUNCSTAT,,FUNC_STAT_TRF,Category,FMTNAME: FUNCSTAT
112,FunctionalStatusTransplant_CAN,RECIPIENT FUNCTIONAL STATUS @TRANSPLANT,TRR,1994-04-01,NaT,PATIENT STATUS,NUM,FUNCSTAT,,FUNC_STAT_TRR,Category,FMTNAME: FUNCSTAT


### PRI_PAYMENT

In [125]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'PRI_PAYMENT', True)

                 count unique top   freq
PRI_PAYMENT_TCR  30653     13   1  15777
PRI_PAYMENT_TRR  30318     11   1  14428

NaNs:
PRI_PAYMENT_TCR     72
PRI_PAYMENT_TRR    407
dtype: int64

Datatypes:
PRI_PAYMENT_TCR    str
PRI_PAYMENT_TRR    str
dtype: object


             Feature                                              Description            FormSection DataType SASAnalysisFormat Comment Information
224  PRI_PAYMENT_TCR  RECIPIENT PRIMARY PROJECTED PAYMENT TYPE @ REGISTRATION  CANDIDATE INFORMATION      NUM           PRIMPAY             Unknown
225  PRI_PAYMENT_TRR            RECIPIENT PRIMARY PAYMENT SOURCE @ TRANSPLANT         PATIENT STATUS      NUM           PRIMPAY             Unknown


PRI_PAYMENT_TCR: <ArrowStringArray>
['2', '3', '1', '4', '6', '7', '10', '11', '12', '8', '5', '9', nan, '13']
Length: 14, dtype: str
PRI_PAYMENT_TRR: <ArrowStringArray>
['2', '3', '1', '4', '6', '7', '10', '12', '8', '5', '9', nan]
Length: 12, dtype: str


In [126]:
findMappingDfFlat(df.PRI_PAYMENT_TCR, df_flat, 'PRIMPAY', NaN=999)

Compare Length: 14 & 13

CODE                                                         LABEL
   1                                             Private insurance
   2                                   Public insurance - Medicaid
   3             Public insurance - Medicare FFS (Fee for Service)
   4                          Public insurance - Medicare & Choice
   5 Public insurance - CHIP (Children's Health Insurance Program)
   6                           Public insurance - Department of VA
   7                           Public insurance - Other government
   8                                                          Self
   9                                                      Donation
  10                                                     Free Care
  11                                                       Pending
  12                                    Foreign Government Specify
  13    Public insurance - Medicare (further detail not collected)


In [127]:
findMappingDfFlat(df.PRI_PAYMENT_TRR, df_flat, 'PRIMPAY', NaN=999)

Compare Length: 12 & 11

CODE                                                         LABEL
   1                                             Private insurance
   2                                   Public insurance - Medicaid
   3             Public insurance - Medicare FFS (Fee for Service)
   4                          Public insurance - Medicare & Choice
   5 Public insurance - CHIP (Children's Health Insurance Program)
   6                           Public insurance - Department of VA
   7                           Public insurance - Other government
   8                                                          Self
   9                                                      Donation
  10                                                     Free Care
  12                                    Foreign Government Specify


In [128]:
# fill NaN with 999: Missing & convert to integer
df[features] = df[features].fillna(999).astype(int)

# df_flat FMTNAME: PRIMPAY
mapping = {
    1: "Private insurance",
    2: "Public insurance - Medicaid",
    3: "Public insurance - Medicare FFS (Fee for Service)",
    4: "Public insurance - Medicare & Choice",
    5: "Public insurance - CHIP (Children's Health Insurance Program)",
    6: "Public insurance - Department of VA",
    7: "Public insurance - Other government",
    8: "Self",
    9: "Donation",
    10: "Free Care",
    11: "Pending",
    12: "Foreign Government Specify",
    13: "Public insurance - Medicare (further detail not collected)",
    999: "Missing"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)
    

# mapping
colMap = {'PRI_PAYMENT_TCR': 'PrimaryPaymentRegistration_CAN', 'PRI_PAYMENT_TRR':'PrimaryPaymentTransplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Object', txt='FMTNAME: PRIMPAY')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column PRI_PAYMENT_TCR Unique Vaue(s) ['Public insurance - Medicaid', 'Public insurance - Medicare FFS (Fee for Serv..., 'Private insurance', 'Public insurance - Medicare & Choice', 'Public insurance - Department of VA', ..., 'Self', 'Public insurance - CHIP (Children's Health In..., 'Donation', 'Missing', 'Public insurance - Medicare (further detail n...]
Length: 14
Categories (14, str): ['Donation', 'Foreign Government Specify', 'Free Care', 'Missing', ..., 'Public insurance - Medicare (further detail n..., 'Public insurance - Medicare FFS (Fee for Serv..., 'Public insurance - Other government', 'Self']
Converted Column PRI_PAYMENT_TRR Unique Vaue(s) ['Public insurance - Medicaid', 'Public insurance - Medicare FFS (Fee for Serv..., 'Private insurance', 'Public insurance - Medicare & Choice', 'Public insurance - Department of VA', ..., 'Foreign Government Specify', 'Self', 'Public insurance - CHIP (Children's Health In..., 'Donation', 'Missing']
Length: 12
Categories (12, st

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
224,PrimaryPaymentRegistration_CAN,RECIPIENT PRIMARY PROJECTED PAYMENT TYPE @ REG...,TCR,1994-04-01,NaT,CANDIDATE INFORMATION,NUM,PRIMPAY,,PRI_PAYMENT_TCR,Object,FMTNAME: PRIMPAY
225,PrimaryPaymentTransplant_CAN,RECIPIENT PRIMARY PAYMENT SOURCE @ TRANSPLANT,TRR,1994-04-01,NaT,PATIENT STATUS,NUM,PRIMPAY,,PRI_PAYMENT_TRR,Object,FMTNAME: PRIMPAY


### DIAB

In [129]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DIAB', True)

                   count unique top   freq
DIAB               30652      6   1  21848
HIST_DIABETES_DON  30721      6   1  29393
DIABETES_DON       30721      3   N  29393

NaNs:
DIAB                 73
HIST_DIABETES_DON     4
DIABETES_DON          4
dtype: int64

Datatypes:
DIAB                 str
HIST_DIABETES_DON    str
DIABETES_DON         str
dtype: object


               Feature                                                    Description           FormSection DataType SASAnalysisFormat                                    Comment Information
72                DIAB                              RECIPIENT DIABETES @ REGISTRATION  CLINICAL INFORMATION      NUM            DIABTY                                                Unknown
73        DIABETES_DON                       DECEASED DONOR-HISTORY OF DIABETES (Y,N)         DONOR HISTORY  CHAR(1)                    UNROLLED Y/N FIELD FROM HIST_DIABETES_DON     Unknown
148  HIST_DIABETES_DON  DECEASED DONOR-HISTORY OF DIABETES, INC

In [130]:
findMappingDfFlat(df.DIAB, df_flat, 'DIABTY', NaN=999)

Compare Length: 7 & 6

CODE                   LABEL
   1                      No
   2                  Type I
   3                 Type II
   4              Type Other
   5            Type Unknown
 998 Diabetes Status Unknown


In [131]:
findMappingDfFlat(df.DIAB, df_flat, 'HISTDIAB', NaN=999)

Compare Length: 7 & 6

CODE                 LABEL
   1                    NO
   2        YES, 0-5 YEARS
   3       YES, 6-10 YEARS
   4        YES, >10 YEARS
   5 YES, DURATION UNKNOWN
 998               UNKNOWN


In [132]:
# update NaNs
df.DIAB = df.DIAB.fillna(999).astype(int) # 999 Missing
df.DIABETES_DON = df.DIABETES_DON.fillna('Unknown') # X Missing
df.HIST_DIABETES_DON = df.HIST_DIABETES_DON.fillna(999).astype(int) # 999 Missing

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# mapping feature
df = uf.mapping_columns(df, 'DIABETES_DON', mapping)


# df_flat FMTNAME: DIABTY
mapping = {
    1: 'No',
    2: 'Type I',
    3: 'Type II',
    4: 'Type Other',
    5: 'Type Unknown',
    998: 'Diabetes Status Unknown',
    999: 'Unknown'
}

# map
df = uf.mapping_columns(df, 'DIAB', mapping)


# df_flat FMTNAME: HISTDIAB
mapping = {
    1: 'No',
    2: 'Yes, 0-5 Years',
    3: 'Yes, 6-10 Years',
    4: 'Yes, >10 Years',
    5: 'Yes, Duration Unknown',
    998: 'Unknown',
    999: 'Unknown'
}

# map
df = uf.mapping_columns(df, 'HIST_DIABETES_DON', mapping)

# mapping
colMap = {'DIAB': 'DiabetesType_CAN', 'DIABETES_DON':'Diabetes_DON', 'HIST_DIABETES_DON':'DiabetesHistory_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='')
df_dict = uf.update_dictionary_information(df_dict, [70], txt='FMTNAME: DIABTY')
df_dict = uf.update_dictionary_information(df_dict, [146], txt='FMTNAME: HISTDIAB').copy()

# convert to category
df = uf.convert_to_category(df,  list(colMap.values())).copy()

# display
df_dict.loc[idx]

Converted Column DIABETES_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column DIAB Unique Vaue(s) ['No', 'Type II', 'Type I', 'Type Unknown', 'Type Other', 'Diabetes Status Unknown', 'Unknown']
Categories (7, str): ['Diabetes Status Unknown', 'No', 'Type I', 'Type II', 'Type Other', 'Type Unknown', 'Unknown']
Converted Column HIST_DIABETES_DON Unique Vaue(s) ['No', 'Yes, 0-5 Years', 'Yes, >10 Years', 'Yes, 6-10 Years', 'Unknown', 'Yes, Duration Unknown']
Categories (6, str): ['No', 'Unknown', 'Yes, 0-5 Years', 'Yes, 6-10 Years', 'Yes, >10 Years', 'Yes, Duration Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
72,DiabetesType_CAN,RECIPIENT DIABETES @ REGISTRATION,TCR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,DIABTY,,DIAB,Category,
73,Diabetes_DON,"DECEASED DONOR-HISTORY OF DIABETES (Y,N)",DDR,1994-04-01,NaT,DONOR HISTORY,CHAR(1),,UNROLLED Y/N FIELD FROM HIST_DIABETES_DON,DIABETES_DON,Category,
148,DiabetesHistory_DON,"DECEASED DONOR-HISTORY OF DIABETES, INCL. DURA...",DDR,1994-04-01,NaT,DONOR HISTORY,NUM,HISTDIAB,,HIST_DIABETES_DON,Category,


### DIAL

In [133]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DIAL_', True)

                 count unique top   freq
DIAL_TY_TCR      30650      5   1  29765
DIAL_AFTER_LIST  30302      3   N  28919
DIAL_PRIOR_TX    30282      3   N  28657

NaNs:
DIAL_TY_TCR         75
DIAL_AFTER_LIST    423
DIAL_PRIOR_TX      443
dtype: int64

Datatypes:
DIAL_TY_TCR        str
DIAL_AFTER_LIST    str
DIAL_PRIOR_TX      str
dtype: object


            Feature                                        Description                         FormSection DataType SASAnalysisFormat                                                   Comment Information
75  DIAL_AFTER_LIST  DIALYSIS OCCURRING BETWEEN LISTING AND TRANSPLANT  PRETRANSPLANT CLINICAL INFORMATION  CHAR(1)                                                                                 Unknown
76    DIAL_PRIOR_TX                Calculated: Ever Dialysis Prior Tx?                                      CHAR(1)                                                                                 Unknown
77      DIAL_TY_TCR            PATIENT

In [134]:
findMappingDfFlat(df.DIAL_TY_TCR, df_flat, 'DIAL_TY', NaN=997)

Compare Length: 6 & 5

CODE                               LABEL
   1                         No dialysis
   2                        Hemodialysis
   3                 Peritoneal Dialysis
 998             Dialysis Status Unknown
 999 Dialysis-Unknown Type was performed


In [135]:
# update NaNs
df[['DIAL_AFTER_LIST','DIAL_PRIOR_TX']] = df[['DIAL_AFTER_LIST','DIAL_PRIOR_TX']].fillna('Unknown')
df.DIAL_TY_TCR = df.DIAL_TY_TCR.fillna(997).astype(int) # 997 Missing

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# mapping feature
df = uf.mapping_columns(df, 'DIAL_AFTER_LIST', mapping, display=True)
df = uf.mapping_columns(df, 'DIAL_PRIOR_TX', mapping, display=True)


# df_flat FMTNAME: DIAL_TY
mapping = {
    1: 'No dialysis',
    2: 'Hemodialysis',
    3: 'Peritoneal Dialysis',
    998: 'Dialysis Status Unknown',
    999: 'Dialysis - Unknown Type was performed',
    997: 'Unknown'
}

# map
df = uf.mapping_columns(df, 'DIAL_TY_TCR', mapping, True).copy()


# mapping
colMap = {'DIAL_AFTER_LIST': 'DialysisBetweenRegistrationTransplant_CAN', 'DIAL_PRIOR_TX':'DialysisPriorRegistration_CAN', 
          'DIAL_TY_TCR':'DialysisTypeRegistration_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='N/Y/U/X to No/Yes/Unknown/Missing')
df_dict = uf.update_dictionary_information(df_dict, [76], txt='FMTNAME: DIAL_TY - Type of Dialysis @ Registration.', feature_type='Object').copy()

# convert to category
df = uf.convert_to_category(df, list(colMap.values())).copy()

# display
df_dict.loc[idx]

Converted Column DIAL_AFTER_LIST Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column DIAL_PRIOR_TX Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column DIAL_TY_TCR Unique Vaue(s) ['No dialysis', 'Hemodialysis', 'Dialysis - Unknown Type was performed', 'Dialysis Status Unknown', 'Peritoneal Dialysis', 'Unknown']
Categories (6, str): ['Dialysis - Unknown Type was performed', 'Dialysis Status Unknown', 'Hemodialysis', 'No dialysis', 'Peritoneal Dialysis', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
75,DialysisBetweenRegistrationTransplant_CAN,DIALYSIS OCCURRING BETWEEN LISTING AND TRANSPLANT,TRR,1994-04-01,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(1),,,DIAL_AFTER_LIST,Category,N/Y/U/X to No/Yes/Unknown/Missing
76,DialysisPriorRegistration_CAN,Calculated: Ever Dialysis Prior Tx?,CALCULATED,NaT,NaT,,CHAR(1),,,DIAL_PRIOR_TX,Object,FMTNAME: DIAL_TY - Type of Dialysis @ Registra...
77,DialysisTypeRegistration_CAN,PATIENT TYPE OF DIALYSIS @ REGISTRATION,TCR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,DIAL_TY,Collected for HR . But removed for HL and LU o...,DIAL_TY_TCR,Category,N/Y/U/X to No/Yes/Unknown/Missing


### CEREB

In [136]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'CEREB', True)

            count unique top   freq
CEREB_VASC  30648      3   N  28526

NaNs:
CEREB_VASC    77
dtype: int64

Datatypes:
CEREB_VASC    str
dtype: object


       Feature                                                 Description           FormSection DataType SASAnalysisFormat Comment Information
29  CEREB_VASC  PATIENT SYMPTOMATIC CEREBROVASCULAR DISEASE @ REGISTRATION  CLINICAL INFORMATION  CHAR(1)                               Unknown


CEREB_VASC: <ArrowStringArray>
['N', 'Y', 'U', nan]
Length: 4, dtype: str


In [137]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)
    

# mapping
colMap = {'CEREB_VASC': 'CerebroVascularDisease_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='N/Y/U/X to No/Yes/Unknown/Missing')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values())).copy()

# display
df_dict.loc[idx]

Converted Column CEREB_VASC Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
29,CerebroVascularDisease_CAN,PATIENT SYMPTOMATIC CEREBROVASCULAR DISEASE @ ...,TCR,1994-04-01,2007-01-01,CLINICAL INFORMATION,CHAR(1),,,CEREB_VASC,Category,N/Y/U/X to No/Yes/Unknown/Missing


### MALIG

In [138]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'MALIG', True)

           count unique top   freq
MALIG_TCR  30652      3   N  27949
MALIG_TRR  10533      3   N  10436
MALIG      30652      3   N  27958

NaNs:
MALIG_TCR       73
MALIG_TRR    20192
MALIG           73
dtype: int64

Datatypes:
MALIG_TCR    str
MALIG_TRR    str
MALIG        str
dtype: object


       Feature                                                                Description                         FormSection DataType SASAnalysisFormat Comment Information
200      MALIG                                                   ANY PREVIOUS MALIGNANCY?                                      CHAR(1)                               Unknown
201  MALIG_TCR  ANY PREVIOUS MALIGNANCY (EXCLUDE NON-MELANOMA SKIN CANCER) @ REGISTRATION                CLINICAL INFORMATION  CHAR(1)                               Unknown
202  MALIG_TRR               RECIPIENT ANY KNOWN MALIGNANCIES SINCE LISTING  @ TRANSPLANT  PRETRANSPLANT CLINICAL INFORMATION  CHAR(1)                               Unknown


MALIG_TCR:

In [139]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)
    

# mapping
colMap = {'MALIG': 'Malignancy_CAN', 'MALIG_TCR': 'PreviousMalignancy_CAN', 'MALIG_TRR': 'MalignancyBetweenRegistrationTransplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='N/Y/U/X to No/Yes/Unknown/Missing')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display  
df_dict.loc[idx]

Converted Column MALIG_TCR Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column MALIG_TRR Unique Vaue(s) ['No', 'Unknown', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column MALIG Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
200,Malignancy_CAN,ANY PREVIOUS MALIGNANCY?,CALCULATED,NaT,NaT,,CHAR(1),,,MALIG,Category,N/Y/U/X to No/Yes/Unknown/Missing
201,PreviousMalignancy_CAN,ANY PREVIOUS MALIGNANCY (EXCLUDE NON-MELANOMA ...,TCR,1994-04-01,NaT,CLINICAL INFORMATION,CHAR(1),,,MALIG_TCR,Category,N/Y/U/X to No/Yes/Unknown/Missing
202,MalignancyBetweenRegistrationTransplant_CAN,RECIPIENT ANY KNOWN MALIGNANCIES SINCE LISTING...,TRR,1999-10-25,2015-03-31,PRETRANSPLANT CLINICAL INFORMATION,CHAR(1),,,MALIG_TRR,Category,N/Y/U/X to No/Yes/Unknown/Missing


### CANCER

#### Cancer Type

In [140]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'CANCER', True)

                         count unique top   freq
EXTRACRANIAL_CANCER_DON  30517      3   N  30278
CANCER_SITE_DON          30711     26   1  30116
INTRACRANIAL_CANCER_DON  30517      3   N  30237
HIST_CANCER_DON          30711      3   N  30116
SKIN_CANCER_DON          30517      3   N  30306

NaNs:
EXTRACRANIAL_CANCER_DON    208
CANCER_SITE_DON             14
INTRACRANIAL_CANCER_DON    208
HIST_CANCER_DON             14
SKIN_CANCER_DON            208
dtype: int64

Datatypes:
EXTRACRANIAL_CANCER_DON    str
CANCER_SITE_DON            str
INTRACRANIAL_CANCER_DON    str
HIST_CANCER_DON            str
SKIN_CANCER_DON            str
dtype: object


                     Feature                                       Description    FormSection DataType SASAnalysisFormat Comment Information
26           CANCER_SITE_DON                        DECEASED DONOR-CANCER SITE  DONOR HISTORY      NUM           HISTCAN             Unknown
109  EXTRACRANIAL_CANCER_DON  DECEASED DONOR-EXTRACANIAL CANCER AT

In [141]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)
    

# mapping
colMap = {'EXTRACRANIAL_CANCER_DON': 'CancerExtraCranial_DON', 'HIST_CANCER_DON':'CancerHistory_DON',
          'INTRACRANIAL_CANCER_DON':'CancerIntraCranial_DON', 'SKIN_CANCER_DON':'CancerSkin_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='N/Y/U/X to No/Yes/Unknown/Missing')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values())).copy()

# display
df_dict.loc[idx]

Converted Column EXTRACRANIAL_CANCER_DON Unique Vaue(s) ['No', 'Unknown', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column CANCER_SITE_DON Unique Vaue(s) ['1', '2', '14', '19', '998', ..., '6', '26', '15', '16', '17']
Length: 27
Categories (27, str): ['1', '12', '13', '14', ..., '9', '998', '999', 'Unknown']
Converted Column INTRACRANIAL_CANCER_DON Unique Vaue(s) ['No', 'Unknown', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column HIST_CANCER_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column SKIN_CANCER_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
26,CANCER_SITE_DON,DECEASED DONOR-CANCER SITE,DDR,1994-04-01,NaT,DONOR HISTORY,NUM,HISTCAN,,CANCER_SITE_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing
109,CancerExtraCranial_DON,DECEASED DONOR-EXTRACANIAL CANCER AT PROCUREMENT,DDR,1994-04-01,NaT,DONOR HISTORY,CHAR(1),,,EXTRACRANIAL_CANCER_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing
145,CancerHistory_DON,DECEASED DONOR-HISTORY OF CANCER (Y/N),DDR,1994-04-01,NaT,DONOR HISTORY,CHAR(1),,,HIST_CANCER_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing
188,CancerIntraCranial_DON,DECEASED DONOR-INTRACANIAL CANCER AT PROCUREMENT,DDR,1994-04-01,NaT,DONOR HISTORY,CHAR(1),,,INTRACRANIAL_CANCER_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing
266,CancerSkin_DON,DECEASED DONOR-SKIN CANCER AT PROCUREMENT (Y/N),DDR,1994-04-01,NaT,DONOR HISTORY,CHAR(1),,,SKIN_CANCER_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing


#### Cancer Site

In [142]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'CANCER_SITE_DON', True)

                 count unique top   freq
CANCER_SITE_DON  30725     27   1  30116

NaNs:
CANCER_SITE_DON    0
dtype: int64

Datatypes:
CANCER_SITE_DON    category
dtype: object


            Feature                 Description    FormSection DataType SASAnalysisFormat Comment                        Information
26  CANCER_SITE_DON  DECEASED DONOR-CANCER SITE  DONOR HISTORY      NUM           HISTCAN          N/Y/U/X to No/Yes/Unknown/Missing


CANCER_SITE_DON: ['1', '2', '14', '19', '998', ..., '6', '26', '15', '16', '17']
Length: 27
Categories (27, str): ['1', '12', '13', '14', ..., '9', '998', '999', 'Unknown']


In [143]:
# convert datatype
df[features] = df[features].fillna('Unknown')

# df_flat FMTNAME: HISTCAN
mapping = {
    '1': "NO",
    '2': "SKIN - SQUAMOUS, BASAL CELL",
    '3': "SKIN - MELANOMA",
    '4': "CNS TUMOR - ASTROCYTOMA",
    '5': "CNS TUMOR - GLIOBLASTOMA MULTIFORME",
    '6': "CNS TUMOR - MEDULLOBLASTOMA",
    '7': "CNS TUMOR - NEUROBLASTOMA",
    '8': "CNS TUMOR - ANGIOBLASTOMA",
    '9': "CNS TUMOR - MENINGIOMA",
    '12': "CNS TUMOR - OTHER",
    '13': "GENITOURINARY - BLADDER",
    '14': "GENITOURINARY - UTERINE CERVIX",
    '15': "GENITOURINARY - UTERINE BODY ENDOMETRIAL",
    '16': "GENITOURINARY - UTERINE BODY CHORIOCARCINOMA",
    '17': "GENITOURINARY - VULVA",
    '18': "GENITOURINARY - OVARIAN",
    '19': "GENITOURINARY - PENIS, TESTICULAR",
    '20': "GENITOURINARY - PROSTATE",
    '21': "GENITOURINARY - KIDNEY",
    '22': "GENITOURINARY - UNKNOWN",
    '23': "GASTROINTESTINAL - ESOPHAGEAL",
    '24': "GASTROINTESTINAL - STOMACH",
    '25': "GASTROINTESTINAL - SMALL INTESTINE",
    '26': "GASTROINTESTINAL - COLO-RECTAL",
    '27': "GASTROINTESTINAL - LIVER & BILIARY TRACT",
    '28': "GASTROINTESTINAL - PANCREAS",
    '29': "BREAST",
    '30': "THYROID",
    '32': "TONGUE/THROAT",
    '33': "LARYNX",
    '34': "LUNG (include bronchial)",
    '35': "LEUKEMIA/LYMPHOMA",
    '998': "Unknown",
    '999': "OTHER, SPECIFY"
}


# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'CANCER_SITE_DON':'CancerSite_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Object', txt=f"FMTNAME: HISTCAN")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values())).copy()

# display
df_dict.loc[idx]

Converted Column CANCER_SITE_DON Unique Vaue(s) ['NO', 'SKIN - SQUAMOUS, BASAL CELL', 'GENITOURINARY - UTERINE CERVIX', 'GENITOURINARY - PENIS, TESTICULAR', 'Unknown', ..., 'CNS TUMOR - MEDULLOBLASTOMA', 'GASTROINTESTINAL - COLO-RECTAL', 'GENITOURINARY - UTERINE BODY ENDOMETRIAL', 'GENITOURINARY - UTERINE BODY CHORIOCARCINOMA', 'GENITOURINARY - VULVA']
Length: 26
Categories (26, str): ['BREAST', 'CNS TUMOR - ASTROCYTOMA', 'CNS TUMOR - GLIOBLASTOMA MULTIFORME', 'CNS TUMOR - MEDULLOBLASTOMA', ..., 'SKIN - MELANOMA', 'SKIN - SQUAMOUS, BASAL CELL', 'THYROID', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
26,CancerSite_DON,DECEASED DONOR-CANCER SITE,DDR,1994-04-01,NaT,DONOR HISTORY,NUM,HISTCAN,,CANCER_SITE_DON,Object,FMTNAME: HISTCAN


### CREATININE

In [144]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'CREAT', False)

                 count unique   top  freq
MOST_RCNT_CREAT  30569    617  1.10  1907
CREAT_TRR        30304    669  1.00  1589
CREAT_DON        30518    895  0.80  2002

NaNs:
MOST_RCNT_CREAT    156
CREAT_TRR          421
CREAT_DON          207
dtype: int64

Datatypes:
MOST_RCNT_CREAT    str
CREAT_TRR          str
CREAT_DON          str
dtype: object


             Feature                                         Description                         FormSection DataType SASAnalysisFormat                                                                 Comment Information
47         CREAT_DON              DECEASED DONOR-TERMINAL LAB CREATININE                CLINICAL INFORMATION      NUM                                                                                               Unknown
48         CREAT_TRR            RECIPIENT SERUM CREATININE AT TIME OF TX  PRETRANSPLANT CLINICAL INFORMATION      NUM                                                                                         

In [145]:
# change datatype
df[features] = df[features].astype(float)

# mapping
colMap = {'CREAT_DON': 'Creatinine_DON', 'CREAT_TRR':'CreatinineTransplant_CAN', 'MOST_RCNT_CREAT':'CreatinineRegistration_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt='')

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
47,Creatinine_DON,DECEASED DONOR-TERMINAL LAB CREATININE,DDR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,,CREAT_DON,Numeric,
48,CreatinineTransplant_CAN,RECIPIENT SERUM CREATININE AT TIME OF TX,TRR,1994-04-01,NaT,PRETRANSPLANT CLINICAL INFORMATION,NUM,,,CREAT_TRR,Numeric,
204,CreatinineRegistration_CAN,PATIENT MOST RECENT ABSOLUTE CREATININE AT LIS...,TCR,1999-10-25,2007-01-01,CLINICAL INFORMATION,NUM,,Collection ended 1/1/07 for Lung (see INIT_CRE...,MOST_RCNT_CREAT,Numeric,


### SERUM

In [146]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'SERUM', False)

                 count unique   top freq
TOT_SERUM_ALBUM  10073     62  3.90  690

NaNs:
TOT_SERUM_ALBUM    20652
dtype: int64

Datatypes:
TOT_SERUM_ALBUM    str
dtype: object


             Feature                                                           Description           FormSection DataType SASAnalysisFormat Comment Information
277  TOT_SERUM_ALBUM  PATIENT TOTAL SERUM ALBUMIN  @ REGISTRATION (pre 1/1/2007 for adult)  CLINICAL INFORMATION      NUM                               Unknown




In [147]:
# change datatype
df[features] = df[features].astype(float)

# mapping
colMap = {'TOT_SERUM_ALBUM': 'TotalSerumAlbuminRegistration_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt='')

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
277,TotalSerumAlbuminRegistration_CAN,PATIENT TOTAL SERUM ALBUMIN @ REGISTRATION (p...,TCR,1999-10-25,NaT,CLINICAL INFORMATION,NUM,,,TOT_SERUM_ALBUM,Numeric,


### DEFIBRILLATOR

In [148]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DEFIBRIL', True)

               count unique top   freq
IMPL_DEFIBRIL  30650      3   Y  22695

NaNs:
IMPL_DEFIBRIL    75
dtype: int64

Datatypes:
IMPL_DEFIBRIL    str
dtype: object


           Feature                                     Description           FormSection DataType SASAnalysisFormat Comment Information
159  IMPL_DEFIBRIL  IMPLANTABLE DEFIBRILLATOR Y/N/U @ REGISTRATION  CLINICAL INFORMATION  CHAR(1)                               Unknown


IMPL_DEFIBRIL: <ArrowStringArray>
['Y', 'N', 'U', nan]
Length: 4, dtype: str


In [149]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'IMPL_DEFIBRIL': 'DefibrillatorImplantRegistration_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='N/Y/U/X to No/Yes/Unknown/Missing')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column IMPL_DEFIBRIL Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
159,DefibrillatorImplantRegistration_CAN,IMPLANTABLE DEFIBRILLATOR Y/N/U @ REGISTRATION,TCR,1994-04-01,NaT,CLINICAL INFORMATION,CHAR(1),,,IMPL_DEFIBRIL,Category,N/Y/U/X to No/Yes/Unknown/Missing


### HEMODYNAMICS

In [150]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'HEMO', False)

                 count unique    top  freq
HEMO_SYS_TCR     29613    109  40.00  1095
HEMO_PA_DIA_TCR  29591     78  20.00  1862
HEMO_PA_MN_TCR   29274    169  30.00  1157
HEMO_PCW_TCR     27801     54  20.00  1506
HEMO_CO_TCR      28924    793   4.00   634
HEMO_CO_TRR      28985    852   4.00   585
HEMO_PA_DIA_TRR  29473     73  20.00  1757
HEMO_PA_MN_TRR   29174    163  22.00  1196
HEMO_PCW_TRR     28124     99  20.00  1341
HEMO_SYS_TRR     29497    113  30.00  1069

NaNs:
HEMO_SYS_TCR       1112
HEMO_PA_DIA_TCR    1134
HEMO_PA_MN_TCR     1451
HEMO_PCW_TCR       2924
HEMO_CO_TCR        1801
HEMO_CO_TRR        1740
HEMO_PA_DIA_TRR    1252
HEMO_PA_MN_TRR     1551
HEMO_PCW_TRR       2601
HEMO_SYS_TRR       1228
dtype: int64

Datatypes:
HEMO_SYS_TCR       str
HEMO_PA_DIA_TCR    str
HEMO_PA_MN_TCR     str
HEMO_PCW_TCR       str
HEMO_CO_TCR        str
HEMO_CO_TRR        str
HEMO_PA_DIA_TRR    str
HEMO_PA_MN_TRR     str
HEMO_PCW_TRR       str
HEMO_SYS_TRR       str
dtype: object


         

In [151]:
# change datatype
df[features] = df[features].astype(float)

# mapping
colMap = {'HEMO_CO_TCR': 'HemodynamicsRegistration_CO_CAN', 'HEMO_CO_TRR':'HemodynamicsTransplant_CO_CAN', 
          'HEMO_PA_DIA_TCR':'HemodynamicsRegistration_PA_DIA_CAN', 'HEMO_PA_DIA_TRR':'HemodynamicsTransplant_PA_DIA_CAN',
          'HEMO_PA_MN_TCR':'HemodynamicsRegistration_PA_MN_CAN','HEMO_PA_MN_TRR':'HemodynamicsTransplant_PA_MN_CAN',
          'HEMO_PCW_TCR':'HemodynamicsRegistration_PCW_CAN','HEMO_PCW_TRR':'HemodynamicsTransplant_PCW_CAN',
          'HEMO_SYS_TCR':'HemodynamicsRegistration_SYS_CAN','HEMO_SYS_TRR':'HemodynamicsTransplant_SYS_CAN' 
         }

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt='')

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
130,HemodynamicsRegistration_CO_CAN,MOST RECENT HEMODYNAMICS CO L/MIN @ REGISTRATION,TCR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,BOTH BEST AND BASELINE COLLECTED BETWEEN 04/01...,HEMO_CO_TCR,Numeric,
131,HemodynamicsTransplant_CO_CAN,MOST RECENT HEMODYNAMICS CO L/MIN @ TRANSPLANT,TRR,1994-04-01,NaT,PRETRANSPLANT CLINICAL INFORMATION,NUM,,BOTH BEST AND BASELINE COLLECTED BETWEEN 04/01...,HEMO_CO_TRR,Numeric,
132,HemodynamicsRegistration_PA_DIA_CAN,MOST RECENT HEMODYNAMICS PA (DIA) MM/HG @ REGI...,TCR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,BOTH BEST AND BASELINE COLLECTED BETWEEN 04/01...,HEMO_PA_DIA_TCR,Numeric,
133,HemodynamicsTransplant_PA_DIA_CAN,MOST RECENT HEMODYNAMICS PA (DIA) MM/HG @ TRAN...,TRR,1994-04-01,NaT,PRETRANSPLANT CLINICAL INFORMATION,NUM,,BOTH BEST AND BASELINE COLLECTED BETWEEN 04/01...,HEMO_PA_DIA_TRR,Numeric,
134,HemodynamicsRegistration_PA_MN_CAN,MOST RECENT HEMODYNAMICS PA (MEAN) MM/HG @ REG...,TCR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,BOTH BEST AND BASELINE COLLECTED BETWEEN 04/01...,HEMO_PA_MN_TCR,Numeric,
135,HemodynamicsTransplant_PA_MN_CAN,MOST RECENT HEMODYNAMICS PA (MEAN) MM/HG @ TRA...,TRR,1994-04-01,NaT,PRETRANSPLANT CLINICAL INFORMATION,NUM,,BOTH BEST AND BASELINE COLLECTED BETWEEN 04/01...,HEMO_PA_MN_TRR,Numeric,
136,HemodynamicsRegistration_PCW_CAN,MOST RECENT HEMODYNAMICS PCW (MEAN) MM/HG @ RE...,TCR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,BOTH BEST AND BASELINE COLLECTED BETWEEN 04/01...,HEMO_PCW_TCR,Numeric,
137,HemodynamicsTransplant_PCW_CAN,MOST RECENT HEMODYNAMICS PCW (MEAN) MM/HG @ TR...,TRR,1994-04-01,NaT,PRETRANSPLANT CLINICAL INFORMATION,NUM,,BOTH BEST AND BASELINE COLLECTED BETWEEN 04/01...,HEMO_PCW_TRR,Numeric,
138,HemodynamicsRegistration_SYS_CAN,MOST RECENT HEMODYNAMICS PA (SYS) MM/HG @ REGI...,TCR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,BOTH BEST AND BASELINE COLLECTED BETWEEN 04/01...,HEMO_SYS_TCR,Numeric,
139,HemodynamicsTransplant_SYS_CAN,MOST RECENT HEMODYNAMICS PA (SYS) MM/HG @ TRAN...,TRR,1994-04-01,NaT,PRETRANSPLANT CLINICAL INFORMATION,NUM,,BOTH BEST AND BASELINE COLLECTED BETWEEN 04/01...,HEMO_SYS_TRR,Numeric,


### INOTROP

In [152]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'INOTROP', True)

                      count unique top   freq
INOTROP_VASO_SYS_TCR  29611      2   N  17314
INOTROP_VASO_DIA_TCR  29589      2   N  17304
INOTROP_VASO_MN_TCR   29272      2   N  17153
INOTROP_VASO_PCW_TCR  27798      2   N  17033
INOTROP_VASO_CO_TCR   28921      2   N  17005
INOTROP_VASO_CO_TRR   28978      2   N  16575
INOTROP_VASO_DIA_TRR  29468      2   N  16766
INOTROP_VASO_MN_TRR   29168      2   N  16621
INOTROP_VASO_PCW_TRR  28119      2   N  16565
INOTROP_VASO_SYS_TRR  29491      2   N  16781
INOTROP_SUPPORT_DON   30519      3   N  18280

NaNs:
INOTROP_VASO_SYS_TCR    1114
INOTROP_VASO_DIA_TCR    1136
INOTROP_VASO_MN_TCR     1453
INOTROP_VASO_PCW_TCR    2927
INOTROP_VASO_CO_TCR     1804
INOTROP_VASO_CO_TRR     1747
INOTROP_VASO_DIA_TRR    1257
INOTROP_VASO_MN_TRR     1557
INOTROP_VASO_PCW_TRR    2606
INOTROP_VASO_SYS_TRR    1234
INOTROP_SUPPORT_DON      206
dtype: int64

Datatypes:
INOTROP_VASO_SYS_TCR    str
INOTROP_VASO_DIA_TCR    str
INOTROP_VASO_MN_TCR     str
INOTROP_VASO_

In [153]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'INOTROP_VASO_CO_TCR': 'InotropesVasodilatorsRegistration_CO_CAN', 'INOTROP_VASO_CO_TRR':'InotropesVasodilatorsTransplant_CO_CAN', 
          'INOTROP_VASO_DIA_TCR':'InotropesVasodilatorsRegistration_DIA_CAN', 'INOTROP_VASO_DIA_TRR':'InotropesVasodilatorsTransplant_DIA_CAN',
          'INOTROP_VASO_MN_TCR':'InotropesVasodilatorsRegistration_MN_CAN','INOTROP_VASO_MN_TRR':'InotropesVasodilatorsTransplant_MN_CAN',
          'INOTROP_VASO_PCW_TCR':'InotropesVasodilatorsRegistration_PCW_CAN','INOTROP_VASO_PCW_TRR':'InotropesVasodilatorsTransplant_PCW_CAN',
          'INOTROP_VASO_SYS_TCR':'InotropesVasodilatorsRegistration_SYS_CAN','INOTROP_VASO_SYS_TRR':'InotropesVasodilatorsTransplant_SYS_CAN' 
         }


# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='N/Y/U/X to No/Yes/Unknown')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# donor
colMap = {'INOTROP_SUPPORT_DON':'InotropicMedicationProcurement_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='N/Y/U/X to No/Yes/Unknown')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column INOTROP_VASO_SYS_TCR Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column INOTROP_VASO_DIA_TCR Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column INOTROP_VASO_MN_TCR Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column INOTROP_VASO_PCW_TCR Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column INOTROP_VASO_CO_TCR Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column INOTROP_VASO_CO_TRR Unique Vaue(s) ['Unknown', 'No', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column INOTROP_VASO_DIA_TRR Unique Vaue(s) ['Unknown', 'No', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column INOTROP_VASO_MN_TRR Unique Vaue(s) ['Unknown', 'No', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Co

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
174,InotropicMedicationProcurement_DON,DECEASED DONOR INOTROPIC MEDICATION AT PROCURE...,DDR,2003-01-27,NaT,CLINICAL INFORMATION,CHAR(1),,"For Heart, this field was collected since 10/2...",INOTROP_SUPPORT_DON,Category,N/Y/U/X to No/Yes/Unknown
175,InotropesVasodilatorsRegistration_CO_CAN,MOST RECENT CO L/MIN INOTROPES/VASODILATORS YE...,TCR,2004-06-30,NaT,HEART/LUNG MEDICAL FACTORS,CHAR(1),,,INOTROP_VASO_CO_TCR,Category,N/Y/U/X to No/Yes/Unknown
176,InotropesVasodilatorsTransplant_CO_CAN,TRR CARDIAC OUTPUT MEASUREMENT OBTAINED WHILE ...,TRR,1999-10-25,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(1),,,INOTROP_VASO_CO_TRR,Category,N/Y/U/X to No/Yes/Unknown
177,InotropesVasodilatorsRegistration_DIA_CAN,MOST RECENT PA (DIA) MM/HG INOTROPES/VAOSDILAT...,TCR,2004-06-30,NaT,HEART/LUNG MEDICAL FACTORS,CHAR(1),,,INOTROP_VASO_DIA_TCR,Category,N/Y/U/X to No/Yes/Unknown
178,InotropesVasodilatorsTransplant_DIA_CAN,TRR DIASTOLIC MEASUREMENT OBTAINED WHILE ON IN...,TRR,1999-10-26,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(1),,,INOTROP_VASO_DIA_TRR,Category,N/Y/U/X to No/Yes/Unknown
179,InotropesVasodilatorsRegistration_MN_CAN,MOST RECENT PA (MEAN) MM/HG INOTROPES/VASODILA...,TCR,2004-06-30,NaT,HEART/LUNG MEDICAL FACTORS,CHAR(1),,,INOTROP_VASO_MN_TCR,Category,N/Y/U/X to No/Yes/Unknown
180,InotropesVasodilatorsTransplant_MN_CAN,TRR MEAN PULMONARY ARTERY MEASUREMENT OBTAINED...,TRR,1999-10-27,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(1),,,INOTROP_VASO_MN_TRR,Category,N/Y/U/X to No/Yes/Unknown
181,InotropesVasodilatorsRegistration_PCW_CAN,MOST RECENT PCW (MEAN) MM/HG INOTROPES/VASODIL...,TCR,2004-06-30,NaT,HEART/LUNG MEDICAL FACTORS,CHAR(1),,,INOTROP_VASO_PCW_TCR,Category,N/Y/U/X to No/Yes/Unknown
182,InotropesVasodilatorsTransplant_PCW_CAN,TRR MEAN PULMONARY CAPILLARY WEDGE MEASUREMENT...,TRR,1999-10-28,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(1),,,INOTROP_VASO_PCW_TRR,Category,N/Y/U/X to No/Yes/Unknown
183,InotropesVasodilatorsRegistration_SYS_CAN,Most Recent PA (sys) mm/Hg Inotropes/Vasodil...,TCR,1999-10-25,NaT,HEART/LUNG MEDICAL FACTORS,CHAR(1),,,INOTROP_VASO_SYS_TCR,Category,N/Y/U/X to No/Yes/Unknown


### CIGARETTES

In [154]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'CIG|ABSTAIN', True)

                 count unique top   freq
CIG_USE          30643      2   N  17123
TCR_DUR_ABSTAIN  13520      9   7   7650
HIST_CIG_DON     30720      3   N  26658

NaNs:
CIG_USE               82
TCR_DUR_ABSTAIN    17205
HIST_CIG_DON           5
dtype: int64

Datatypes:
CIG_USE            str
TCR_DUR_ABSTAIN    str
HIST_CIG_DON       str
dtype: object


             Feature                                                 Description            FormSection DataType SASAnalysisFormat Comment        Information
31           CIG_USE                                    HISTORY OF CIGARETTE USE   CLINICAL INFORMATION  CHAR(1)                                      Unknown
146     HIST_CIG_DON  DECEASED DONOR-HISTORY OF CIGARETTES IN PAST @ >20PACK YRS          DONOR HISTORY  CHAR(1)                            FMTNAME: HISTDIAB
275  TCR_DUR_ABSTAIN                    DURATION OF ABSTINENCE FOR CIGARETTE USE  CANDIDATE INFORMATION      NUM          CIGDURAB                    Unknown


CIG_USE: <

In [155]:
findMappingDfFlat(df.TCR_DUR_ABSTAIN, df_flat, 'CIGDUR', NaN=999)

Compare Length: 10 & 8

CODE            LABEL
   1       0-2 Months
   2      3-12 Months
   3     13-24 Months
   4     25-36 Months
   5     37-48 Months
   6     49-60 Months
   7       >60 Months
 998 Unknown Duration


In [156]:
# fill NaN with X or 999: Missing
df[['CIG_USE','HIST_CIG_DON']] = df[['CIG_USE','HIST_CIG_DON']].fillna('Unknown')
df['TCR_DUR_ABSTAIN'] = df['TCR_DUR_ABSTAIN'].fillna(999).astype(int)

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# mapping feature
df = uf.mapping_columns(df, 'CIG_USE', mapping, display=True)
df = uf.mapping_columns(df, 'HIST_CIG_DON', mapping, display=True)


# df_flat FMTNAME: CIGDURAB
mapping = {
    1: '0-2 months',
    2: '3-12 months',
    3: '13-24 months',
    4: '25-36 months',
    5: '37-48 months',
    6: '49-60 months',
    7: '>60 months',
    8: 'Continues to smoke',
    998: 'Unknown duration',
    999: "Unknown"
}

# map
df = uf.mapping_columns(df, 'TCR_DUR_ABSTAIN', mapping, False)


# mapping
colMap = {'CIG_USE': 'CigaretteUse_CAN', 'HIST_CIG_DON':'CigaretteHistory_DON', 'TCR_DUR_ABSTAIN': 'CigaretteAbstinence_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='N/Y/U/X to No/Yes/Unknown/Missing')
df_dict = uf.update_dictionary_information(df_dict, [297], txt='FMTNAME: CIGDURAB')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column CIG_USE Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column HIST_CIG_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
31,CigaretteUse_CAN,HISTORY OF CIGARETTE USE,TCR,2004-06-30,NaT,CLINICAL INFORMATION,CHAR(1),,,CIG_USE,Category,N/Y/U/X to No/Yes/Unknown/Missing
146,CigaretteHistory_DON,DECEASED DONOR-HISTORY OF CIGARETTES IN PAST @...,DDR,1994-04-01,NaT,DONOR HISTORY,CHAR(1),,,HIST_CIG_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing
275,CigaretteAbstinence_CAN,DURATION OF ABSTINENCE FOR CIGARETTE USE,TCR,2004-06-30,NaT,CANDIDATE INFORMATION,NUM,CIGDURAB,,TCR_DUR_ABSTAIN,Category,N/Y/U/X to No/Yes/Unknown/Missing


### PRIOR_CARD

In [157]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'PRIOR_CARD', True)

                                count unique   top   freq
PRIOR_CARD_SURG_TCR             30522      3     N  18035
PRIOR_CARD_SURG_TYPE_TCR        12022     23    16   5203
PRIOR_CARD_SURG_TYPE_OSTXT_TCR   6318   1788  LVAD   1967
PRIOR_CARD_SURG_TYPE_TRR         7092     21    16   4668
PRIOR_CARD_SURG_TRR             30302      3     N  23104

NaNs:
PRIOR_CARD_SURG_TCR                 203
PRIOR_CARD_SURG_TYPE_TCR          18703
PRIOR_CARD_SURG_TYPE_OSTXT_TCR    24407
PRIOR_CARD_SURG_TYPE_TRR          23633
PRIOR_CARD_SURG_TRR                 423
dtype: int64

Datatypes:
PRIOR_CARD_SURG_TCR               str
PRIOR_CARD_SURG_TYPE_TCR          str
PRIOR_CARD_SURG_TYPE_OSTXT_TCR    str
PRIOR_CARD_SURG_TYPE_TRR          str
PRIOR_CARD_SURG_TRR               str
dtype: object


                      Feature                                                               Description                         FormSection DataType SASAnalysisFormat Comment Information
226       PRIOR_CARD_SURG_T

In [158]:
# get unique values
print(uf.combine_get_unique(df, 'PRIOR_CARD_SURG_TYPE_TCR', 'PRIOR_CARD_SURG_TYPE_TRR', 302, False))

[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(24), np.int64(25), np.int64(26), np.int64(28), np.int64(302)]


In [159]:
# check for differences between two sets
uf.symmetric_difference(set(df.PRIOR_CARD_SURG_TYPE_TCR.dropna().unique().astype(int)), set(df.PRIOR_CARD_SURG_TYPE_TRR.dropna().unique().astype(int)))

Symmetric difference: [np.int64(11), np.int64(12), np.int64(14), np.int64(15), np.int64(25), np.int64(28)]


In [160]:
# fill NaN with X or 302: Missing
df[['PRIOR_CARD_SURG_TCR','PRIOR_CARD_SURG_TRR']] = df[['PRIOR_CARD_SURG_TCR','PRIOR_CARD_SURG_TRR']].fillna('Unknown')
df[['PRIOR_CARD_SURG_TYPE_TCR', 'PRIOR_CARD_SURG_TYPE_TRR']] = df[['PRIOR_CARD_SURG_TYPE_TCR', 'PRIOR_CARD_SURG_TYPE_TRR']].fillna(302).astype(int)

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# mapping feature
df = uf.mapping_columns(df, 'PRIOR_CARD_SURG_TCR', mapping, display=True)
df = uf.mapping_columns(df, 'PRIOR_CARD_SURG_TRR', mapping, display=True)


# df_flat FMTNAME: CARDSURG
mapping = {
    302: "Unknown",
    303: "Not Reported",
    1: "CABG",
    2: "Valve Replace/Repair",
    3: "CABG; Valve Replace/Repair",
    4: "Congenital",
    5: "CABG; Congenital",
    6: "Valve Replace/Repair; Congenital",
    7: "CABG; Valve Replace/Repair; Congenital",
    8: "Left Vent. Remodeling",
    9: "CABG; Left Vent. Remodeling",
    10: "Valve Replace/Repair; Left Vent. Remodeling",
    11: "CABG; Valve Replace/Repair; Left Vent. Remodeling",
    12: "Congenital; Left Vent. Remodeling",
    13: "CABG; Congenital; Left Vent. Remodeling",
    14: "Valve Replace/Repair; Congenital; Left Vent. Remodeling",
    15: "CABG; Valve Replace/Repair; Congenital; Left Vent. Remodeling",
    16: "Other, specify",
    17: "CABG; Other, specify",
    18: "Valve Replace/Repair; Other, specify",
    19: "CABG; Valve Replace/Repair; Other, specify",
    20: "Congenital; Other, specify",
    21: "CABG; Congenital; Other, specify",
    22: "Valve Replace/Repair; Congenital; Other, specify",
    23: "CABG; Valve Replace/Repair; Congenital; Other, specify",
    24: "Left Vent. Remodeling; Other, specify",
    25: "CABG; Left Vent. Remodeling; Other, specify",
    26: "Valve Replace/Repair; Left Vent. Remodeling; Other, specify",
    27: "CABG; Valve Replace/Repair; Left Vent. Remodeling; Other, specify",
    28: "Congenital; Left Vent. Remodeling; Other, specify",
    29: "CABG; Congenital; Left Vent. Remodeling; Other, specify",
    30: "Valve Replace/Repair; Congenital; Left Vent. Remodeling; Other, specify",
    31: "CABG; Valve Replace/Repair; Congenital; Left Vent. Remodeling; Other, specify",
    334: "Unknown"
}

# mapping feature
df = uf.mapping_columns(df, 'PRIOR_CARD_SURG_TYPE_TCR', mapping, display=True)
df = uf.mapping_columns(df, 'PRIOR_CARD_SURG_TYPE_TRR', mapping, display=True)


# mapping
colMap = {'PRIOR_CARD_SURG_TCR': 'PriorCardiacSurgery_CAN', 'PRIOR_CARD_SURG_TRR':'PriorCardiacSurgeryListAndTransplant_CAN',
          'PRIOR_CARD_SURG_TYPE_OSTXT_TCR': 'PriorCardiacSurgeryTypeText_CAN',
          'PRIOR_CARD_SURG_TYPE_TCR': 'PriorCardiacSurgeryType_CAN', 'PRIOR_CARD_SURG_TYPE_TRR': 'PriorCardiacSurgeryTypeListAndTransplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='')
df_dict = uf.update_dictionary_information(df_dict, [248,249], txt='N/Y/U/X to No/Yes/Unknown')
df_dict = uf.update_dictionary_information(df_dict, [225,226], txt='FMTNAME: CARDSURG').copy()

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column PRIOR_CARD_SURG_TCR Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column PRIOR_CARD_SURG_TRR Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column PRIOR_CARD_SURG_TYPE_TCR Unique Vaue(s) ['Valve Replace/Repair', 'Unknown', 'CABG; Other, specify', 'Other, specify', 'CABG', ..., 'CABG; Left Vent. Remodeling', 'CABG; Left Vent. Remodeling; Other, specify', 'Valve Replace/Repair; Left Vent. Remodeling; ..., 'Valve Replace/Repair; Congenital; Left Vent. ..., 'CABG; Valve Replace/Repair; Congenital']
Length: 24
Categories (24, str): ['CABG', 'CABG; Congenital', 'CABG; Congenital; Other, specify', 'CABG; Left Vent. Remodeling', ..., 'Valve Replace/Repair; Congenital; Other, spec..., 'Valve Replace/Repair; Left Vent. Remodeling', 'Valve Replace/Repair; Left Vent. Remodeling; ..., 'Valve Replace/Repair; Other, specify']
Converted Column PRIOR_CARD_SURG_TYPE_TRR Unique Vaue(s) ['Ot

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
226,PriorCardiacSurgery_CAN,TCR PRIOR CARDIAC SURGERY AT LISTING (NON-TRAN...,TCR,2004-06-30,NaT,CLINICAL INFORMATION,CHAR(1),,,PRIOR_CARD_SURG_TCR,Category,FMTNAME: CARDSURG
227,PriorCardiacSurgeryListAndTransplant_CAN,TRR CARDIAC SURGERY BETWEEN LISTING AND TRANSP...,TRR,2004-06-30,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(1),,,PRIOR_CARD_SURG_TRR,Category,
228,PriorCardiacSurgeryType_CAN,TRR PRIOR CARDIAC SURGERY TYPE AT LISTING (NON...,TCR,2004-06-30,NaT,PRETRANSPLANT CLINICAL INFORMATION,NUM,CARDSURG,,PRIOR_CARD_SURG_TYPE_TCR,Category,
229,PriorCardiacSurgeryTypeListAndTransplant_CAN,TRR CARDIAC SURGERY TYPE BETWEEN LISTING AND T...,TRR,2004-06-30,NaT,PRETRANSPLANT CLINICAL INFORMATION,NUM,CARDSURG,,PRIOR_CARD_SURG_TYPE_TRR,Category,


### DAYS

In [161]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DAYS_', False)

             count unique top   freq
DAYS_STAT1   30725      3   0  30723
DAYS_STAT1A  30725    455   0  14673
DAYS_STAT2   30725   1005   0  24589
DAYS_STAT1B  30725   1142   0  14646
DAYS_STATA4  30725    688   0  26065
DAYS_STATA5  30725    118   0  30476
DAYS_STATA2  30725    126   0  25509
DAYS_STATA3  30725    285   0  27210
DAYS_STATA1  30725     49   0  29761
DAYS_STATA6  30725    376   0  28818

NaNs:
DAYS_STAT1     0
DAYS_STAT1A    0
DAYS_STAT2     0
DAYS_STAT1B    0
DAYS_STATA4    0
DAYS_STATA5    0
DAYS_STATA2    0
DAYS_STATA3    0
DAYS_STATA1    0
DAYS_STATA6    0
dtype: int64

Datatypes:
DAYS_STAT1     str
DAYS_STAT1A    str
DAYS_STAT2     str
DAYS_STAT1B    str
DAYS_STATA4    str
DAYS_STATA5    str
DAYS_STATA2    str
DAYS_STATA3    str
DAYS_STATA1    str
DAYS_STATA6    str
dtype: object


        Feature             Description FormSection DataType SASAnalysisFormat Comment Information
54   DAYS_STAT1        DAYS IN STATUS 1                  NUM                          

In [162]:
# mapping
colMap = {'DAYS_STAT1': 'DAYS_STAT1','DAYS_STAT1A':'StatusDays_1A', 'DAYS_STAT1B':'StatusDays_1B', 'DAYS_STAT2':'StatusDays_2',
          'DAYS_STATA1':'StatusDays_1', 'DAYS_STATA2':'StatusDays_A2','DAYS_STATA3':'StatusDays_A3',
          'DAYS_STATA4':'StatusDays_A4', 'DAYS_STATA5':'StatusDays_A5','DAYS_STATA6':'StatusDays_A6'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Nemeric', txt=f"{UNKNOWN}")

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
54,DAYS_STAT1,DAYS IN STATUS 1,CALCULATED,NaT,NaT,,NUM,,,DAYS_STAT1,Nemeric,** UNKNOWN **
55,StatusDays_1A,DAYS IN STATUS 1A,CALCULATED,NaT,NaT,,NUM,,,DAYS_STAT1A,Nemeric,** UNKNOWN **
56,StatusDays_1B,DAYS IN STATUS 1B,CALCULATED,NaT,NaT,,NUM,,,DAYS_STAT1B,Nemeric,** UNKNOWN **
57,StatusDays_2,DAYS IN STATUS 2,CALCULATED,NaT,NaT,,NUM,,,DAYS_STAT2,Nemeric,** UNKNOWN **
58,StatusDays_1,DAYS IN ADULT STATUS 1,CALCULATED,2018-10-18,NaT,,NUM,,,DAYS_STATA1,Nemeric,** UNKNOWN **
59,StatusDays_A2,DAYS IN ADULT STATUS 2,CALCULATED,2018-10-18,NaT,,NUM,,,DAYS_STATA2,Nemeric,** UNKNOWN **
60,StatusDays_A3,DAYS IN ADULT STATUS 3,CALCULATED,2018-10-18,NaT,,NUM,,,DAYS_STATA3,Nemeric,** UNKNOWN **
61,StatusDays_A4,DAYS IN ADULT STATUS 4,CALCULATED,2018-10-18,NaT,,NUM,,,DAYS_STATA4,Nemeric,** UNKNOWN **
62,StatusDays_A5,DAYS IN ADULT STATUS 5,CALCULATED,2018-10-18,NaT,,NUM,,,DAYS_STATA5,Nemeric,** UNKNOWN **
63,StatusDays_A6,DAYS IN ADULT STATUS 6,CALCULATED,2018-10-18,NaT,,NUM,,,DAYS_STATA6,Nemeric,** UNKNOWN **


### INACTACTIVE STATUS REASON

In [163]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'INACT', True)

                  count unique top  freq
LAST_INACT_REASON  8042     15   7  3732

NaNs:
LAST_INACT_REASON    22683
dtype: int64

Datatypes:
LAST_INACT_REASON    str
dtype: object


               Feature                                Description FormSection DataType SASAnalysisFormat Comment Information
191  LAST_INACT_REASON  Candidate Reason for Last Inactive Status                  NUM                               Unknown


LAST_INACT_REASON: <ArrowStringArray>
[ nan, '11',  '7', '13',  '4',  '8',  '2', '10',  '5',  '3', '12',  '1',  '9',
  '6', '15', '16']
Length: 16, dtype: str


In [164]:
# mapping
colMap = {'LAST_INACT_REASON': 'LastInactiveStatusReason'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"{UNKNOWN} No Mapping Information.")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
191,LastInactiveStatusReason,Candidate Reason for Last Inactive Status,NaN,NaT,NaT,,NUM,,,LAST_INACT_REASON,Category,** UNKNOWN ** No Mapping Information.


### INIT_STAT

In [165]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'INIT_STAT', True)

           count unique   top   freq
INIT_STAT  30725     11  2020  10077

NaNs:
INIT_STAT    0
dtype: int64

Datatypes:
INIT_STAT    str
dtype: object


       Feature                       Description        FormSection DataType SASAnalysisFormat Comment                Information
171  INIT_STAT  INITIAL WAITING LIST STATUS CODE  WAITING LIST DATA      NUM              STAT          SASAnalysisFormat: INOMED


INIT_STAT: <ArrowStringArray>
['2010', '2020', '2030', '2999', '2090', '2120', '2160', '2140', '2110',
 '2130', '2150']
Length: 11, dtype: str


In [166]:
# change datatype to integer
df[features] = df[features].astype(int)

# df_flat FMTNAME: STAT
mapping = {
  2010: 'HR: Status 1A',
  2020: 'HR: Status 1B',
  2030: 'HR: Status 2',
  2090: 'HR: Old Status 1',
  2110: 'HR: Adult Status 1',
  2120: 'HR: Adult Status 2',
  2130: 'HR: Adult Status 3',
  2140: 'HR: Adult Status 4',
  2150: 'HR: Adult Status 5',
  2160: 'HR: Adult Status 6',
  2999: 'HR: Temporarily inactive'
}

# map
df = uf.mapping_columns(df, 'INIT_STAT', mapping, False)

# mapping
colMap = {'INIT_STAT': 'InitialWaitingListStatusCode_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"UNKNOWN Unable to Determine the Meaning.")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
171,InitialWaitingListStatusCode_CAN,INITIAL WAITING LIST STATUS CODE,WAITING LIST DATA,1990-01-01,NaT,WAITING LIST DATA,NUM,STAT,,INIT_STAT,Numeric,UNKNOWN Unable to Determine the Meaning.


### REM_CD

In [167]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'REM_CD', True)

        count unique top   freq
REM_CD  30725      3   4  30665

NaNs:
REM_CD    0
dtype: int64

Datatypes:
REM_CD    str
dtype: object


    Feature                               Description        FormSection DataType SASAnalysisFormat                                                              Comment Information
262  REM_CD  REASON FOR REMOVAL FROM THE WAITING LIST  WAITING LIST DATA      NUM             REMCD  THIS IS MISSING IF PATIENT IS STILL WAITING AT TIME DATASET CREATED     Unknown


REM_CD: <ArrowStringArray>
['4', '21', '15']
Length: 3, dtype: str


In [168]:
# df_flat FMTNAME: REMCD
mapping = {
  4: 'Deceased Donor tx, removed by tx center',
  15: 'Living Donor tx, removed by tx center',
  21: 'Patient died during TX procedure'
}

# map
df = uf.mapping_columns(df, 'REM_CD', mapping, False)

# mapping
colMap = {'REM_CD': 'ReasonRemovalWaitingList_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='FMTNAME: REMCD')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
262,ReasonRemovalWaitingList_CAN,REASON FOR REMOVAL FROM THE WAITING LIST,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,REMCD,THIS IS MISSING IF PATIENT IS STILL WAITING AT...,REM_CD,Category,FMTNAME: REMCD


### TXED

In [169]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'TXED', True)

      count unique top   freq
TXED  30725      2   1  30722

NaNs:
TXED    0
dtype: int64

Datatypes:
TXED    str
dtype: object


    Feature                                          Description        FormSection DataType SASAnalysisFormat Comment Information
285    TXED  CANDIDATE RECEIVED DECEASED DONOR TRANSPLANT? 1=YES  WAITING LIST DATA      NUM                               Unknown


TXED: <ArrowStringArray>
['1', '0']
Length: 2, dtype: str


In [170]:
# mapping
mapping = {
    '1': "Yes",
    '0': "No"
}

# Convert only selected columns
df[features[0]] = df[features[0]].map(mapping)


# mapping
colMap = {'TXED': 'ReceivedDeceasedDonorTramsplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='')

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
285,ReceivedDeceasedDonorTramsplant_CAN,CANDIDATE RECEIVED DECEASED DONOR TRANSPLANT? ...,WL,NaT,NaT,WAITING LIST DATA,NUM,,,TXED,Category,


### DAYSWAIT

In [171]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DAYSWAIT', True)

                count unique top freq
DAYSWAIT_CHRON  30725   1808   2  657

NaNs:
DAYSWAIT_CHRON    0
dtype: int64

Datatypes:
DAYSWAIT_CHRON    str
dtype: object


           Feature                 Description FormSection DataType SASAnalysisFormat Comment Information
64  DAYSWAIT_CHRON  TOTAL DAYS ON WAITING LIST                  NUM                               Unknown


DAYSWAIT_CHRON: <ArrowStringArray>
[  '11',  '348',   '55',  '392',   '42',  '580',  '128',    '6',   '21',
   '16',
 ...
 '1324', '2123', '1510', '1584', '2343', '2848', '4938',  '956', '2297',
  '940']
Length: 1808, dtype: str


In [172]:
# change datatypes
df[features] = df[features].astype(float)

# mapping
colMap = {'DAYSWAIT_CHRON': 'TotalDayWaitList_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt='')

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
64,TotalDayWaitList_CAN,TOTAL DAYS ON WAITING LIST,CALCULATED,NaT,NaT,,NUM,,,DAYSWAIT_CHRON,Numeric,


### END_STAT

In [173]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'END_STAT', True)

          count unique   top   freq
END_STAT  30725      9  2010  13311

NaNs:
END_STAT    0
dtype: int64

Datatypes:
END_STAT    str
dtype: object


      Feature                                               Description        FormSection DataType SASAnalysisFormat Comment Information
104  END_STAT  CANDIDATE STATUS AT TRANSPLANT OFFER/REMOVALCURRENT TIME  WAITING LIST DATA      NUM              STAT             Unknown


END_STAT: <ArrowStringArray>
['2010', '2020', '2030', '2120', '2140', '2110', '2130', '2160', '2150']
Length: 9, dtype: str


In [174]:
# df_label FMTNAME: CHDMULT
mapping = {
    '2010': "Hypoplastic Left Heart Syndrome; Atrioventricular Septal Defect; Other left Heart Valvar/Structural",
    '2020': "Transposition of the Great Arteries; Truncus Arteriosus; Congenitally Corrected Transposition (L-TGA)",
    '2030': "Hypoplastic Left Heart Syndrome; Transposition of the Great Arteries; Atrioventricular Septal Defect",
    '2110': "Hypoplastic Left Heart Syndrome; Transposition of the Great Arteries; Atrioventricular Septal Defect",
    '2120': "Atrioventricular Septal Defect; Congenitally Corrected Transposition (L-TGA); Other",
    '2130': "Hypoplastic Left Heart Syndrome; Other left Heart Valvar/Structural Hypoplasia; Congenitally Correct",
    '2140': "Transposition of the Great Arteries; Atrioventricular Septal Defect; Other left Heart Valvar/Structural",
    '2150': "Hypoplastic Left Heart Syndrome; Transposition of the Great Arteries; Truncus Arteriosus; Congenital",
    '2160': "Other left Heart Valvar/Structural Hypoplasia; Truncus Arteriosus; Congenitally Corrected Transposition"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)
    

# mapping
colMap = {'END_STAT': 'StatusAtTransplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='FMTNAME: CHDMULT - This Feature could be Ordinal but using as Nominal')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column END_STAT Unique Vaue(s) ['Hypoplastic Left Heart Syndrome; Atrioventric..., 'Transposition of the Great Arteries; Truncus ..., 'Hypoplastic Left Heart Syndrome; Transpositio..., 'Atrioventricular Septal Defect; Congenitally ..., 'Transposition of the Great Arteries; Atrioven..., 'Hypoplastic Left Heart Syndrome; Other left H..., 'Other left Heart Valvar/Structural Hypoplasia..., 'Hypoplastic Left Heart Syndrome; Transpositio...]
Categories (8, str): ['Atrioventricular Septal Defect; Congenitally ..., 'Hypoplastic Left Heart Syndrome; Atrioventric..., 'Hypoplastic Left Heart Syndrome; Other left H..., 'Hypoplastic Left Heart Syndrome; Transpositio..., 'Hypoplastic Left Heart Syndrome; Transpositio..., 'Other left Heart Valvar/Structural Hypoplasia..., 'Transposition of the Great Arteries; Atrioven..., 'Transposition of the Great Arteries; Truncus ...]


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
104,StatusAtTransplant_CAN,CANDIDATE STATUS AT TRANSPLANT OFFER/REMOVALCU...,TRR>TCR,1990-01-01,NaT,WAITING LIST DATA,NUM,STAT,,END_STAT,Category,FMTNAME: CHDMULT - This Feature could be Ordin...


### ETHNICITY

In [175]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'ETHNICITY|ETHCAT', True)

              count unique  top   freq      mean       std  min  25%  50%  75%  max
ETHNICITY   30725.0    NaN  NaN    NaN  0.088365  0.283829  0.0  0.0  0.0  0.0  1.0
ETHCAT      30725.0    NaN  NaN    NaN  1.715281  1.287619  1.0  1.0  1.0  2.0  9.0
ETHCAT_DON    30725      7    1  19535       NaN       NaN  NaN  NaN  NaN  NaN  NaN

NaNs:
ETHNICITY     0
ETHCAT        0
ETHCAT_DON    0
dtype: int64

Datatypes:
ETHNICITY     int64
ETHCAT        int64
ETHCAT_DON      str
dtype: object


        Feature                                      Description            FormSection DataType SASAnalysisFormat Comment Information
106      ETHCAT                     RECIPIENT ETHNICITY CATEGORY  CANDIDATE INFORMATION      NUM            ETHCAT             Unknown
107  ETHCAT_DON                         DONOR ETHNICITY CATEGORY      DONOR INFORMATION      NUM                               Unknown
108   ETHNICITY  RECIPIENT ETHNICITY (HISPANIC VS. NON-HISPANIC)  CANDIDATE INFORMATION      NUM       

In [176]:
# change datatype
df[features] = df[features].astype(int)

# df_flat FMTNAME: ETHCAT
mapping = {
    1: 'White, Non-Hispanic',
    2: 'Black',
    4: 'Hispanic',
    5: 'Asian',
    6: 'Amer Ind/Alaska Native',
    7: 'Native Hawaiian/other Pacific Islander',
    9: 'Multiracial'
}

# map
df = uf.mapping_columns(df, 'ETHCAT', mapping, False)
df = uf.mapping_columns(df, 'ETHCAT_DON', mapping, False).copy()

# df_flat FMTNAME: ETHN (2 as Non-Hispanic/Non-Latino and Infer to 0)
mapping = {
    1: 'Hispanic/Latino',
    0: 'Non-Hispanic/Non-Latino'
}

# map
df = uf.mapping_columns(df, 'ETHNICITY', mapping, False)


# mapping
colMap = {'ETHCAT': 'Ethnicity_CAN', 'ETHCAT_DON':'Ethnicity_DON', 'ETHNICITY':'Hispanic_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='FMTNAME: ETHCAT')
df_dict = uf.update_dictionary_information(df_dict, [119], txt=f"FMTNAME: ETHN (2 as Non-Hispanic/Non-Latino and Infer 0 as Non-Hispanic/Non-Latino)").copy()

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
106,Ethnicity_CAN,RECIPIENT ETHNICITY CATEGORY,TCR-CALCULATED,NaT,NaT,CANDIDATE INFORMATION,NUM,ETHCAT,,ETHCAT,Category,FMTNAME: ETHCAT
107,Ethnicity_DON,DONOR ETHNICITY CATEGORY,DDR/LDR-CALCULATED,NaT,NaT,DONOR INFORMATION,NUM,,,ETHCAT_DON,Category,FMTNAME: ETHCAT
108,Hispanic_CAN,RECIPIENT ETHNICITY (HISPANIC VS. NON-HISPANIC),TCR,1994-04-01,NaT,CANDIDATE INFORMATION,NUM,ETHN,,ETHNICITY,Category,FMTNAME: ETHCAT


### VENTILATOR

In [177]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'VENTILATOR', True)

                  count unique  top   freq      mean      std  min  25%  50%  75%  max
VENTILATOR_TCR  30725.0    NaN  NaN    NaN  0.014841  0.12092  0.0  0.0  0.0  0.0  1.0
VENTILATOR_TRR    30725      2    0  30253       NaN      NaN  NaN  NaN  NaN  NaN  NaN

NaNs:
VENTILATOR_TCR    0
VENTILATOR_TRR    0
dtype: int64

Datatypes:
VENTILATOR_TCR    int64
VENTILATOR_TRR      str
dtype: object


            Feature                                          Description            FormSection DataType SASAnalysisFormat Comment Information
299  VENTILATOR_TCR  PATIENT ON LIFE SUPPORT - VENTILATOR @ REGISTRATION  CANDIDATE INFORMATION      NUM                               Unknown
300  VENTILATOR_TRR    PATIENT ON LIFE SUPPORT - VENTILATOR @ TRANSPLANT         PATIENT STATUS      NUM                               Unknown


VENTILATOR_TCR: [0 1]
VENTILATOR_TRR: <ArrowStringArray>
['0', '1']
Length: 2, dtype: str


In [178]:
# change datatypes
df[features] = df[features].astype(int)

# mapping
colMap = {'VENTILATOR_TCR': 'VentilatorRegistration_CAN', 'VENTILATOR_TRR':'VentilatorTransplant_CAN'}

# df_flat FMTNAME: STAT
mapping = {
  0: 'No',
  1: 'Yes'
}

# map
for feature in features:
    df = uf.mapping_columns(df, feature, mapping, True)


# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt='')

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column VENTILATOR_TCR Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']
Converted Column VENTILATOR_TRR Unique Vaue(s) ['No', 'Yes']
Categories (2, str): ['No', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
299,VentilatorRegistration_CAN,PATIENT ON LIFE SUPPORT - VENTILATOR @ REGISTR...,TCR,1994-04-01,NaT,CANDIDATE INFORMATION,NUM,,,VENTILATOR_TCR,Category,
300,VentilatorTransplant_CAN,PATIENT ON LIFE SUPPORT - VENTILATOR @ TRANSPLANT,TRR,1987-10-01,NaT,PATIENT STATUS,NUM,,,VENTILATOR_TRR,Category,


### PROC_TY_HR

In [179]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'PROC_TY_HR', True)

            count unique top   freq
PROC_TY_HR  30302      4   1  24487

NaNs:
PROC_TY_HR    423
dtype: int64

Datatypes:
PROC_TY_HR    str
dtype: object


        Feature                    Description                      FormSection DataType SASAnalysisFormat Comment Information
231  PROC_TY_HR  PROCEDURE TYPE FOR HEART ONLY  TRANSPLANT CLINICAL INFORMATION      NUM           HR_PROC             Unknown


PROC_TY_HR: <ArrowStringArray>
['1', '2', '3', '4', nan]
Length: 5, dtype: str


In [180]:
# fill NaN with X: Missing
df[features] = df[features].fillna(999).astype(int)

# df_flat FMTNAME: HR_PROC
mapping = {
    1: "Orthotopic Bicaval",
    2: "Orthotopic Traditional",
    3: "Orthotopic Total (Bicaval, PV)",
    4: "Heterotopic",
    999: "Unknown"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'PROC_TY_HR':'HeartProcedureType_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknown/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column PROC_TY_HR Unique Vaue(s) ['Orthotopic Bicaval', 'Orthotopic Traditional', 'Orthotopic Total (Bicaval, PV)', 'Heterotopic', 'Unknown']
Categories (5, str): ['Heterotopic', 'Orthotopic Bicaval', 'Orthotopic Total (Bicaval, PV)', 'Orthotopic Traditional', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
231,HeartProcedureType_CAN,PROCEDURE TYPE FOR HEART ONLY,TRR,1999-10-25,NaT,TRANSPLANT CLINICAL INFORMATION,NUM,HR_PROC,,PROC_TY_HR,Category,N/Y/U/X to No/Yes/Unknown/Missing


### REGION

In [181]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'REGION', True)

          count      mean       std  min  25%  50%  75%   max
REGION  30725.0  5.972889  3.150287  1.0  3.0  5.0  9.0  11.0

NaNs:
REGION    0
dtype: int64

Datatypes:
REGION    int64
dtype: object


    Feature                            Description FormSection DataType SASAnalysisFormat Comment Information
261  REGION  UNOS REGION WHERE TRANSPLANTED/LISTED                  NUM                               Unknown


REGION: [ 5  4  9 11  1 10  7  3  2  8  6]


In [182]:
# mapping
colMap = {'REGION': 'TransplantRegion_CAN'}

# feature value mapping
mapping = {1: 'Region 1', 2: 'Region 2', 3: 'Region 3', 4: 'Region 4', 5: 'Region 5',
           6: 'Region 6', 7: 'Region 7', 8: 'Region 8', 9: 'Region 9', 10: 'Region 10',
           11: 'Region 11'
          }

# iterate
for feature in features:
    # mapping feature
    df = uf.mapping_columns(df, feature, mapping, display=True)


# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"UNKNOWN No Mapping Information.")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column REGION Unique Vaue(s) ['Region 5', 'Region 4', 'Region 9', 'Region 11', 'Region 1', ..., 'Region 7', 'Region 3', 'Region 2', 'Region 8', 'Region 6']
Length: 11
Categories (11, str): ['Region 1', 'Region 10', 'Region 11', 'Region 2', ..., 'Region 6', 'Region 7', 'Region 8', 'Region 9']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
261,TransplantRegion_CAN,UNOS REGION WHERE TRANSPLANTED/LISTED,CALCULATED,NaT,NaT,,NUM,,,REGION,Category,UNKNOWN No Mapping Information.


### Work Income

In [183]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'WORK', True)

                 count unique top   freq
WORK_INCOME_TCR  30445      3   N  24442
WORK_INCOME_TRR  30256      3   N  25759

NaNs:
WORK_INCOME_TCR    280
WORK_INCOME_TRR    469
dtype: int64

Datatypes:
WORK_INCOME_TCR    str
WORK_INCOME_TRR    str
dtype: object


             Feature                               Description            FormSection DataType SASAnalysisFormat Comment Information
304  WORK_INCOME_TCR          WORK FOR INCOME AT REGISTRATION?  CANDIDATE INFORMATION  CHAR(1)                               Unknown
305  WORK_INCOME_TRR  RECIPIENT WORK FOR INCOME AT TRANSPLANT?         PATIENT STATUS  CHAR(1)                               Unknown


WORK_INCOME_TCR: <ArrowStringArray>
['N', 'Y', nan, 'U']
Length: 4, dtype: str
WORK_INCOME_TRR: <ArrowStringArray>
['N', 'U', 'Y', nan]
Length: 4, dtype: str


In [184]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'WORK_INCOME_TCR': 'WorkIncomeRegistration_CAN', 'WORK_INCOME_TRR':'WorkIncomeTransplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.loc[idx]

Converted Column WORK_INCOME_TCR Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column WORK_INCOME_TRR Unique Vaue(s) ['No', 'Unknown', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
304,WorkIncomeRegistration_CAN,WORK FOR INCOME AT REGISTRATION?,TCR,2004-06-30,NaT,CANDIDATE INFORMATION,CHAR(1),,,WORK_INCOME_TCR,Category,
305,WorkIncomeTransplant_CAN,RECIPIENT WORK FOR INCOME AT TRANSPLANT?,TRR,2004-06-30,NaT,PATIENT STATUS,CHAR(1),,,WORK_INCOME_TRR,Category,


### DQ

In [185]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DQ', True)

     count unique top   freq
DQ1  30725     25   0  20812
DQ2  30725     25   0  22002

NaNs:
DQ1    0
DQ2    0
dtype: int64

Datatypes:
DQ1    str
DQ2    str
dtype: object


   Feature                                                      Description        FormSection DataType SASAnalysisFormat Comment Information
82     DQ1  Candidate Most Recent/at Removal DQB1 Antigen From Waiting List  WAITING LIST DATA      NUM             DQHLA             Unknown
83     DQ2  Candidate Most Recent/at Removal DQB2 Antigen From Waiting List  WAITING LIST DATA      NUM             DQHLA             Unknown


DQ1: <ArrowStringArray>
[  '0',   '6',   '2',   '4',   '5',   '8',   '7',   '3',   '9',   '1', '303',
 '302', '201', '202', '319', '301', '501', '602', '401', '609', '402', '502',
 '603', '503', '604']
Length: 25, dtype: str
DQ2: <ArrowStringArray>
[  '0',   '6',   '5',   '7',   '8',   '9',   '4',   '2',   '1',   '3', '603',
 '602', '201', '301', '501', '609', '303', '202', '502', '402', '604',

In [186]:
# convert datatype
df[features] = df[features].astype(int)

# df_flat FMTNAME: DQHLA
mapping = {
    0: 'No Antigen',
    1: '1',
    2: '2',
    3: '3',
    4: '4',
    5: '5',
    6: '6',
    7: '7',
    8: '8',
    9: '9',
    97: 'Unknown',
    98: 'No second antigen detected',
    99: 'Not Tested',
    201: '02:01',
    202: '02:02',
    301: '03:01',
    302: '03:02',
    303: '03:03',
    319: '03:19',
    401: '04:01',
    402: '04:02',
    501: '05:01',
    502: '05:02',
    503: '05:03',
    601: '06:01',
    602: '06:02',
    603: '06:03',
    604: '06:04',
    609: '06:09'
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'DQ1': 'AntigenDQ1_CAN', 'DQ2':'AntigenDQ2_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: DQHLA")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column DQ1 Unique Vaue(s) ['No Antigen', '6', '2', '4', '5', ..., '04:02', '05:02', '06:03', '05:03', '06:04']
Length: 25
Categories (25, str): ['02:01', '02:02', '03:01', '03:02', ..., '7', '8', '9', 'No Antigen']
Converted Column DQ2 Unique Vaue(s) ['No Antigen', '6', '5', '7', '8', ..., '06:04', '03:02', '05:03', '03:19', '06:01']
Length: 25
Categories (25, str): ['02:01', '02:02', '03:01', '03:02', ..., '7', '8', '9', 'No Antigen']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
82,AntigenDQ1_CAN,Candidate Most Recent/at Removal DQB1 Antigen ...,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,DQHLA,,DQ1,Category,FMTNAME: DQHLA
83,AntigenDQ2_CAN,Candidate Most Recent/at Removal DQB2 Antigen ...,WAITING LIST DATA,1987-10-01,NaT,WAITING LIST DATA,NUM,DQHLA,,DQ2,Category,FMTNAME: DQHLA


### MEDICAL CONDITION

In [187]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'MED_COND', True)

              count unique top   freq
MED_COND_TRR  30325      3   3  14532

NaNs:
MED_COND_TRR    400
dtype: int64

Datatypes:
MED_COND_TRR    str
dtype: object


          Feature                                                Description     FormSection DataType SASAnalysisFormat Comment Information
203  MED_COND_TRR  RECIPIENT MEDICAL CONDITION PRE-TRANSPLANT   @ TRANSPLANT  PATIENT STATUS      NUM           MEDCOND             Unknown


MED_COND_TRR: <ArrowStringArray>
['1', '3', '2', nan]
Length: 4, dtype: str


In [188]:
# fill NaN with 999: Missing
df[features] = df[features].fillna(999).astype(int)

# df_flat FMTNAME: MEDCOND
mapping = {
    1: "In Intensive Care Unit",
    2: "Hospitalized Not in ICU",
    3: "Not Hospitalized",
    999: "Unknown"
}

# map
df = uf.mapping_columns(df, 'MED_COND_TRR', mapping, False).copy()


# mapping
colMap = {'MED_COND_TRR': 'MedicalConditionTransplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: MEDCOND")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
203,MedicalConditionTransplant_CAN,RECIPIENT MEDICAL CONDITION PRE-TRANSPLANT @...,TRR,1987-10-01,NaT,PATIENT STATUS,NUM,MEDCOND,,MED_COND_TRR,Category,FMTNAME: MEDCOND


### STATUS

In [189]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'STATUS_', True)

            count unique top   freq
STATUS_TRR  30725      2   V  30223
STATUS_TCR  30725      2   V  30644
STATUS_DDR  30722      3   V  30495

NaNs:
STATUS_TRR    0
STATUS_TCR    0
STATUS_DDR    3
dtype: int64

Datatypes:
STATUS_TRR    str
STATUS_TCR    str
STATUS_DDR    str
dtype: object


        Feature      Description FormSection DataType SASAnalysisFormat Comment Information
267  STATUS_DDR  DDR Form Status              CHAR(1)           FRMSTAT             Unknown
268  STATUS_TCR  TCR Form Status              CHAR(1)           FRMSTAT             Unknown
269  STATUS_TRR  TRR Form Status              CHAR(1)           FRMSTAT             Unknown


STATUS_TRR: <ArrowStringArray>
['V', 'E']
Length: 2, dtype: str
STATUS_TCR: <ArrowStringArray>
['V', 'E']
Length: 2, dtype: str
STATUS_DDR: <ArrowStringArray>
['V', 'S', nan, 'E']
Length: 4, dtype: str


In [190]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# mapping
colMap = {}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"{UNKNOWN}")

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
267,STATUS_DDR,DDR Form Status,DDR,NaT,NaT,,CHAR(1),FRMSTAT,,STATUS_DDR,Category,** UNKNOWN **
268,STATUS_TCR,TCR Form Status,TCR,NaT,NaT,,CHAR(1),FRMSTAT,,STATUS_TCR,Category,** UNKNOWN **
269,STATUS_TRR,TRR Form Status,TRR,NaT,NaT,,CHAR(1),FRMSTAT,,STATUS_TRR,Category,** UNKNOWN **


### DRUG & COCAINE & TATTOOS

In [191]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DRUG_DON|COCAINE|TATTOOS', True)

                     count unique top   freq
HIST_COCAINE_DON     30518      3   N  23366
CONTIN_COCAINE_DON    6605      3   Y   3410
CONTIN_OTH_DRUG_DON  16730      3   Y  12564
HIST_OTH_DRUG_DON    30518      3   Y  16730
TATTOOS              30518      3   Y  16927

NaNs:
HIST_COCAINE_DON         207
CONTIN_COCAINE_DON     24120
CONTIN_OTH_DRUG_DON    13995
HIST_OTH_DRUG_DON        207
TATTOOS                  207
dtype: int64

Datatypes:
HIST_COCAINE_DON       str
CONTIN_COCAINE_DON     str
CONTIN_OTH_DRUG_DON    str
HIST_OTH_DRUG_DON      str
TATTOOS                str
dtype: object


                 Feature                                                   Description    FormSection DataType SASAnalysisFormat Comment Information
42    CONTIN_COCAINE_DON          DECEASED DONOR-HISTORY OF COCAINE USE+RECENT 6MO USE  DONOR HISTORY  CHAR(1)                               Unknown
43   CONTIN_OTH_DRUG_DON  DECEASED DONOR-HISTORY OF OTHER DRUGS IN PAST+RECENT 6MO USE  DONOR HISTORY  C

In [192]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'CONTIN_COCAINE_DON': 'CocaineUse_DON', 'HIST_COCAINE_DON':'PastCocaineUse_DON', 'TATTOOS':'Tatoos_DON',
          'CONTIN_OTH_DRUG_DON': 'OtherDrugUse_DON', 'HIST_OTH_DRUG_DON':'PastOtherDrugUse_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknow/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column HIST_COCAINE_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column CONTIN_COCAINE_DON Unique Vaue(s) ['Unknown', 'No', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column CONTIN_OTH_DRUG_DON Unique Vaue(s) ['Unknown', 'Yes', 'No']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column HIST_OTH_DRUG_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column TATTOOS Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
42,CocaineUse_DON,DECEASED DONOR-HISTORY OF COCAINE USE+RECENT 6...,DDR,1999-10-25,NaT,DONOR HISTORY,CHAR(1),,,CONTIN_COCAINE_DON,Category,N/Y/U/X to No/Yes/Unknow/Missing
43,OtherDrugUse_DON,DECEASED DONOR-HISTORY OF OTHER DRUGS IN PAST+...,DDR,1994-04-01,NaT,DONOR HISTORY,CHAR(1),,,CONTIN_OTH_DRUG_DON,Category,N/Y/U/X to No/Yes/Unknow/Missing
147,PastCocaineUse_DON,DECEASED DONOR-HISTORY OF COCAINE USE IN PAST,DDR,1999-10-25,NaT,DONOR HISTORY,CHAR(1),,,HIST_COCAINE_DON,Category,N/Y/U/X to No/Yes/Unknow/Missing
151,PastOtherDrugUse_DON,DECEASED DONOR-HISTORY OF OTHER DRUG USE IN PAST,DDR,1994-04-01,NaT,DONOR HISTORY,CHAR(1),,,HIST_OTH_DRUG_DON,Category,N/Y/U/X to No/Yes/Unknow/Missing
271,Tatoos_DON,DECEASED DONOR-TATOOS,DDR,1999-10-25,NaT,DONOR HISTORY,CHAR(1),,,TATTOOS,Category,N/Y/U/X to No/Yes/Unknow/Missing


### LIFE SUPPORT

In [193]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'LIFE_SUP', True)

              count unique top   freq
LIFE_SUP_TCR  30650      2   Y  19303
LIFE_SUP_TRR  30328      2   Y  24803

NaNs:
LIFE_SUP_TCR     75
LIFE_SUP_TRR    397
dtype: int64

Datatypes:
LIFE_SUP_TCR    str
LIFE_SUP_TRR    str
dtype: object


          Feature                                           Description            FormSection DataType SASAnalysisFormat Comment Information
193  LIFE_SUP_TCR                 CANDIDATE LIFE SUPPORT @ REGISTRATION  CANDIDATE INFORMATION  CHAR(1)                               Unknown
194  LIFE_SUP_TRR  RECIPIENT LIFE SUPPORT PRE-TRANSPLANT   @ TRANSPLANT         PATIENT STATUS  CHAR(1)                               Unknown


LIFE_SUP_TCR: <ArrowStringArray>
['Y', 'N', nan]
Length: 3, dtype: str
LIFE_SUP_TRR: <ArrowStringArray>
['Y', 'N', nan]
Length: 3, dtype: str


In [194]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'LIFE_SUP_TCR': 'LifeSupportRegistration_CAN', 'LIFE_SUP_TRR':'LifeSupportTransplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknow/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column LIFE_SUP_TCR Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column LIFE_SUP_TRR Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
193,LifeSupportRegistration_CAN,CANDIDATE LIFE SUPPORT @ REGISTRATION,CALCULATED TCR,1987-10-01,NaT,CANDIDATE INFORMATION,CHAR(1),,,LIFE_SUP_TCR,Category,N/Y/U/X to No/Yes/Unknow/Missing
194,LifeSupportTransplant_CAN,RECIPIENT LIFE SUPPORT PRE-TRANSPLANT @ TRAN...,CALCULATED TRR,1987-10-01,NaT,PATIENT STATUS,CHAR(1),,,LIFE_SUP_TRR,Category,N/Y/U/X to No/Yes/Unknow/Missing


### LUNG SURGERY

In [195]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'PRIOR_LUNG', True)

                     count unique top   freq
PRIOR_LUNG_SURG_TRR  30303      3   N  30068

NaNs:
PRIOR_LUNG_SURG_TRR    422
dtype: int64

Datatypes:
PRIOR_LUNG_SURG_TRR    str
dtype: object


                 Feature                                                       Description                         FormSection DataType SASAnalysisFormat Comment Information
230  PRIOR_LUNG_SURG_TRR  TRR LUNG SURGERY BETWEEN LISTING AND TRANSPLANT (NON-TRANSPLANT)  PRETRANSPLANT CLINICAL INFORMATION  CHAR(1)                               Unknown


PRIOR_LUNG_SURG_TRR: <ArrowStringArray>
['N', 'Y', 'U', nan]
Length: 4, dtype: str


In [196]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'PRIOR_LUNG_SURG_TRR': 'PriorLungSurgeryAfterRegistration_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknow/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column PRIOR_LUNG_SURG_TRR Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
230,PriorLungSurgeryAfterRegistration_CAN,TRR LUNG SURGERY BETWEEN LISTING AND TRANSPLAN...,TRR,1999-10-25,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(1),,,PRIOR_LUNG_SURG_TRR,Category,N/Y/U/X to No/Yes/Unknow/Missing


### STEROID

In [197]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'STEROID', True)

                 count unique top   freq
STEROID          30296      3   N  27854
PT_STEROIDS_DON  30519      3   Y  21819

NaNs:
STEROID            429
PT_STEROIDS_DON    206
dtype: int64

Datatypes:
STEROID            str
PT_STEROIDS_DON    str
dtype: object


             Feature                                                         Description                         FormSection DataType SASAnalysisFormat Comment Information
243  PT_STEROIDS_DON  DECEASED DONOR-STEROIDS B/N BRAIN DEATH W/IN 24 HRS OF PROCUREMENT                CLINICAL INFORMATION  CHAR(1)                               Unknown
270          STEROID                              CHRONIC STEROID USE Y/N/U @ TRANSPLANT  PRETRANSPLANT CLINICAL INFORMATION  CHAR(1)                               Unknown


STEROID: <ArrowStringArray>
['N', 'Y', 'U', nan]
Length: 4, dtype: str
PT_STEROIDS_DON: <ArrowStringArray>
['N', 'Y', nan, 'U']
Length: 4, dtype: str


In [198]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'PT_STEROIDS_DON': 'SteroidsUse_DON','STEROID':'SteroidsUse_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Ubkbown/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column STEROID Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column PT_STEROIDS_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
243,SteroidsUse_DON,DECEASED DONOR-STEROIDS B/N BRAIN DEATH W/IN 2...,DDR,1994-04-01,NaT,CLINICAL INFORMATION,CHAR(1),,,PT_STEROIDS_DON,Category,N/Y/U/X to No/Yes/Ubkbown/Missing
270,SteroidsUse_CAN,CHRONIC STEROID USE Y/N/U @ TRANSPLANT,TRR,1994-04-01,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(1),,,STEROID,Category,N/Y/U/X to No/Yes/Ubkbown/Missing


### BILIRUBIN

In [199]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'TBILI', False)

           count unique   top  freq
TBILI      30241    340  0.50  3429
TBILI_DON  30515    270  0.50  3291

NaNs:
TBILI        484
TBILI_DON    210
dtype: int64

Datatypes:
TBILI        str
TBILI_DON    str
dtype: object


       Feature                                     Description                         FormSection DataType SASAnalysisFormat Comment Information
272      TBILI  MOST RECENT SERUM TOTAL BILIRUBIN @ TRANSPLANT  PRETRANSPLANT CLINICAL INFORMATION      NUM                               Unknown
273  TBILI_DON         DECEASED DONOR-TERMINAL TOTAL BILIRUBIN                CLINICAL INFORMATION      NUM                               Unknown




In [200]:
# change datatype
df[features] = df[features].astype(float)
# mapping
colMap = {'TBILI': 'TotalBilirubinTransplant_CAN','TBILI_DON':'TerminalTotalBilirubin_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"")

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
272,TotalBilirubinTransplant_CAN,MOST RECENT SERUM TOTAL BILIRUBIN @ TRANSPLANT,TRR,1994-04-01,NaT,PRETRANSPLANT CLINICAL INFORMATION,NUM,,,TBILI,Numeric,
273,TerminalTotalBilirubin_DON,DECEASED DONOR-TERMINAL TOTAL BILIRUBIN,DDR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,,TBILI_DON,Numeric,


### TRANSFUSIONS

In [201]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'TRANSFUSIONS', False)

              count unique top   freq
TRANSFUSIONS  30301      3   N  23524

NaNs:
TRANSFUSIONS    424
dtype: int64

Datatypes:
TRANSFUSIONS    str
dtype: object


          Feature                                                          Description                         FormSection DataType SASAnalysisFormat Comment Information
279  TRANSFUSIONS  EVENTS OCCURRING BETWEEN LISTING AND TRANSPLANT: TRANSFUSIONS Y/N/U  PRETRANSPLANT CLINICAL INFORMATION  CHAR(1)                               Unknown




In [202]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'TRANSFUSIONS': 'TransfusionAfterRegistration_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Ubkbown/Missing")

# display
df_dict.iloc[idx]

Converted Column TRANSFUSIONS Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
279,TransfusionAfterRegistration_CAN,EVENTS OCCURRING BETWEEN LISTING AND TRANSPLAN...,TRR,1994-04-01,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(1),,,TRANSFUSIONS,Category,N/Y/U/X to No/Yes/Ubkbown/Missing


### TransfusionNumber

In [203]:
df_flat[['CODE','LABEL']][df_flat.FMTNAME.fillna('').str.contains('(?i)TRANSFUS')]

,CODE,LABEL
36264,Null or Missing,Not Reported
36265,0,NONE
36266,1,1 - 5
36267,2,6 - 10
36268,3,GREATER THAN 10
36269,998,UNKNOWN
36270,**OTHER**,Unknown


In [204]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'TRANSFUS_TERM_DON', False)

                   count unique top   freq
TRANSFUS_TERM_DON  30696      5   0  14555

NaNs:
TRANSFUS_TERM_DON    29
dtype: int64

Datatypes:
TRANSFUS_TERM_DON    str
dtype: object


               Feature                                                         Description           FormSection DataType SASAnalysisFormat Comment Information
278  TRANSFUS_TERM_DON  DDR:Number of transfusions during this (terminal) hospitalization:  CLINICAL INFORMATION      NUM          TRANSFUS             Unknown




In [205]:
# fill NaN with 999: Missing
df[features] = df[features].fillna(999).astype(int)

# df_flat FMTNAME: TRANSFUS
mapping = {
    0: "NONE",
    1: "1 - 5",
    2: "6 - 10",
    3: "GREATER THAN 10",
    998: "Unknown",
    999: "Unknown"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'TRANSFUS_TERM_DON':'TransfusionNumber_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"")

# display
df_dict.iloc[idx]

Converted Column TRANSFUS_TERM_DON Unique Vaue(s) ['1 - 5', '6 - 10', 'NONE', 'GREATER THAN 10', 'Unknown']
Categories (5, str): ['1 - 5', '6 - 10', 'GREATER THAN 10', 'NONE', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
278,TransfusionNumber_DON,DDR:Number of transfusions during this (termin...,DDR,2004-06-30,NaT,CLINICAL INFORMATION,NUM,TRANSFUS,,TRANSFUS_TERM_DON,Category,


### VENTILATORY

In [206]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'VENT', False)

                         count unique top   freq
VENT_SUPPORT_TRR         30301      3   N  24307
VENT_SUPPORT_AFTER_LIST  30301      3   N  24307

NaNs:
VENT_SUPPORT_TRR           424
VENT_SUPPORT_AFTER_LIST    424
dtype: int64

Datatypes:
VENT_SUPPORT_TRR           str
VENT_SUPPORT_AFTER_LIST    str
dtype: object


                     Feature                                                                      Description                         FormSection DataType SASAnalysisFormat Comment        Information
297  VENT_SUPPORT_AFTER_LIST  EVENTS OCCURRING BETWEEN LISTING AND TRANSPLANT: EPISODE OF VENTILATORY SUPPORT  PRETRANSPLANT CLINICAL INFORMATION  CHAR(1)                            FMTNAME: CIGDURAB
298         VENT_SUPPORT_TRR                                               TRR EPISODE OF VENTILATORY SUPPORT                                      CHAR(1)                                      Unknown




In [207]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'VENT_SUPPORT_AFTER_LIST': 'VentilatorySupportAfterRegistration_CAN', 'VENT_SUPPORT_TRR': 'VentilatorySupport_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Ubkbown/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column VENT_SUPPORT_TRR Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column VENT_SUPPORT_AFTER_LIST Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
297,VentilatorySupportAfterRegistration_CAN,EVENTS OCCURRING BETWEEN LISTING AND TRANSPLAN...,TRR,1999-10-25,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(1),,,VENT_SUPPORT_AFTER_LIST,Category,N/Y/U/X to No/Yes/Ubkbown/Missing
298,VentilatorySupport_CAN,TRR EPISODE OF VENTILATORY SUPPORT,NaN,NaT,NaT,,CHAR(1),,,VENT_SUPPORT_TRR,Category,N/Y/U/X to No/Yes/Ubkbown/Missing


### EPSTEIN BARR

In [208]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'EBV', True)

                 count unique top   freq
EBV_SEROSTATUS   30310      4   P  24839
EBV_IGG_CAD_DON  30700      6   P  27398
EBV_IGM_CAD_DON  30695      6   N  24687

NaNs:
EBV_SEROSTATUS     415
EBV_IGG_CAD_DON     25
EBV_IGM_CAD_DON     30
dtype: int64

Datatypes:
EBV_SEROSTATUS     str
EBV_IGG_CAD_DON    str
EBV_IGM_CAD_DON    str
dtype: object


            Feature                                           Description           FormSection DataType SASAnalysisFormat Comment Information
91  EBV_IGG_CAD_DON  DECEASED DONOR EPSTEIN BARR VIRUS BY IGG TEST RESULT  CLINICAL INFORMATION  CHAR(2)           SERSTAT             Unknown
92  EBV_IGM_CAD_DON  DECEASED DONOR EPSTEIN BARR VIRUS BY IGM TEST RESULT  CLINICAL INFORMATION  CHAR(2)           SERSTAT             Unknown
93   EBV_SEROSTATUS                     RECIPIENT EBV STATUS @ TRANSPLANT                        CHAR(2)           SERSTAT             Unknown


EBV_SEROSTATUS: <ArrowStringArray>
['ND', 'P', 'N', 'U', nan]
Length: 5, dty

In [209]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# df_flat FMTNAME: SERSTAT
SERSTAT = {
    'C': 'Cannot Disclose',
    'I': 'Indeterminate',
    'N': 'Negative',
    'ND': 'Not Done',
    'P': 'Positive',
    'PD': 'Pending',
    'U': 'Unknown'
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, SERSTAT, display=True)


# mapping
colMap = {'EBV_IGG_CAD_DON':'EpsteinBarr_IGG_DON','EBV_IGM_CAD_DON':'EpsteinBarr_IGM_DON', 'EBV_SEROSTATUS':'EpsteinBarrSeroStatusTransplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: SERSTAT")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column EBV_SEROSTATUS Unique Vaue(s) ['Not Done', 'Positive', 'Negative', 'Unknown']
Categories (4, str): ['Negative', 'Not Done', 'Positive', 'Unknown']
Converted Column EBV_IGG_CAD_DON Unique Vaue(s) ['Positive', 'Negative', 'Not Done', 'Indeterminate', 'Pending', 'Unknown']
Categories (6, str): ['Indeterminate', 'Negative', 'Not Done', 'Pending', 'Positive', 'Unknown']
Converted Column EBV_IGM_CAD_DON Unique Vaue(s) ['Negative', 'Not Done', 'Positive', 'Indeterminate', 'Pending', 'Unknown']
Categories (6, str): ['Indeterminate', 'Negative', 'Not Done', 'Pending', 'Positive', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
91,EpsteinBarr_IGG_DON,DECEASED DONOR EPSTEIN BARR VIRUS BY IGG TEST ...,DDR,2006-05-03,NaT,CLINICAL INFORMATION,CHAR(2),SERSTAT,,EBV_IGG_CAD_DON,Category,FMTNAME: SERSTAT
92,EpsteinBarr_IGM_DON,DECEASED DONOR EPSTEIN BARR VIRUS BY IGM TEST ...,DDR,2006-05-03,NaT,CLINICAL INFORMATION,CHAR(2),SERSTAT,,EBV_IGM_CAD_DON,Category,FMTNAME: SERSTAT
93,EpsteinBarrSeroStatusTransplant_CAN,RECIPIENT EBV STATUS @ TRANSPLANT,CALCULATED,NaT,NaT,,CHAR(2),SERSTAT,,EBV_SEROSTATUS,Category,FMTNAME: SERSTAT


### HBV

In [210]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'HBV', True)

                     count unique top   freq
HBV_CORE             30317      4   N  27407
HBV_SUR_ANTIGEN      30316      4   N  29304
HBV_SURF_TOTAL       19393      4   N  13460
HBV_NAT              13241      4  ND  10698
HBV_CORE_DON         30725      5   N  30089
HBV_SUR_ANTIGEN_DON  30709      4   N  30669
HBV_NAT_DON          20185      4   N  20051

NaNs:
HBV_CORE                 408
HBV_SUR_ANTIGEN          409
HBV_SURF_TOTAL         11332
HBV_NAT                17484
HBV_CORE_DON               0
HBV_SUR_ANTIGEN_DON       16
HBV_NAT_DON            10540
dtype: int64

Datatypes:
HBV_CORE               str
HBV_SUR_ANTIGEN        str
HBV_SURF_TOTAL         str
HBV_NAT                str
HBV_CORE_DON           str
HBV_SUR_ANTIGEN_DON    str
HBV_NAT_DON            str
dtype: object


                 Feature                                        Description                         FormSection DataType SASAnalysisFormat Comment                                                      

In [211]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, SERSTAT, display=True)


# mapping
colMap = {'HBV_CORE': 'Hepatitis_B_CoreAntibody_CAN','HBV_CORE_DON':'Hepatitis_B_CoreAntibody_DON', 
          'HBV_NAT':'HBV_NAT_Result_CAN', 'HBV_NAT_DON':'HBV_NAT_Result_DON', 
          'HBV_SURF_TOTAL':'SurfaceHBVAntibodyTotalTransplant_CAN', 
          'HBV_SUR_ANTIGEN':'SurfaceAntigenHEP_B_CAN', 'HBV_SUR_ANTIGEN_DON':'SurfaceAntigenHEP_B_DON'}


# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: SERSTAT")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column HBV_CORE Unique Vaue(s) ['Negative', 'Positive', 'Not Done', 'Unknown']
Categories (4, str): ['Negative', 'Not Done', 'Positive', 'Unknown']
Converted Column HBV_SUR_ANTIGEN Unique Vaue(s) ['Negative', 'Positive', 'Not Done', 'Unknown']
Categories (4, str): ['Negative', 'Not Done', 'Positive', 'Unknown']
Converted Column HBV_SURF_TOTAL Unique Vaue(s) ['Unknown', 'Negative', 'Not Done', 'Positive']
Categories (4, str): ['Negative', 'Not Done', 'Positive', 'Unknown']
Converted Column HBV_NAT Unique Vaue(s) ['Unknown', 'Not Done', 'Negative', 'Positive']
Categories (4, str): ['Negative', 'Not Done', 'Positive', 'Unknown']
Converted Column HBV_CORE_DON Unique Vaue(s) ['Negative', 'Positive', 'Not Done', 'Pending', 'Indeterminate']
Categories (5, str): ['Indeterminate', 'Negative', 'Not Done', 'Pending', 'Positive']
Converted Column HBV_SUR_ANTIGEN_DON Unique Vaue(s) ['Negative', 'Indeterminate', 'Unknown', 'Not Done', 'Positive']
Categories (5, str): ['Indeterminate', 'Neg

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
119,Hepatitis_B_CoreAntibody_CAN,RECIPIENT HEPATITIS B-CORE ANTIBODY,TRR,1987-10-01,NaT,,CHAR(2),SERSTAT,,HBV_CORE,Category,FMTNAME: SERSTAT
120,Hepatitis_B_CoreAntibody_DON,DONOR HBV CORE ANTIBODY,DDR/LDR,1994-04-01,NaT,,CHAR(2),SERSTAT,,HBV_CORE_DON,Category,FMTNAME: SERSTAT
121,HBV_NAT_Result_CAN,TRR HBV NAT RESULT,TRR,2018-02-28,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(2),,,HBV_NAT,Category,FMTNAME: SERSTAT
122,HBV_NAT_Result_DON,DDR HBV NAT Results:,DDR,2015-03-31,NaT,CLINICAL INFORMATION,CHAR(2),SERSTAT,,HBV_NAT_DON,Category,FMTNAME: SERSTAT
123,SurfaceAntigenHEP_B_CAN,RECIPIENT HEP B SURFACE ANTIGEN,TRR,1987-10-01,NaT,,CHAR(2),SERSTAT,,HBV_SUR_ANTIGEN,Category,FMTNAME: SERSTAT
124,SurfaceAntigenHEP_B_DON,DONOR HEP B SURFACE ANTIGEN,DDR/LDR,1987-10-01,NaT,,CHAR(2),SERSTAT,,HBV_SUR_ANTIGEN_DON,Category,FMTNAME: SERSTAT
125,SurfaceHBVAntibodyTotalTransplant_CAN,RECIPIENT HBV Surface Antibody Total @ TRANSPLANT,TRR,2015-03-31,NaT,CLINICAL INFORMATION,CHAR(2),SERSTAT,,HBV_SURF_TOTAL,Category,FMTNAME: SERSTAT


### CMV

In [212]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'CMV', True)

            count unique top   freq
CMV_STATUS  30318      4   P  17152
CMV_IGG     10538      4   P   6133
CMV_IGM     10537      4   N   7072
CMV_DON     30716      5   P  18811

NaNs:
CMV_STATUS      407
CMV_IGG       20187
CMV_IGM       20188
CMV_DON           9
dtype: int64

Datatypes:
CMV_STATUS    str
CMV_IGG       str
CMV_IGM       str
CMV_DON       str
dtype: object


       Feature                                                     Description                         FormSection DataType SASAnalysisFormat                                                                       Comment Information
35     CMV_DON  DONOR SEROLOGY ANTI CMV (FOR LIVING DONOR, PRE UNET DATA ONLY)                CLINICAL INFORMATION  CHAR(2)           SERSTAT  START DATE: DECEASED DONORS: 10/1/87-PRESENT; LIVING DONORS 10/1/90-10/25/99     Unknown
36     CMV_IGG                  RECIPIENT-CMV BY IGG TEST RESULT  @ TRANSPLANT  PRETRANSPLANT CLINICAL INFORMATION  CHAR(2)           SERSTAT               

In [213]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, SERSTAT, display=True)


# mapping
colMap = {'CMV_DON': 'SerologyAntiCMV_DON','CMV_IGG': 'CMV_IGG_Transplant_CAN', 'CMV_IGM': 'CMV_IGM_Transplant_CAN', 'CMV_STATUS': 'CMVStatus_Transplant_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column CMV_STATUS Unique Vaue(s) ['Positive', 'Negative', 'Unknown', 'Not Done']
Categories (4, str): ['Negative', 'Not Done', 'Positive', 'Unknown']
Converted Column CMV_IGG Unique Vaue(s) ['Positive', 'Negative', 'Not Done', 'Unknown']
Categories (4, str): ['Negative', 'Not Done', 'Positive', 'Unknown']
Converted Column CMV_IGM Unique Vaue(s) ['Negative', 'Not Done', 'Unknown', 'Positive']
Categories (4, str): ['Negative', 'Not Done', 'Positive', 'Unknown']
Converted Column CMV_DON Unique Vaue(s) ['Positive', 'Negative', 'Indeterminate', 'Not Done', 'Unknown']
Categories (5, str): ['Indeterminate', 'Negative', 'Not Done', 'Positive', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
35,SerologyAntiCMV_DON,"DONOR SEROLOGY ANTI CMV (FOR LIVING DONOR, PRE...",DDR/LDR,NaT,NaT,CLINICAL INFORMATION,CHAR(2),SERSTAT,START DATE: DECEASED DONORS: 10/1/87-PRESENT; ...,CMV_DON,Category,
36,CMV_IGG_Transplant_CAN,RECIPIENT-CMV BY IGG TEST RESULT @ TRANSPLANT,TRR,1999-10-25,2015-03-31,PRETRANSPLANT CLINICAL INFORMATION,CHAR(2),SERSTAT,,CMV_IGG,Category,
37,CMV_IGM_Transplant_CAN,RECIPIENT-CMV BY IGM TEST RESULT @ TRANSPLANT,TRR,1999-10-25,2015-03-31,PRETRANSPLANT CLINICAL INFORMATION,CHAR(2),SERSTAT,,CMV_IGM,Category,
38,CMVStatus_Transplant_CAN,RECIPIENT CMV Status @ TRANSPLANT,TRR,2015-03-31,NaT,CLINICAL INFORMATION,CHAR(2),SERSTAT,,CMV_STATUS,Category,


### HIV

In [214]:
# display feature info
features, idx = uf.feature_information(df, df_dict, '^HIV', True)

                count unique top   freq
HIV_SEROSTATUS  30316      4   N  29447
HIV_NAT         13242      4  ND  10878
HIV_NAT_DON     20185      3   N  20123

NaNs:
HIV_SEROSTATUS      409
HIV_NAT           17483
HIV_NAT_DON       10540
dtype: int64

Datatypes:
HIV_SEROSTATUS    str
HIV_NAT           str
HIV_NAT_DON       str
dtype: object


            Feature                             Description                         FormSection DataType SASAnalysisFormat Comment Information
152         HIV_NAT                      TRR HIV NAT RESULT  PRETRANSPLANT CLINICAL INFORMATION  CHAR(2)                               Unknown
153     HIV_NAT_DON                    DDR HIV NAT Results:                CLINICAL INFORMATION  CHAR(2)           SERSTAT             Unknown
154  HIV_SEROSTATUS  RECIPIENT HIV SEROSTATUS AT TRANSPLANT                CLINICAL INFORMATION  CHAR(2)                               Unknown


HIV_SEROSTATUS: <ArrowStringArray>
['N', 'U', 'ND', 'P', nan]
Length: 5, dtype: 

In [215]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, SERSTAT, display=True)


# mapping
colMap = {'HIV_NAT':'HIV_NAT_PreTransplant_CAN','HIV_NAT_DON':'HIV_NAT_Result_DON', 'HIV_SEROSTATUS':'HIV_SeroStatusTransplant_CAN'}


# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: SERSTAT")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column HIV_SEROSTATUS Unique Vaue(s) ['Negative', 'Unknown', 'Not Done', 'Positive']
Categories (4, str): ['Negative', 'Not Done', 'Positive', 'Unknown']
Converted Column HIV_NAT Unique Vaue(s) ['Unknown', 'Not Done', 'Negative', 'Positive']
Categories (4, str): ['Negative', 'Not Done', 'Positive', 'Unknown']
Converted Column HIV_NAT_DON Unique Vaue(s) ['Unknown', 'Negative', 'Not Done']
Categories (3, str): ['Negative', 'Not Done', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
152,HIV_NAT_PreTransplant_CAN,TRR HIV NAT RESULT,TRR,2018-02-28,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(2),,,HIV_NAT,Category,FMTNAME: SERSTAT
153,HIV_NAT_Result_DON,DDR HIV NAT Results:,DDR,2015-03-31,NaT,CLINICAL INFORMATION,CHAR(2),SERSTAT,,HIV_NAT_DON,Category,FMTNAME: SERSTAT
154,HIV_SeroStatusTransplant_CAN,RECIPIENT HIV SEROSTATUS AT TRANSPLANT,TRR,1987-10-01,NaT,CLINICAL INFORMATION,CHAR(2),,,HIV_SEROSTATUS,Category,FMTNAME: SERSTAT


### HCV

In [216]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'HCV', True)

                count unique top   freq
HCV_SEROSTATUS  30315      4   N  28977
HCV_NAT         13243      4  ND   9984
HCV_NAT_DON     20185      5   N  19231

NaNs:
HCV_SEROSTATUS      410
HCV_NAT           17482
HCV_NAT_DON       10540
dtype: int64

Datatypes:
HCV_SEROSTATUS    str
HCV_NAT           str
HCV_NAT_DON       str
dtype: object


            Feature             Description                         FormSection DataType SASAnalysisFormat Comment Information
126         HCV_NAT      TRR HCV NAT RESULT  PRETRANSPLANT CLINICAL INFORMATION  CHAR(2)                               Unknown
127     HCV_NAT_DON    DDR HCV NAT Results:                CLINICAL INFORMATION  CHAR(2)           SERSTAT             Unknown
128  HCV_SEROSTATUS  RECIPIENT HEP C STATUS                                      CHAR(2)           SERSTAT             Unknown


HCV_SEROSTATUS: <ArrowStringArray>
['N', 'U', 'P', 'ND', nan]
Length: 5, dtype: str
HCV_NAT: <ArrowStringArray>
[nan, 'ND', 'U', 'N', 'P']
Lengt

In [217]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, SERSTAT, display=True)


# mapping
colMap = {'HCV_NAT': 'HCV_NAT_PreTranspant_CAN','HCV_NAT_DON':'HCV_NAT_Result_DON', 'HCV_SEROSTATUS':'HEP_C_SerostatusStatus_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: SERSTAT")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column HCV_SEROSTATUS Unique Vaue(s) ['Negative', 'Unknown', 'Positive', 'Not Done']
Categories (4, str): ['Negative', 'Not Done', 'Positive', 'Unknown']
Converted Column HCV_NAT Unique Vaue(s) ['Unknown', 'Not Done', 'Negative', 'Positive']
Categories (4, str): ['Negative', 'Not Done', 'Positive', 'Unknown']
Converted Column HCV_NAT_DON Unique Vaue(s) ['Unknown', 'Negative', 'Not Done', 'Positive', 'Indeterminate']
Categories (5, str): ['Indeterminate', 'Negative', 'Not Done', 'Positive', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
126,HCV_NAT_PreTranspant_CAN,TRR HCV NAT RESULT,TRR,2018-02-28,NaT,PRETRANSPLANT CLINICAL INFORMATION,CHAR(2),,,HCV_NAT,Category,FMTNAME: SERSTAT
127,HCV_NAT_Result_DON,DDR HCV NAT Results:,DDR,2015-03-31,NaT,CLINICAL INFORMATION,CHAR(2),SERSTAT,,HCV_NAT_DON,Category,FMTNAME: SERSTAT
128,HEP_C_SerostatusStatus_CAN,RECIPIENT HEP C STATUS,TRR,1994-04-01,NaT,,CHAR(2),SERSTAT,,HCV_SEROSTATUS,Category,FMTNAME: SERSTAT


### HBSAB_DON & VDRL_DON
- HBsAb (Hepatitis B Surface Antibody) is a blood test that detects antibodies produced by the immune system in response to the hepatitis B virus (HBV)
- RPR (Rapid Plasma Reagin) and VDRL (Venereal Disease Research Laboratory) are both nontreponemal tests used to screen for syphilis.

In [218]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'HBSAB_DON|VDRL', True)

           count unique top   freq
VDRL_DON   30704      5   N  29931
HBSAB_DON  30679      6  ND  25949

NaNs:
VDRL_DON     21
HBSAB_DON    46
dtype: int64

Datatypes:
VDRL_DON     str
HBSAB_DON    str
dtype: object


       Feature                       Description           FormSection DataType SASAnalysisFormat Comment Information
118  HBSAB_DON  DECEASED DONOR HBSAB TEST RESULT  CLINICAL INFORMATION  CHAR(2)           SERSTAT             Unknown
296   VDRL_DON    DECEASED DONOR-RPR-VDRL RESULT  CLINICAL INFORMATION  CHAR(2)           SERSTAT             Unknown


VDRL_DON: <ArrowStringArray>
['N', 'ND', 'P', nan, 'I', 'U']
Length: 6, dtype: str
HBSAB_DON: <ArrowStringArray>
['ND', 'P', 'N', 'U', 'I', nan, 'C']
Length: 7, dtype: str


In [219]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, SERSTAT, display=True)


# mapping
colMap = {'HBSAB_DON':'AntibodyResultHBSAB_DON', 'VDRL_DON': 'AntibodyResultRPR_VDRL_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: SERSTAT")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column VDRL_DON Unique Vaue(s) ['Negative', 'Not Done', 'Positive', 'Unknown', 'Indeterminate']
Categories (5, str): ['Indeterminate', 'Negative', 'Not Done', 'Positive', 'Unknown']
Converted Column HBSAB_DON Unique Vaue(s) ['Not Done', 'Positive', 'Negative', 'Unknown', 'Indeterminate', 'Cannot Disclose']
Categories (6, str): ['Cannot Disclose', 'Indeterminate', 'Negative', 'Not Done', 'Positive', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
118,AntibodyResultHBSAB_DON,DECEASED DONOR HBSAB TEST RESULT,DDR,2006-05-03,NaT,CLINICAL INFORMATION,CHAR(2),SERSTAT,,HBSAB_DON,Category,FMTNAME: SERSTAT
296,AntibodyResultRPR_VDRL_DON,DECEASED DONOR-RPR-VDRL RESULT,DDR,1987-10-01,NaT,CLINICAL INFORMATION,CHAR(2),SERSTAT,,VDRL_DON,Category,FMTNAME: SERSTAT


### HEP_C_ANTI_DON

In [220]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'HEP_C_ANTI_DON', True)

                count unique top   freq
HEP_C_ANTI_DON  30722      5   N  29271

NaNs:
HEP_C_ANTI_DON    3
dtype: int64

Datatypes:
HEP_C_ANTI_DON    str
dtype: object


            Feature                                    Description           FormSection DataType SASAnalysisFormat Comment Information
140  HEP_C_ANTI_DON  DECEASED DONOR-ANTIBODY TO HEP C VIRUS RESULT  CLINICAL INFORMATION  CHAR(2)           SERSTAT             Unknown


HEP_C_ANTI_DON: <ArrowStringArray>
['N', 'P', nan, 'PD', 'ND', 'I']
Length: 6, dtype: str


In [221]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, SERSTAT, display=True)


# mapping
colMap = {'HEP_C_ANTI_DON': 'Antibody_HEP_C_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: SERSTAT")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column HEP_C_ANTI_DON Unique Vaue(s) ['Negative', 'Positive', 'Unknown', 'Pending', 'Not Done', 'Indeterminate']
Categories (6, str): ['Indeterminate', 'Negative', 'Not Done', 'Pending', 'Positive', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
140,Antibody_HEP_C_DON,DECEASED DONOR-ANTIBODY TO HEP C VIRUS RESULT,DDR,1994-04-01,NaT,CLINICAL INFORMATION,CHAR(2),SERSTAT,,HEP_C_ANTI_DON,Category,FMTNAME: SERSTAT


### CDC_RISK

In [222]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'CDC_RISK', True)

                  count unique top   freq
CDC_RISK_HIV_DON  30721      3   N  22967

NaNs:
CDC_RISK_HIV_DON    4
dtype: int64

Datatypes:
CDC_RISK_HIV_DON    str
dtype: object


             Feature                                                                           Description    FormSection DataType SASAnalysisFormat Comment Information
28  CDC_RISK_HIV_DON  DDR: Per PHS, Does the Donor Have Risk Factors for Blood-Borne Disease Transmission?  DONOR HISTORY  CHAR(1)                               Unknown


CDC_RISK_HIV_DON: <ArrowStringArray>
['N', 'Y', nan, 'U']
Length: 4, dtype: str


In [223]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'CDC_RISK_HIV_DON': 'HIV_Risk_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknown/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column CDC_RISK_HIV_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
28,HIV_Risk_DON,"DDR: Per PHS, Does the Donor Have Risk Factors...",DDR,2004-06-30,NaT,DONOR HISTORY,CHAR(1),,,CDC_RISK_HIV_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing


### TX (PROCEDURE TYPE)

In [224]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'TX_', True)

                  count unique  top   freq         mean       std     min     25%     50%     75%     max
TX_PROCEDUR_TY    30725      1  501  30725          NaN       NaN     NaN     NaN     NaN     NaN     NaN
TX_TYPE           30302      2    O  30287          NaN       NaN     NaN     NaN     NaN     NaN     NaN
TX_YEAR         30725.0    NaN  NaN    NaN  2016.143206  3.413931  2010.0  2013.0  2016.0  2019.0  2021.0

NaNs:
TX_PROCEDUR_TY      0
TX_TYPE           423
TX_YEAR             0
dtype: int64

Datatypes:
TX_PROCEDUR_TY      str
TX_TYPE             str
TX_YEAR           int64
dtype: object


            Feature                            Description FormSection DataType SASAnalysisFormat Comment Information
282  TX_PROCEDUR_TY  RECIPIENT PROCEDURE TYPE - CALCULATED                  NUM            THPROC             Unknown
283         TX_TYPE                     TYPE OF TRANSPLANT              CHAR(1)        TX_TYPE_TH             Unknown
284         TX_YEAR                 

In [225]:
# fill NaN with X: Missing
df['TX_TYPE'] = df['TX_TYPE'].fillna('Unknown')

# SASAnalysisFormat: THPROC
mapping = { 
    501: "Heart"
}

# mapping feature
df = uf.mapping_columns(df, 'TX_PROCEDUR_TY', mapping, display=True)


# SASAnalysisFormat: TX_TYPE_TH
mapping = { 
    'D': 'Double',
    'H': 'Heterotopic',
    'O': 'Orthotopic',
    'S': 'Single',
    'X': 'Unknown'
}

# mapping feature
df = uf.mapping_columns(df, 'TX_PROCEDUR_TY', mapping, display=True)


# mapping
colMap = {'TX_PROCEDUR_TY':'TransplantProcedure_CAN','TX_TYPE':'TransplantType_CAN','TX_YEAR':'TransplantYear_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"YYYY")
df_dict = uf.update_dictionary_information(df_dict, [279], txt=f'SASAnalysisFormat: TX_TYPE_TH', feature_type='Category')
df_dict = uf.update_dictionary_information(df_dict, [278], txt=f'{DROP} SASAnalysisFormat: THPROC - Same Value', feature_type='Category').copy()

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column TX_PROCEDUR_TY Unique Vaue(s) ['501']
Categories (1, str): ['501']
Converted Column TX_PROCEDUR_TY Unique Vaue(s) ['501']
Categories (1, str): ['501']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
282,TransplantProcedure_CAN,RECIPIENT PROCEDURE TYPE - CALCULATED,CALCULATED,NaT,NaT,,NUM,THPROC,,TX_PROCEDUR_TY,Numeric,YYYY
283,TransplantType_CAN,TYPE OF TRANSPLANT,CALCULATED,NaT,NaT,,CHAR(1),TX_TYPE_TH,,TX_TYPE,Numeric,YYYY
284,TransplantYear_CAN,TRANSPLANT YEAR,CALCULATED,NaT,NaT,,NUM,,,TX_YEAR,Numeric,YYYY


### RETYP

In [226]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'RETYP', True)

           count unique top   freq
DON_RETYP  30044      2   Y  18232

NaNs:
DON_RETYP    681
dtype: int64

Datatypes:
DON_RETYP    str
dtype: object


      Feature                          Description       FormSection DataType SASAnalysisFormat Comment Information
80  DON_RETYP  DECEASED DONOR-RETYPED AT TX CENTER  TEST INFORMATION  CHAR(1)                               Unknown


DON_RETYP: <ArrowStringArray>
['N', 'Y', nan]
Length: 3, dtype: str


In [227]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'DON_RETYP': 'DeceasedRetyped_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/X to No/Yes/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column DON_RETYP Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
80,DeceasedRetyped_DON,DECEASED DONOR-RETYPED AT TX CENTER,RH,1987-10-01,NaT,TEST INFORMATION,CHAR(1),,,DON_RETYP,Category,N/Y/X to No/Yes/Missing


### CRSMATCH

In [228]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'CRSMATCH', True)

               count unique top   freq
CRSMATCH_DONE  30046      2   Y  28294

NaNs:
CRSMATCH_DONE    679
dtype: int64

Datatypes:
CRSMATCH_DONE    str
dtype: object


          Feature          Description       FormSection DataType SASAnalysisFormat Comment Information
49  CRSMATCH_DONE  CROSSMATCH DONE Y/N  TEST INFORMATION  CHAR(1)                               Unknown


CRSMATCH_DONE: <ArrowStringArray>
['N', 'Y', nan]
Length: 3, dtype: str


In [229]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'CRSMATCH_DONE': 'CrossMatchDone'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/X to No/Yes/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column CRSMATCH_DONE Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
49,CrossMatchDone,CROSSMATCH DONE Y/N,RH,1994-04-01,NaT,TEST INFORMATION,CHAR(1),,,CRSMATCH_DONE,Category,N/Y/X to No/Yes/Missing


### CPRA

###### PanelReactiveAntibody
- The Panel Reactive Antibody (PRA), or Calculated Panel Reactive Antibody (cPRA), is a metric used in organ transplantation.
    - cPRA = 0%: The recipient is unlikely to have antibodies against most potential donors and is considered less "sensitized," meaning they have a broad range of compatible donor options.
    - Higher cPRA (e.g., 80%+): The recipient has a high level of sensitization, reducing the likelihood of finding compatible donors.
    - cPRA = 100%: The recipient has antibodies against nearly all potential donors, making finding a compatible organ highly challenging.

- CPRA values typically range from 0% to 100%.
    - The results are often grouped into categories, such as:
        - 0%       No Sensitization
        - 1-20%    Low Sensitization
        - 21-50%   Some Sensitization
        - 51-80%   Moderate Sensitization
        - 81-98%   High Sensitization
        - 99-100%  Extreme Sensitization
    - Higher CPRA values indicate a higher degree of sensitization:
    - CPRA of 80-100% is considered highly sensitized
    - CPRA >98% may receive extra priority in organ allocation system

In [230]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'CPRA', False)

           count unique top   freq
CPRA       15524    101   0  10579
CPRA_PEAK  15512    101   0   9302

NaNs:
CPRA         15201
CPRA_PEAK    15213
dtype: int64

Datatypes:
CPRA         str
CPRA_PEAK    str
dtype: object


      Feature                 Description           FormSection DataType SASAnalysisFormat Comment Information
45       CPRA  Recipient Most Recent CPRA  CLINICAL INFORMATION      NUM                               Unknown
46  CPRA_PEAK          RecipientPeak CPRA  CLINICAL INFORMATION      NUM                               Unknown




In [231]:
# change datatype
df[features] = df[features].astype(float)
# mapping
colMap = {'CPRA': 'CPRA_Recent_CAN', 'CPRA_PEAK':'CPRA_Peak_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"")

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
45,CPRA_Recent_CAN,Recipient Most Recent CPRA,RH,2015-03-31,NaT,CLINICAL INFORMATION,NUM,,,CPRA,Numeric,
46,CPRA_Peak_CAN,RecipientPeak CPRA,RH,2015-03-31,NaT,CLINICAL INFORMATION,NUM,,,CPRA_PEAK,Numeric,


### DA1 & DA2
- DA1 and DA2 refer to specific epitopes associated with HLA-DA molecules, which are a part of the major histocompatibility complex (MHC) class II. HLA-DA molecules play critical roles in the immune system by presenting peptide antigens to CD4+ T helper cells.

In [232]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DA\d', False)

     count unique top   freq
DA1  30722     40   2  12262
DA2  30712     46  24   3399

NaNs:
DA1     3
DA2    13
dtype: int64

Datatypes:
DA1    str
DA2    str
dtype: object


   Feature       Description                             FormSection DataType SASAnalysisFormat Comment Information
50     DA1  DONOR A1 ANTIGEN  DONOR CENTER HISTOCOMPATIBILITY TYPING      NUM            ALOCUS             Unknown
51     DA2  DONOR A2 ANTIGEN  DONOR CENTER HISTOCOMPATIBILITY TYPING      NUM            ALOCUS             Unknown




In [233]:
# fill NaN with X: Missing
df[features] = df[features].fillna(999).astype(int)

# df_flat FMTNAME: ALOCUS
ALOCUS = {
    999: "Unknown",
    0: "No Antigen",
    1: "1",
    2: "2",
    3: "3",
    9: "9",
    10: "10",
    11: "11",
    19: "19",
    23: "23",
    24: "24",
    25: "25",
    26: "26",
    28: "28",
    29: "29",
    30: "30",
    31: "31",
    32: "32",
    33: "33",
    34: "34",
    36: "36",
    43: "43",
    66: "66",
    68: "68",
    69: "69",
    74: "74",
    80: "80",
    97: "Unknown",
    98: "No second antigen detected",
    99: "Not Tested",
    101: "01:01",
    102: "01:02",
    201: "02:01",
    202: "02:02",
    203: "02:03",
    205: "02:05",
    206: "02:06",
    207: "02:07",
    210: "210",
    211: "02:10",
    218: "02:18",
    301: "03:01",
    302: "03:02",
    1101: "11:01",
    1102: "11:02",
    2402: "24:02",
    2403: "24:03",
    2601: "26:01",
    2602: "26:02",
    2603: "26:03",
    2901: "29:01",
    2902: "29:02",
    3001: "30:01",
    3002: "30:02",
    3204: "32:04",
    3301: "33:01",
    3303: "33:03",
    3401: "34:01",
    3402: "34:02",
    6601: "66:01",
    6602: "66:02",
    6801: "68:01",
    6802: "68:02"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, ALOCUS, display=True)


# mapping
colMap = {'DA1': 'AntigenDA1_DON', 'DA2':'AntigenDA2_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: ALOCUS")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column DA1 Unique Vaue(s) ['2', '33', '3', '36', '1', ..., '26:01', '02:02', '03:02', '01:02', '28']
Length: 41
Categories (41, str): ['01:01', '01:02', '02:01', '02:02', ..., '68:02', '69', '74', 'Unknown']
Converted Column DA2 Unique Vaue(s) ['29', '68', '3', '24', '74', ..., '26:01', '01:01', '02:06', '29:01', 'No Antigen']
Length: 47
Categories (47, str): ['01:01', '02:01', '02:03', '02:05', ..., '80', 'No Antigen', 'No second antigen detected', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
50,AntigenDA1_DON,DONOR A1 ANTIGEN,CALCULATED,1987-10-01,NaT,DONOR CENTER HISTOCOMPATIBILITY TYPING,NUM,ALOCUS,,DA1,Category,FMTNAME: ALOCUS
51,AntigenDA2_DON,DONOR A2 ANTIGEN,CALCULATED,1987-10-01,NaT,DONOR CENTER HISTOCOMPATIBILITY TYPING,NUM,ALOCUS,,DA2,Category,FMTNAME: ALOCUS


### DB1 & DB2

In [234]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DB\d', False)

     count unique top  freq
DB1  30722     83   7  6132
DB2  30709     90  44  4319

NaNs:
DB1     3
DB2    16
dtype: int64

Datatypes:
DB1    str
DB2    str
dtype: object


   Feature       Description                             FormSection DataType SASAnalysisFormat Comment Information
65     DB1  DONOR B1 ANTIGEN  DONOR CENTER HISTOCOMPATIBILITY TYPING      NUM            BLOCUS             Unknown
66     DB2  DONOR B2 ANTIGEN  DONOR CENTER HISTOCOMPATIBILITY TYPING      NUM            BLOCUS             Unknown




In [235]:
# fill NaN with X: Missing
df[features] = df[features].fillna(999).astype(int)

# df_flat FMTNAME: BLOCUS
BLOCUS = {
   999: "Unknown",
    0: 'No Antigen',
    5: '5',
    7: '7',
    8: '8',
    12: '12',
    13: '13',
    14: '14',
    15: '15',
    16: '16',
    17: '17',
    18: '18',
    21: '21',
    22: '22',
    27: '27',
    35: '35',
    37: '37',
    38: '38',
    39: '39',
    40: '40',
    41: '41',
    42: '42',
    44: '44',
    45: '45',
    46: '46',
    47: '47',
    48: '48',
    49: '49',
    50: '50',
    51: '51',
    52: '52',
    53: '53',
    54: '54',
    55: '55',
    56: '56',
    57: '57',
    58: '58',
    59: '59',
    60: '60',
    61: '61',
    62: '62',
    63: '63',
    64: '64',
    65: '65',
    67: '67',
    70: '70',
    71: '71',
    72: '72',
    73: '73',
    75: '75',
    76: '76',
    77: '77',
    78: '78',
    81: '81',
    82: '82',
    97: 'Unknown',
    98: 'No second antigen detected',
    99: 'Not Tested',
    702: '07:02',
    703: '703',
    704: '07:03',
    714: '07:14',
    801: '08:01',
    802: '08:02',
    803: '08:03',
    804: '08:04',
    1301: '13:01',
    1302: '13:02',
    1304: '13:04',
    1401: '14:01',
    1402: '14:02',
    1501: '15:01',
    1502: '15:02',
    1503: '15:03',
    1504: '15:04',
    1506: '15:06',
    1507: '15:07',
    1510: '15:10',
    1511: '15:11',
    1512: '15:12',
    1513: '15:13',
    1516: '15:16',
    1517: '15:17',
    1518: '15:18',
    1520: '15:20',
    1521: '15:21',
    1522: '15:22',
    1524: '15:24',
    1527: '15:27',
    2703: '27:03',
    2704: '27:04',
    2705: '27:05',
    2706: '27:06',
    2708: '27:08',
    3501: '35:01',
    3502: '35:02',
    3503: '35:03',
    3508: '35:08',
    3512: '35:12',
    3801: '38:01',
    3802: '38:02',
    3901: '39:01',
    3902: '39:02',
    3904: '39:04',
    3905: '39:05',
    3906: '39:06',
    3913: '39:13',
    4001: '40:01',
    4002: '40:02',
    4003: '40:03',
    4004: '40:04',
    4005: '40:05',
    4006: '40:06',
    4101: '41:01',
    4102: '41:02',
    4201: '42:01',
    4202: '42:02',
    4402: '44:02',
    4403: '44:03',
    4415: '44:15',
    4801: '48:01',
    4802: '48:02',
    5001: '50:01',
    5002: '50:02',
    5101: '51:01',
    5102: '51:02',
    5103: '51:03',
    5501: '55:01',
    5502: '55:02',
    5504: '55:04',
    5601: '56:01',
    5603: '56:03',
    5701: '57:01',
    5703: '57:03',
    7801: '78:01',
    8201: '82:01',
    8301: '83:01'
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, BLOCUS, display=True)
    

# mapping
colMap = {'DB1': 'AntigenDB1_DON', 'DB2':'AntigenDB2_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: BLOCUS")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column DB1 Unique Vaue(s) ['44', '49', '7', '39', '61', ..., '35:02', '15:16', '15:10', '56:03', '82']
Length: 84
Categories (84, str): ['07:02', '08:01', '13', '13:02', ..., '8', '81', '82', 'Unknown']
Converted Column DB2 Unique Vaue(s) ['49', '60', '44', '51', '64', ..., '41:02', '15:16', '42:02', '17', '82:01']
Length: 91
Categories (91, str): ['07:02', '08:01', '12', '13', ..., '82', '82:01', 'No second antigen detected', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
65,AntigenDB1_DON,DONOR B1 ANTIGEN,CALCULATED,1987-10-01,NaT,DONOR CENTER HISTOCOMPATIBILITY TYPING,NUM,BLOCUS,,DB1,Category,FMTNAME: BLOCUS
66,AntigenDB2_DON,DONOR B2 ANTIGEN,CALCULATED,1987-10-01,NaT,DONOR CENTER HISTOCOMPATIBILITY TYPING,NUM,BLOCUS,,DB2,Category,FMTNAME: BLOCUS


### RA1 & RA2

In [236]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'RA\d', False)

     count unique top   freq
RA1  28338     50   2  10339
RA2  28338     51  68   2822

NaNs:
RA1    2387
RA2    2387
dtype: int64

Datatypes:
RA1    str
RA2    str
dtype: object


    Feature           Description           FormSection DataType SASAnalysisFormat Comment Information
252     RA1  RECIPIENT A1 ANTIGEN  RECIPIENT HLA TYPING      NUM            ALOCUS             Unknown
253     RA2  RECIPIENT A2 ANTIGEN  RECIPIENT HLA TYPING      NUM            ALOCUS             Unknown




In [237]:
findMappingDfFlat(df.RA1, df_flat, 'ALOCUS', 999)

Compare Length: 51 & 50

CODE      LABEL
   0          0
   1          1
   2          2
   3          3
  10         10
  11         11
  23         23
  24         24
  25         25
  26         26
  28         28
  29         29
  30         30
  31         31
  32         32
  33         33
  34         34
  36         36
  66         66
  68         68
  69         69
  74         74
  80         80
  99 Not Tested
 101      01:01
 102      01:02
 201      02:01
 202      02:02
 203      02:03
 205      02:05
 206      02:06
 207      02:07
 210        210
 301      03:01
 302      03:02
1101      11:01
2402      24:02
2403      24:03
2601      26:01
2901      29:01
2902      29:02
3001      30:01
3002      30:02
3301      33:01
3303      33:03
3402      34:02
6601      66:01
6602      66:02
6801      68:01
6802      68:02


In [238]:
# fill NaN
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'0': 'No Antigen', '99': 'Not Tested'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'RA1': 'AntigenRA1_CAN', 'RA2':'AntigenRA2_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Number', txt=f"FMTNAME: ALOCUS")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column RA1 Unique Vaue(s) ['24', '30', '2', '3', 'Unknown', ..., '3402', '80', '3301', '6601', '207']
Length: 51
Categories (51, str): ['1', '10', '101', '102', ..., '80', 'No Antigen', 'Not Tested', 'Unknown']
Converted Column RA2 Unique Vaue(s) ['24', '68', '3', '26', 'Unknown', ..., '10', '3401', '205', '302', '1102']
Length: 52
Categories (52, str): ['1', '10', '101', '11', ..., '98', 'No Antigen', 'Not Tested', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
252,AntigenRA1_CAN,RECIPIENT A1 ANTIGEN,RH,1987-10-01,NaT,RECIPIENT HLA TYPING,NUM,ALOCUS,,RA1,Number,FMTNAME: ALOCUS
253,AntigenRA2_CAN,RECIPIENT A2 ANTIGEN,RH,1987-10-01,NaT,RECIPIENT HLA TYPING,NUM,ALOCUS,,RA2,Number,FMTNAME: ALOCUS


### RB1 & RB1

In [239]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'RB\d')

     count unique top  freq
RB1  28338    101   7  5210
RB2  28338     98  44  3715

NaNs:
RB1    2387
RB2    2387
dtype: int64

Datatypes:
RB1    str
RB2    str
dtype: object


    Feature           Description           FormSection DataType SASAnalysisFormat Comment Information
254     RB1  RECIPIENT B1 ANTIGEN  RECIPIENT HLA TYPING      NUM            BLOCUS             Unknown
255     RB2  RECIPIENT B2 ANTIGEN  RECIPIENT HLA TYPING      NUM            BLOCUS             Unknown




In [240]:
findMappingDfFlat(df.RB1, df_flat, 'BLOCUS', 999)

Compare Length: 102 & 101

CODE      LABEL
   0          0
   5          5
   7          7
   8          8
  12         12
  13         13
  14         14
  15         15
  16         16
  18         18
  21         21
  22         22
  27         27
  35         35
  37         37
  38         38
  39         39
  40         40
  41         41
  42         42
  44         44
  45         45
  46         46
  47         47
  48         48
  49         49
  50         50
  51         51
  52         52
  53         53
  54         54
  55         55
  56         56
  57         57
  58         58
  60         60
  61         61
  62         62
  63         63
  64         64
  65         65
  70         70
  71         71
  72         72
  73         73
  75         75
  76         76
  77         77
  78         78
  81         81
  82         82
  99 Not Tested
 702      07:02
 714      07:14
 801      08:01
1301      13:01
1302      13:02
1401      14:01
1402      14:02
1501      15:

In [241]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'0': 'No Antigen', '99': 'Not Tested'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)
    
    
# mapping
colMap = {'RB1': 'AntigenRB1_CAN', 'RB2':'AntigenRB2_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: BLOCUS")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column RB1 Unique Vaue(s) ['35', '48', '7', 'Unknown', '18', ..., '1517', '1513', '1502', '1524', '5102']
Length: 102
Categories (102, str): ['12', '13', '1301', '1302', ..., '82', 'No Antigen', 'Not Tested', 'Unknown']
Converted Column RB2 Unique Vaue(s) ['61', '65', '44', '38', 'Unknown', ..., '4202', '3508', '2703', '2704', '5']
Length: 99
Categories (99, str): ['13', '1302', '14', '1401', ..., '98', 'No Antigen', 'Not Tested', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
254,AntigenRB1_CAN,RECIPIENT B1 ANTIGEN,RH,1987-10-01,NaT,RECIPIENT HLA TYPING,NUM,BLOCUS,,RB1,Category,FMTNAME: BLOCUS
255,AntigenRB2_CAN,RECIPIENT B2 ANTIGEN,RH,1987-10-01,NaT,RECIPIENT HLA TYPING,NUM,BLOCUS,,RB2,Category,FMTNAME: BLOCUS


### [RDR1 & RDR1](https://www.sciencedirect.com/science/article/pii/S0041134503006481?casa_token=ysGMHfKhFkcAAAAA:_oK775b3pZYzCZuzGrA13GO1c9kLTI7SKGOv9panij8OIQLUyKEDVTEeg-nxnp3R9ouIeaB2Wcp7)

In [242]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'RDR\d', True)

      count unique top  freq
RDR1  28338     53   4  5940
RDR2  28338     56  15  5910

NaNs:
RDR1    2387
RDR2    2387
dtype: int64

Datatypes:
RDR1    str
RDR2    str
dtype: object


    Feature            Description           FormSection DataType SASAnalysisFormat Comment Information
256    RDR1  RECIPIENT DR1 ANTIGEN  RECIPIENT HLA TYPING      NUM           DRLOCUS             Unknown
257    RDR2  RECIPIENT DR2 ANTIGEN  RECIPIENT HLA TYPING      NUM           DRLOCUS             Unknown


RDR1: <ArrowStringArray>
[    '4',     '8',    '13',     nan,    '17',    '18',     '1',     '7',
    '15',    '12',    '14',    '11',     '9',    '16',    '10',   '103',
     '3',  '1404',     '0',     '2',   '302',  '1101',  '1301',  '1201',
   '301',  '1104',  '1303',   '401',   '901',   '101',  '1501',  '1503',
   '403',  '1454',   '102',   '407',    '99',   '404',  '1302',   '801',
   '405',   '802',     '6',  '1602',     '5',   '402',   '411',  '1305',
  '1601',  '1502',  '1401',  '1202',  

In [243]:
findMappingDfFlat(df.RDR2, df_flat, 'BLOCUS', 999)

Compare Length: 57 & 22

CODE                      LABEL
   0                          0
   7                          7
   8                          8
  12                         12
  13                         13
  14                         14
  15                         15
  16                         16
  17                         17
  18                         18
  98 No second antigen detected
  99                 Not Tested
 801                      08:01
 802                      08:02
 803                      08:03
1301                      13:01
1302                      13:02
1401                      14:01
1402                      14:02
1501                      15:01
1502                      15:02
1503                      15:03


In [244]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'0': 'No Antigen',
           '98': 'No second antigen detected',
           '99': 'Not Tested'
          }

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, DRLOCUS, display=True)
    
# mapping
colMap = {'RDR1': 'AntigenRDR1_CAN', 'RDR2':'AntigenRDR2_CAN'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: DRLOCUS")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column RDR1 Unique Vaue(s) ['4', '8', '13', 'Unknown', '17', ..., '1502', '1401', '1202', '406', '10300']
Length: 54
Categories (54, str): ['0', '1', '10', '101', ..., '9', '901', '99', 'Unknown']
Converted Column RDR2 Unique Vaue(s) ['4', '9', '13', '11', '14', ..., '1305', '1403', '802', '1406', '10300']
Length: 57
Categories (57, str): ['0', '1', '10', '101', ..., '901', '98', '99', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
256,AntigenRDR1_CAN,RECIPIENT DR1 ANTIGEN,RH,1987-10-01,NaT,RECIPIENT HLA TYPING,NUM,DRLOCUS,,RDR1,Category,FMTNAME: DRLOCUS
257,AntigenRDR2_CAN,RECIPIENT DR2 ANTIGEN,RH,1987-10-01,NaT,RECIPIENT HLA TYPING,NUM,DRLOCUS,,RDR2,Category,FMTNAME: DRLOCUS


### PRAMR

In [245]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'PRAMR', True)

           count unique top  freq
PRAMR_CL1  10035    101   0  7221
PRAMR_CL2   9830    100   0  7785

NaNs:
PRAMR_CL1    20690
PRAMR_CL2    20895
dtype: int64

Datatypes:
PRAMR_CL1    str
PRAMR_CL2    str
dtype: object


       Feature                                        Description           FormSection DataType SASAnalysisFormat Comment Information
220  PRAMR_CL1   RECIPIENT MOST RECENT PRA% CLASS I @ TRANSPLANT   RECIPIENT HLA TYPING      NUM                               Unknown
221  PRAMR_CL2  RECIPIENT MOST RECENT PRA% CLASS II @ TRANSPLANT   RECIPIENT HLA TYPING      NUM                               Unknown


PRAMR_CL1: <ArrowStringArray>
[ '2',  '0',  '7', '11',  nan, '12', '84',  '9', '23', '42',
 ...
 '25', '40', '64', '53', '90', '39', '56', '69', '57', '81']
Length: 102, dtype: str
PRAMR_CL2: <ArrowStringArray>
[ '0', '29',  nan, '10',  '3', '75',  '1', '12', '57', '56',
 ...
 '53', '96', '34', '49', '70', '63', '89', '90', '54', '71']
Length: 101, dtype: str


In [246]:
# change datatypes
df[features] = df[features].astype(float)

# mapping
colMap = {'PRAMR_CL1': 'Class1PRA_TransplantPercentage_CAN', 'PRAMR_CL2':'Class2PRA_TransplantPercentage_CAN'}


# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"")

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
220,Class1PRA_TransplantPercentage_CAN,RECIPIENT MOST RECENT PRA% CLASS I @ TRANSPLANT,RH,2004-06-30,2015-03-31,RECIPIENT HLA TYPING,NUM,,,PRAMR_CL1,Numeric,
221,Class2PRA_TransplantPercentage_CAN,RECIPIENT MOST RECENT PRA% CLASS II @ TRANSPLANT,RH,2004-06-30,2015-03-31,RECIPIENT HLA TYPING,NUM,,,PRAMR_CL2,Numeric,


### DON_TY

In [247]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DON_TY', True)

        count unique top   freq
DON_TY  30725      2   C  30722

NaNs:
DON_TY    0
dtype: int64

Datatypes:
DON_TY    str
dtype: object


   Feature                      Description        FormSection DataType SASAnalysisFormat Comment Information
81  DON_TY  DONOR TYPE - DECEASED OR LIVING  DONOR INFORMATION  CHAR(3)           DON_TYP             Unknown


DON_TY: <ArrowStringArray>
['C', 'L']
Length: 2, dtype: str


In [248]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# df_flat FMTNAME: DON_TYP
mapping = {'C': 'Deceased Donor', 'L': 'Living Donor', 'F': 'Foreign Donor'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'DON_TY': 'DeceasedOrLiving_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: DON_TYP")

# display
df_dict.iloc[idx]

Converted Column DON_TY Unique Vaue(s) ['Deceased Donor', 'Living Donor']
Categories (2, str): ['Deceased Donor', 'Living Donor']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
81,DeceasedOrLiving_DON,DONOR TYPE - DECEASED OR LIVING,TRR,1987-10-01,NaT,DONOR INFORMATION,CHAR(3),DON_TYP,,DON_TY,Category,FMTNAME: DON_TYP


### HRT

In [249]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'HRT', True)

             count unique top   freq
TXHRT        30725      1   Y  30725
NON_HRT_DON  30720      2   N  30402

NaNs:
TXHRT          0
NON_HRT_DON    5
dtype: int64

Datatypes:
TXHRT          str
NON_HRT_DON    str
dtype: object


         Feature                             Description     FormSection DataType SASAnalysisFormat Comment Information
205  NON_HRT_DON  DECEASED DONOR-NON-HEART BEATING DONOR  ORGAN RECOVERY  CHAR(1)                               Unknown
286        TXHRT                      SIMULTANEOUS HEART                  CHAR(1)                               Unknown


TXHRT: <ArrowStringArray>
['Y']
Length: 1, dtype: str
NON_HRT_DON: <ArrowStringArray>
['N', nan, 'Y']
Length: 3, dtype: str


In [250]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'NON_HRT_DON': 'NonHeartBeating_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknown/Missing")
df_dict = uf.update_dictionary_information(df_dict, [282], txt=f'{DROP}').copy()

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column TXHRT Unique Vaue(s) ['Yes']
Categories (1, str): ['Yes']
Converted Column NON_HRT_DON Unique Vaue(s) ['No', 'Unknown', 'Yes']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
205,NonHeartBeating_DON,DECEASED DONOR-NON-HEART BEATING DONOR,DDR,1994-04-01,NaT,ORGAN RECOVERY,CHAR(1),,,NON_HRT_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing
286,TXHRT,SIMULTANEOUS HEART,CALCULATED,NaT,NaT,,CHAR(1),,,TXHRT,Category,N/Y/U/X to No/Yes/Unknown/Missing


### PCO2
- Maintaining appropriate PCO2 levels is crucial for preserving organ function and optimizing outcomes in transplantation. 

In [251]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'PCO2_DON', False)

          count unique    top  freq
PCO2_DON  30488    525  37.00  1165

NaNs:
PCO2_DON    237
dtype: int64

Datatypes:
PCO2_DON    str
dtype: object


      Feature Description     FormSection DataType SASAnalysisFormat Comment Information
211  PCO2_DON   DDR:pCO2:  ORGAN RECOVERY      NUM                               Unknown




In [252]:
# change dataype
df[features] = df[features].astype(float)
# mapping
colMap = {'PCO2_DON': 'OrganRecovery_PCO2_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"Maintaining appropriate PCO2 levels is crucial for preserving \
organ function and optimizing outcomes in transplantation.")

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
211,OrganRecovery_PCO2_DON,DDR:pCO2:,DDR,2004-06-30,NaT,ORGAN RECOVERY,NUM,,,PCO2_DON,Numeric,Maintaining appropriate PCO2 levels is crucial...


### PT_

In [253]:
# display feature info
features, idx = uf.feature_information(df, df_dict, '^PT_', True)

                   count unique         top   freq
PT_DIURETICS_DON   30518      3           Y  21373
PT_T3_DON          30518      3           N  30370
PT_T4_DON          30518      3           Y  20261
PT_OTH2_OSTXT_DON  26425  12546       ZOSYN   1025
PT_OTH3_OSTXT_DON  17962   7666  ROCURONIUM    938
PT_OTH1_OSTXT_DON  29814  14873       ZOSYN   1323

NaNs:
PT_DIURETICS_DON       207
PT_T3_DON              207
PT_T4_DON              207
PT_OTH2_OSTXT_DON     4300
PT_OTH3_OSTXT_DON    12763
PT_OTH1_OSTXT_DON      911
dtype: int64

Datatypes:
PT_DIURETICS_DON     str
PT_T3_DON            str
PT_T4_DON            str
PT_OTH2_OSTXT_DON    str
PT_OTH3_OSTXT_DON    str
PT_OTH1_OSTXT_DON    str
dtype: object


              Feature                                                                    Description           FormSection DataType SASAnalysisFormat Comment    Information
242  PT_DIURETICS_DON            DECEASED DONOR-DIURETICS B/N BRAIN DEATH W/IN 24 HRS OF PROCUREMENT  CLINICAL

In [254]:
# get new features
features = ['PT_DIURETICS_DON','PT_T3_DON','PT_T4_DON']

# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'PT_DIURETICS_DON': 'Diuretics_DON', 'PT_T3_DON': 'TriiodothyronineT3_DON', 'PT_T4_DON':'ThyroxineT4_DON',
         'PT_OTH1_OSTXT_DON':'OtherMedsText1_DON', 'PT_OTH2_OSTXT_DON':'OtherMedsText2_DON','PT_OTH3_OSTXT_DON':'OtherMedsText3_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknown/Missing")

# convert to category
df = uf.convert_to_category(df,  ['Diuretics_DON','TriiodothyronineT3_DON','ThyroxineT4_DON'])

# display
df_dict.iloc[idx]

Converted Column PT_DIURETICS_DON Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column PT_T3_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column PT_T4_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
242,Diuretics_DON,DECEASED DONOR-DIURETICS B/N BRAIN DEATH W/IN ...,DDR,1994-04-01,NaT,CLINICAL INFORMATION,CHAR(1),,,PT_DIURETICS_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing
244,TriiodothyronineT3_DON,DECEASED DONOR-TRIIODOTHYRONINE-T3 B/N BRAIN D...,DDR,1999-10-25,NaT,CLINICAL INFORMATION,CHAR(1),,,PT_T3_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing
245,ThyroxineT4_DON,DECEASED DONOR-THYROXINE-T4 B/N BRAIN DEATH W/...,DDR,1994-04-01,NaT,CLINICAL INFORMATION,CHAR(1),,,PT_T4_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing


### PULM_CATH_DON & PROTEIN_URINE

In [255]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'PULM|PROTEIN_URINE', True)

                   count unique      top   freq
PULM_INF_CONF_DON  30725      3  Unknown  24065
PROTEIN_URINE      30519      3        N  16278
PULM_CATH_DON      30518      2        N  28357

NaNs:
PULM_INF_CONF_DON      0
PROTEIN_URINE        206
PULM_CATH_DON        207
dtype: int64

Datatypes:
PULM_INF_CONF_DON    category
PROTEIN_URINE             str
PULM_CATH_DON             str
dtype: object


               Feature                                          Description           FormSection DataType SASAnalysisFormat Comment                Information
236      PROTEIN_URINE                      DECEASED DONOR PROTEIN IN URINE  CLINICAL INFORMATION  CHAR(1)                                              Unknown
247      PULM_CATH_DON                                    DDR PA CATH (Y,N)        ORGAN RECOVERY  CHAR(1)                                              Unknown
248  PULM_INF_CONF_DON  DECEASED DONOR-INFECTION PULMONARY SOURCE-CONFIRMED  CLINICAL INFORMATION  CHAR(1)         

In [256]:
# update features
features = ['PROTEIN_URINE', 'PULM_CATH_DON']

# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'PULM_CATH_DON':'PulmonaryCatheter_DON', 'PROTEIN_URINE':'UrineProtein_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknown")

# display
df_dict.iloc[idx]

Converted Column PROTEIN_URINE Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column PULM_CATH_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
236,UrineProtein_DON,DECEASED DONOR PROTEIN IN URINE,DDR,1999-10-25,NaT,CLINICAL INFORMATION,CHAR(1),,,PROTEIN_URINE,Category,N/Y/U/X to No/Yes/Unknown
247,PulmonaryCatheter_DON,"DDR PA CATH (Y,N)",DDR,1999-10-25,NaT,ORGAN RECOVERY,CHAR(1),,,PULM_CATH_DON,Category,N/Y/U/X to No/Yes/Unknown
248,PULM_INF_CONF_DON,DECEASED DONOR-INFECTION PULMONARY SOURCE-CONF...,DDR,1994-04-01,2015-03-31,CLINICAL INFORMATION,CHAR(1),,,PULM_INF_CONF_DON,Category,N/Y/U/X to No/Yes/Unknown


### SGOT & SGPT

In [257]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'SGOT|SGPT', True)

          count unique    top freq
SGOT_DON  30514   1059  21.00  544
SGPT_DON  30514   1303  19.00  612

NaNs:
SGOT_DON    211
SGPT_DON    211
dtype: int64

Datatypes:
SGOT_DON    str
SGPT_DON    str
dtype: object


      Feature                       Description           FormSection DataType SASAnalysisFormat Comment Information
263  SGOT_DON  DECEASED DONOR-TERMINAL SGOT/AST  CLINICAL INFORMATION      NUM                               Unknown
264  SGPT_DON  DECEASED DONOR-TERMINAL SGPT/ALT  CLINICAL INFORMATION      NUM                               Unknown


SGOT_DON: <ArrowStringArray>
[  '46.00',   '38.00',  '445.00',  '248.00',  '137.00',   '17.00',   '25.00',
   '85.00',  '132.00',   '21.00',
 ...
  '870.00', '1323.00', '2111.00', '1363.00',  '540.00', '5330.00',  '754.00',
 '4376.00',  '574.00', '2229.00']
Length: 1060, dtype: str
SGPT_DON: <ArrowStringArray>
[  '40.00',  '171.00',  '217.00',  '165.00',  '174.00',   '13.00',   '43.00',
   '54.00',   '57.00',   '10.00',
 ...
 

In [258]:
# change datatype
df[features] = df[features].astype(float)

# mapping
colMap = {'SGOT_DON': 'Level_SGOT_AST_DON', 'SGPT_DON':'Level_SGOT_ALT_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"")

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
263,Level_SGOT_AST_DON,DECEASED DONOR-TERMINAL SGOT/AST,DDR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,,SGOT_DON,Numeric,
264,Level_SGOT_ALT_DON,DECEASED DONOR-TERMINAL SGPT/ALT,DDR,1994-04-01,NaT,CLINICAL INFORMATION,NUM,,,SGPT_DON,Numeric,


### CARDARREST_NEURO & DDAVP_DON

In [259]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'CARDARREST|DDAVP', True)

                  count unique top   freq
DDAVP_DON         30518      3   N  26174
CARDARREST_NEURO  30211      2   N  28025

NaNs:
DDAVP_DON           207
CARDARREST_NEURO    514
dtype: int64

Datatypes:
DDAVP_DON           str
CARDARREST_NEURO    str
dtype: object


             Feature                                             Description           FormSection DataType SASAnalysisFormat Comment Information
27  CARDARREST_NEURO          DECEASED DONOR-CARDIAC ARREST POST BRAIN DEATH  CLINICAL INFORMATION  CHAR(1)                               Unknown
67         DDAVP_DON  DECEASED DONOR-SYNTHETIC ANTI DIURETIC HORMONE (DDAVP)  CLINICAL INFORMATION  CHAR(1)                               Unknown


DDAVP_DON: <ArrowStringArray>
['Y', 'N', nan, 'U']
Length: 4, dtype: str
CARDARREST_NEURO: <ArrowStringArray>
['N', 'Y', nan]
Length: 3, dtype: str


In [260]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# mapping
colMap = {'CARDARREST_NEURO':'CardiacArrest_DON', 'DDAVP_DON': 'SyntheticAntiDiureticHormone_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknown/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column DDAVP_DON Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column CARDARREST_NEURO Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
27,CardiacArrest_DON,DECEASED DONOR-CARDIAC ARREST POST BRAIN DEATH,DDR,1999-10-25,NaT,CLINICAL INFORMATION,CHAR(1),,,CARDARREST_NEURO,Category,N/Y/U/X to No/Yes/Unknown/Missing
67,SyntheticAntiDiureticHormone_DON,DECEASED DONOR-SYNTHETIC ANTI DIURETIC HORMONE...,DDR,1999-10-25,NaT,CLINICAL INFORMATION,CHAR(1),,,DDAVP_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing


### CHEST_XRAY_DON

In [261]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'CHEST_XRAY', True)

                count unique top   freq
CHEST_XRAY_DON  29123      7   5  16047

NaNs:
CHEST_XRAY_DON    1602
dtype: int64

Datatypes:
CHEST_XRAY_DON    str
dtype: object


           Feature     Description     FormSection DataType SASAnalysisFormat Comment Information
30  CHEST_XRAY_DON  DDR CHEST XRAY  ORGAN RECOVERY      NUM          LUNGXRAY             Unknown


CHEST_XRAY_DON: <ArrowStringArray>
['4', '3', '5', '2', '998', nan, '1', '999']
Length: 8, dtype: str


In [262]:
# fill NaN with 1000: Unknown
df[features] = df[features].fillna(1000).astype(int)

# SASAnalysisFormat LUNGXRAY
mapping = {
    1: "No chest x-ray",
    2: "Normal",
    3: "Abnormal-left",
    4: "Abnormal-right",
    5: "Abnormal-both",
    998: "Unknown",
    999: "Unknown if chest x-ray performed",
    1000: "Unknown"
}

# mapping feature
df = uf.mapping_columns(df, 'CHEST_XRAY_DON', mapping, display=True)


# mapping
colMap = {'CHEST_XRAY_DON':'ChestXray_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"SASAnalysisFormat LUNGXRAY N/Y/U/X to No/Yes/Unknown/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column CHEST_XRAY_DON Unique Vaue(s) ['Abnormal-right', 'Abnormal-left', 'Abnormal-both', 'Normal', 'Unknown', 'No chest x-ray', 'Unknown if chest x-ray performed']
Categories (7, str): ['Abnormal-both', 'Abnormal-left', 'Abnormal-right', 'No chest x-ray', 'Normal', 'Unknown', 'Unknown if chest x-ray performed']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
30,ChestXray_DON,DDR CHEST XRAY,DDR,1999-10-25,NaT,ORGAN RECOVERY,NUM,LUNGXRAY,,CHEST_XRAY_DON,Category,SASAnalysisFormat LUNGXRAY N/Y/U/X to No/Yes/U...


### CORONARY_ANGIO

In [263]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'CORONARY_ANGIO', True)

                count unique top   freq
CORONARY_ANGIO  30515      3   1  19542

NaNs:
CORONARY_ANGIO    210
dtype: int64

Datatypes:
CORONARY_ANGIO    str
dtype: object


           Feature                                                                                          Description                     FormSection DataType SASAnalysisFormat Comment Information
44  CORONARY_ANGIO  DECEASED DONOR CORONARY ANGIOGRAM: Y/N IF YES NORMAL: Y/N IF ABNORMAL # VESSELS WITH > 50% STENOSIS  HEART DONOR'S CARDIAC FUNCTION      NUM             ANGIO             Unknown


CORONARY_ANGIO: <ArrowStringArray>
['1', '2', '3', nan]
Length: 4, dtype: str


In [264]:
# fill NaN with 1000: Unknown
df[features] = df[features].fillna(999).astype(int)

# df_flat FMTNAME: ANGIO
mapping = {
    1: "No",
    2: "Yes, normal",
    3: "Yes, not normal",
    999: "Unknown"
}

# mapping feature
df = uf.mapping_columns(df, 'CORONARY_ANGIO', mapping, display=True)


# mapping
colMap = {'CORONARY_ANGIO':'CoronaryAngiogram_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: ANGIO")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column CORONARY_ANGIO Unique Vaue(s) ['No', 'Yes, normal', 'Yes, not normal', 'Unknown']
Categories (4, str): ['No', 'Unknown', 'Yes, normal', 'Yes, not normal']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
44,CoronaryAngiogram_DON,DECEASED DONOR CORONARY ANGIOGRAM: Y/N IF YES ...,DDR,1999-10-25,NaT,HEART DONOR'S CARDIAC FUNCTION,NUM,ANGIO,,CORONARY_ANGIO,Category,FMTNAME: ANGIO


### DISTANCE

In [265]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'DISTANCE|LOS', False)

          count unique top  freq
LOS       29790    270  10  1970
DISTANCE  30725   1058   0  2278

NaNs:
LOS         935
DISTANCE      0
dtype: int64

Datatypes:
LOS         str
DISTANCE    str
dtype: object


      Feature                                             Description                                        FormSection DataType SASAnalysisFormat Comment Information
79   DISTANCE  DISTANCE FROM DONOR HOSP TO TX CENTER (Nautical Miles)                                                         NUM                               Unknown
196       LOS                        RECIPIENT LENGTH OF STAY POST TX  PATIENT STATUS(PRIORITY KIDNEY,THEN PANCREAS TRR)      NUM                               Unknown




In [266]:
# change datatypes
df[features] = df[features].astype(float)
# mapping
colMap = {'DISTANCE':'DistanceFromDonorHospitaltoTXCenter_CAN', 'LOS':'LengthOfStay'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"{LABEL} N/Y/U/X to No/Yes/Unknown/Missing")

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
79,DistanceFromDonorHospitaltoTXCenter_CAN,DISTANCE FROM DONOR HOSP TO TX CENTER (Nautica...,CALCULATED,NaT,NaT,,NUM,,,DISTANCE,Numeric,** LABEL ** N/Y/U/X to No/Yes/Unknown/Missing
196,LengthOfStay,RECIPIENT LENGTH OF STAY POST TX,TRR-CALCULATED,1999-10-25,NaT,"PATIENT STATUS(PRIORITY KIDNEY,THEN PANCREAS TRR)",NUM,,,LOS,Numeric,** LABEL ** N/Y/U/X to No/Yes/Unknown/Missing


### ECD_DONOR

In [267]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'ECD', True)

           count unique top   freq
ECD_DONOR  30722      2   0  29876

NaNs:
ECD_DONOR    3
dtype: int64

Datatypes:
ECD_DONOR    str
dtype: object


      Feature                                            Description FormSection DataType SASAnalysisFormat Comment Information
94  ECD_DONOR  EXPANDED DONOR PER KIDNEY ALLOCATION DEFINITION 1=YES                  NUM                               Unknown


ECD_DONOR: <ArrowStringArray>
['0', '1', nan]
Length: 3, dtype: str


In [268]:
# fill NaN with X: Missing
df[features] = df[features].fillna(999).astype(int)

# mapping
colMap = {'ECD_DONOR':'KidneyAllocation_DON'}

# feature value mapping
mapping = {0: 'No', 1: 'Yes', 999: 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column ECD_DONOR Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
94,KidneyAllocation_DON,EXPANDED DONOR PER KIDNEY ALLOCATION DEFINITIO...,CALCULATED,1994-04-01,NaT,,NUM,,,ECD_DONOR,Category,


### HEMATOCRIT & ISCHTIME
- Hematocrit is often measured alongside hemoglobin levels as part of a complete blood count (CBC) to assess overall blood health and oxygen-carrying capacity
- Ischemic time is the period when an organ is preserved in a hypothermic state before being transplanted. Shortening the ischemic time can reduce the risk of graft failure and patient mortality after transplantation, and may also shorten the hospital stay. The cold ischemic time (CIT) for an organ donor is the time the organ spends outside of the body between procurement and transplantation.

In [269]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'HEMATOCRIT|ISCHTIME|PH_DON', False)

                count unique    top  freq
ISCHTIME        30133     96    3.3  1442
PH_DON          30494     80   7.43  2090
HEMATOCRIT_DON  30519    404  28.00   672

NaNs:
ISCHTIME          592
PH_DON            231
HEMATOCRIT_DON    206
dtype: int64

Datatypes:
ISCHTIME          str
PH_DON            str
HEMATOCRIT_DON    str
dtype: object


            Feature             Description           FormSection DataType SASAnalysisFormat Comment Information
129  HEMATOCRIT_DON         DDR:Hematocrit:  CLINICAL INFORMATION      NUM                               Unknown
190        ISCHTIME  ISCHEMIC TIME IN HOURS                            NUM                               Unknown
216          PH_DON           DDR:Blood PH:  CLINICAL INFORMATION      NUM                               Unknown




In [270]:
# change datatypes
df[features] = df[features].astype(float)
# mapping
colMap = {'HEMATOCRIT_DON':'Hematocrit_DON', 'ISCHTIME':'IschemicTimeHour_DON', 'PH_DON':'BloodPH_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"")

# display
df_dict.iloc[idx]

,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
129,Hematocrit_DON,DDR:Hematocrit:,DDR,2004-06-30,NaT,CLINICAL INFORMATION,NUM,,,HEMATOCRIT_DON,Numeric,
190,IschemicTimeHour_DON,ISCHEMIC TIME IN HOURS,CALCULATED,NaT,NaT,,NUM,,,ISCHTIME,Numeric,
216,BloodPH_DON,DDR:Blood PH:,DDR,2004-06-30,NaT,CLINICAL INFORMATION,NUM,,,PH_DON,Numeric,


### HIST_HYPERTENS & HEPARIN & HIST_MI

In [271]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'HEPARIN|HIST_HYPERTENS|HIST_MI|INSULIN|VASODIL', True)

                    count unique top   freq
VASODIL_DON         30518      3   N  25500
HIST_HYPERTENS_DON  30721      3   N  25792
HEPARIN_DON         30518      3   Y  29995
INSULIN_DON         30519      3   Y  17796
HIST_MI             30517      3   N  30066

NaNs:
VASODIL_DON           207
HIST_HYPERTENS_DON      4
HEPARIN_DON           207
INSULIN_DON           206
HIST_MI               208
dtype: int64

Datatypes:
VASODIL_DON           str
HIST_HYPERTENS_DON    str
HEPARIN_DON           str
INSULIN_DON           str
HIST_MI               str
dtype: object


                Feature                                                                     Description                     FormSection DataType SASAnalysisFormat Comment Information
141         HEPARIN_DON  DECEASED DONOR MANAGEMENT - HEPARIN (NO COLLECTION BETWEEN 10/25/99 - 1/27/03)            CLINICAL INFORMATION  CHAR(1)                               Unknown
149  HIST_HYPERTENS_DON                                       

In [272]:
# fill NaN with X: Missing
df[features] = df[features].fillna('Unknown')

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'HEPARIN_DON':'HeparinManagement_DON', 'HIST_HYPERTENS_DON': 'HypertensionHistory_DON', 'VASODIL_DON':'Vasodilators_DON',
         'HIST_MI':'MyocardialInfarctionHistory_DON', 'INSULIN_DON':'InsulinManagement_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"N/Y/U/X to No/Yes/Unknown/Missing")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column VASODIL_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column HIST_HYPERTENS_DON Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column HEPARIN_DON Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column INSULIN_DON Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']
Converted Column HIST_MI Unique Vaue(s) ['No', 'Yes', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
141,HeparinManagement_DON,DECEASED DONOR MANAGEMENT - HEPARIN (NO COLLEC...,DDR,1994-04-01,NaT,CLINICAL INFORMATION,CHAR(1),,,HEPARIN_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing
149,HypertensionHistory_DON,DECEASED DONOR-HISTORY OF HYPERTENSION,DDR,1994-04-01,NaT,DONOR HISTORY,CHAR(1),,,HIST_HYPERTENS_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing
150,MyocardialInfarctionHistory_DON,DECEASED DONOR HISTORY OF PREVIOUS MI (MYOCARD...,DDR,1999-10-25,NaT,HEART DONOR'S CARDIAC FUNCTION,CHAR(1),,,HIST_MI,Category,N/Y/U/X to No/Yes/Unknown/Missing
187,InsulinManagement_DON,DECEASED DONOR-WAS DONOR GIVEN INSULIN WITHIN ...,DDR,2004-06-30,NaT,CLINICAL INFORMATION,CHAR(1),,,INSULIN_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing
295,Vasodilators_DON,DECEASED DONOR-VASODILATORS W/IN 24HRS PRE-CRO...,DDR,1994-04-01,NaT,CLINICAL INFORMATION,CHAR(1),,,VASODIL_DON,Category,N/Y/U/X to No/Yes/Unknown/Missing


### LV_EJECT & LV_EJECT_METH

In [273]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'LV_EJECT', True)

               count unique    top   freq
LV_EJECT_METH  30675      2      1  30030
LV_EJECT       30689    153  60.00   7288

NaNs:
LV_EJECT_METH    50
LV_EJECT         36
dtype: int64

Datatypes:
LV_EJECT_METH    str
LV_EJECT         str
dtype: object


           Feature                                                        Description                     FormSection DataType SASAnalysisFormat Comment Information
198       LV_EJECT                              DECEASED DONOR LV EJECTION FRACTION %  HEART DONOR'S CARDIAC FUNCTION      NUM                               Unknown
199  LV_EJECT_METH  DECEASED DONOR LV EJECTION FRACTION METHOD: ECHO, MUGA, ANGIOGRAM  HEART DONOR'S CARDIAC FUNCTION      NUM          LVEJECTM             Unknown


LV_EJECT_METH: <ArrowStringArray>
['1', '3', nan]
Length: 3, dtype: str
LV_EJECT: <ArrowStringArray>
['60.00', '55.00', '59.00', '65.00', '35.00', '63.00', '64.00', '67.00',
 '50.00', '62.00',
 ...
 '56.20', '49.50', '65.40', '62.30', '68.30', '62

In [274]:
# fill NaN with 999: Missing
df['LV_EJECT_METH'] = df['LV_EJECT_METH'].fillna(999).astype(int)
df['LV_EJECT'] = df['LV_EJECT'].astype(float)

# df_flat FMTNAME: LVEJECTM
mapping = {
    999: "Unknown",
    1: "Echo",
    2: "MUGA",
    3: "Angiogram"
}

# mapping feature
df = uf.mapping_columns(df, 'LV_EJECT_METH', mapping, display=True)


 # mapping
colMap = {'LV_EJECT':'LV_EjectionFractionPercent_DON', 'LV_EJECT_METH':'LV_EjectionFractionMedthod_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"")
df_dict = uf.update_dictionary_information(df_dict, [196], txt='FMTNAME: LVEJECTM', feature_type='Category').copy()

# display
df_dict.iloc[idx]

Converted Column LV_EJECT_METH Unique Vaue(s) ['Echo', 'Angiogram', 'Unknown']
Categories (3, str): ['Angiogram', 'Echo', 'Unknown']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
198,LV_EjectionFractionPercent_DON,DECEASED DONOR LV EJECTION FRACTION %,DDR,1999-10-25,NaT,HEART DONOR'S CARDIAC FUNCTION,NUM,,,LV_EJECT,Numeric,
199,LV_EjectionFractionMedthod_DON,DECEASED DONOR LV EJECTION FRACTION METHOD: EC...,DDR,1999-10-25,NaT,HEART DONOR'S CARDIAC FUNCTION,NUM,LVEJECTM,,LV_EJECT_METH,Numeric,


### SHARE_TY

In [275]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'SHARE_TY', True)

          count unique top   freq
SHARE_TY  30725      4   3  15883

NaNs:
SHARE_TY    0
dtype: int64

Datatypes:
SHARE_TY    str
dtype: object


      Feature                                                                        Description FormSection DataType SASAnalysisFormat Comment Information
265  SHARE_TY  ALLOCATION TYPE-LOCAL/REGIONAL/NATIONAL - 3=LOCAL/4=REGIONAL/5=NATIONAL/6=FOREIGN                  NUM           SHARETY             Unknown


SHARE_TY: <ArrowStringArray>
['3', '4', '5', '6']
Length: 4, dtype: str


In [276]:
findMappingDfFlat(df.SHARE_TY, df_flat, 'SHARETY', 999)

Compare Length: 4 & 4

CODE         LABEL
   3         Local
   4      Regional
   5      National
   6 Foreign Donor


In [277]:
# df_flat FMTNAME: SHARETY
mapping = {
    '3': "Local",
    '4': "Regional",
    '5': "National",
    '6': "Foreign Donor"
}

# iterate
for col in features:
    # mapping feature
    df = uf.mapping_columns(df, col, mapping, display=True)


# mapping
colMap = {'SHARE_TY':'AllocationType_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Category', txt=f"FMTNAME: SHARETY")

# convert to category
df = uf.convert_to_category(df,  list(colMap.values()))

# display
df_dict.iloc[idx]

Converted Column SHARE_TY Unique Vaue(s) ['Local', 'Regional', 'National', 'Foreign Donor']
Categories (4, str): ['Foreign Donor', 'Local', 'National', 'Regional']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
265,AllocationType_DON,ALLOCATION TYPE-LOCAL/REGIONAL/NATIONAL - 3=LO...,CALCULATED,NaT,NaT,,NUM,SHARETY,,SHARE_TY,Category,FMTNAME: SHARETY


### PO2
- Lung PO2 (Partial Pressure of Oxygen) is a measure of the pressure exerted by oxygen in the lungs. It is an important indicator of how well oxygen is being transferred from the air in the lungs to the blood. The value of PO2 is typically measured in millimeters of mercury (mmHg) and provides insight into the efficiency of the respiratory system. In clinical and transplant contexts, particularly with donors, it helps assess the oxygenation status of the donor’s lungs, which is critical when evaluating organs for transplantation.

In [278]:
# display feature info
features, idx = uf.feature_information(df, df_dict, 'PO2', True)

              count unique     top   freq
PO2           30463   2917  112.00    125
PO2_DONE_DON  10699      3       Y  10649
PO2_FIO2_DON  30384     73  100.00  19272

NaNs:
PO2               262
PO2_DONE_DON    20026
PO2_FIO2_DON      341
dtype: int64

Datatypes:
PO2             str
PO2_DONE_DON    str
PO2_FIO2_DON    str
dtype: object


          Feature                                          Description     FormSection DataType SASAnalysisFormat Comment Information
217           PO2                           DECEASED DONOR PO2 ON 100%  ORGAN RECOVERY      NUM                               Unknown
218  PO2_DONE_DON                             DDR:Lung - Was pO2 done:  ORGAN RECOVERY  CHAR(1)                               Unknown
219  PO2_FIO2_DON  DDR:Lung pO2 on Fio2 //If Yes, Lung pO2 on FiO2 of:  ORGAN RECOVERY      NUM                               Unknown


PO2: <ArrowStringArray>
['182.00', '323.00', '401.00', '267.00', '209.00', '369.00', '386.00',
 '319.00', '162.00', '344

In [279]:
# fill NaN with X: Missing
df['PO2_DONE_DON'] = df['PO2_DONE_DON'].fillna('Unknown')
df['PO2'] = df['PO2'].astype(float)
df['PO2_FIO2_DON'] = df['PO2_FIO2_DON'].astype(float)

# feature value mapping
mapping = {'N': 'No', 'Y': 'Yes', 'U': 'Unknown'}

# mapping feature
df = uf.mapping_columns(df, 'PO2_DONE_DON', mapping, display=True)


# mapping
colMap = {'PO2':'LungPO2_DON', 'PO2_DONE_DON':'LungPO2_Done_DON', 'PO2_FIO2_DON':'LungPO2_FIO2_DON'}

# update column names & data dictionary
df, df_dict = uf.mapping_data_and_dictionary(df, df_dict, colMap, idx, feature_info='Numeric', txt=f"")
df_dict = uf.update_dictionary_information(df_dict, [215], txt='N/Y/U/X to No/Yes/Unknown', feature_type='Category')

# display
df_dict.iloc[idx]

Converted Column PO2_DONE_DON Unique Vaue(s) ['Yes', 'No', 'Unknown']
Categories (3, str): ['No', 'Unknown', 'Yes']


,Feature,Description,Form,FeatureStartDate,FeatureEndDate,FormSection,DataType,SASAnalysisFormat,Comment,OrginalFeature,FeatureType,Information
217,LungPO2_DON,DECEASED DONOR PO2 ON 100%,DDR,1999-10-25,NaT,ORGAN RECOVERY,NUM,,,PO2,Numeric,
218,LungPO2_Done_DON,DDR:Lung - Was pO2 done:,DDR,1999-10-25,NaT,ORGAN RECOVERY,CHAR(1),,,PO2_DONE_DON,Numeric,
219,LungPO2_FIO2_DON,"DDR:Lung pO2 on Fio2 //If Yes, Lung pO2 on FiO...",DDR,1999-10-25,NaT,ORGAN RECOVERY,NUM,,,PO2_FIO2_DON,Numeric,


In [280]:
# remove unused categories
df = u.remove_cat_zero_count(df)

In [281]:
# write the file
u.write_to_file(df, 'Heart_main', format='parquet')
# write dict file
u.write_to_file(df_dict, 'Dict_main', format='parquet')

30,725 records written to ../Data/Heart_main.parquet
306 records written to ../Data/Dict_main.parquet
